# Experiments walkthrough

Every experiment behind the report, and the research diagnostics around it,
unrolled step by step: how each run is made, and how each figure, table and
quoted number comes out of it. One section per ID of
[experiments.md](experiments.md), in the same order.

**Two modes** (the toggle in the panel below):

- **Redraw** reads the runs that already exist (`results/`, `experiments/`,
  `report/experiments/`, synced from the cluster). It follows the report's own
  choices of segment, model and parameters, so with it every section reproduces
  the report. A section whose runs are missing says which files and which
  `report/run_all.sh` job make them, and skips its remaining cells.
- **Live** makes a short run now with the panel's values. Its runs go only to
  `experiments/walkthrough/<ID>/`, never to `results/`. `n_frames` and the sweep
  extent set how big a live run is: small here, longer on the cluster.

**Same-code check.** Each section ends by calling the report's own function on
the same input and comparing with the unrolled result, loosely (`rtol=1e-2`).
It prints `[ok]` or `[DIFF]` and never stops the notebook.

The tracking run itself (`evaluation.experiment_tracking`) is not unrolled
here; section A0 links the per-model walkthroughs that show the inside of one.

Run from the repository root (the setup cell finds it). Every angular velocity
is shown in deg/s (`style.to_deg_s`); the code itself computes in rad/s.

<a id="overview"></a>

# Experiments — index

Every active experiment, research diagnostic and investigation in the repo: what
question it answers, what produces it, where its output lands and who reads it.
One row per experiment; the walkthrough notebook has one section per ID, in the
same order ([experiments_walkthrough.ipynb](experiments_walkthrough.ipynb), generated
from [walkthrough/](walkthrough/README.md): A0–A19
built). Add or remove an experiment here first.

Scanned 2026-11-25. Orchestration: [report/run_all.sh](report/run_all.sh)
(guide [report/RUN_ALL.md](report/RUN_ALL.md)); `R` below =
`python report/run_experiments.py --no-frames`.

## A. Report experiments

Run by `run_all.sh`; tables by [make_tables.py](report/make_tables.py), figures by
[make_figures.py](report/make_figures.py), quoted numbers by
[text_numbers.py](report/text_numbers.py). All on the Table II segments
(`report_runs.SEGMENTS`), `C_full`, 20 ms, 75 iterations unless stated.

| ID | Experiment | Question | Produced by (`run_all.sh` job) | Output | Read by | In report |
|----|-----------|----------|------------------|--------|---------|-----------|
| [A1](#A1) | Window choice | How constant is ω over a segment, how far does the scene move in one window? (GT only, no network) | `window`: `R --what window` | `report/experiments/window_choice.csv` | text_numbers (`segment_speeds`; the window lines of `intensity` and `track_length`) | text |
| [A2](#A2) | Every model, 3 s | Every model on every segment, 150 frames | `main_<seq>_<model>`: `R --what main` | `results/`, `main_base_*.csv` | text_numbers (`poster_segments`, `inloop_identical`, `track_length`; also the 3 s β runs of A6) | text |
| [A3](#A3) | Main table (Table III), 1.5 s | Every model on every segment, 75 frames; also the network column of the baseline table | `dissoc_<seq>_<model>`: `R --what main --duration 1.5` | `results/`, `dissoc_1p5s_*.csv` | table_main, table_baseline, fig_tracking, fig_dissociation, fig_maps/fig_appendix (with A13), fig_image_settings (with A12), text_numbers (`unanchored`, `reversal`, `reference`, first part of `intensity`) | tab:main, tab:baseline, fig:tracking, fig:dissociation |
| [A4](#A4) | Curl regulariser | Same with `delta_curl=0.6` | `curl_<seq>_<model>`: `R --what main --duration 1.5 --vary delta_curl=0.6` | `main_curl_*.csv` | table_curl, table_image | tab:curl |
| [A5](#A5) | Anchor alone (baseline) | What do CMax and the gyro score with no network? | `baseline`: `R --what baseline --duration 1.5` | `baseline_dt20ms*.csv/.npz` | table_baseline, fig_cmax_baseline, text_numbers | tab:baseline |
| [A6](#A6) | β vs speed | Does β depend on rotation speed? 7 real segments × 3 anchor settings | `beta_<i>`: `R --what beta --index i` | `results/` | fig_beta_speed, text_numbers | fig:beta_speed |
| [A7](#A7) | Anchor across poster segments | Every model on poster seg_A / seg_D (3 s) | `anchor_<seg>_<model>`: `evaluation.py --exp 2` | `results/poster_rotation/` | table_anchor | (table not in tex) |
| [A8](#A8) | Sensitivity sweep | One parameter at a time around the adopted config (δ_IMU, δ_FR, n_iters, dt, relaxation rates) | `sens_{cmax,imu,rates}_<seq>`: `R --what sweep --name sens` | `experiments/sens/` | table_sweep, fig_sensitivity, text_numbers | tab:sweep |
| [A9](#A9) | (δ_FR, anchor) grid | Full grid, CMax-anchor, 1.5 s | `grid_<seq>_FR<x>`: `R --what sweep --name grid --mode grid` | `experiments/grid/` | table_grid, table_params, fig_sensitivity, text_numbers | tab:params |
| [A10](#A10) | Old coordinate sweep | One axis at a time around δ_FR 0.5 / anchor 0.3 | `sweep_<seq>_{cmax,imu}_<v>`: `R --what sweep --name sweep` | `experiments/sweep/` | text_numbers reads only `…_cmax_delta_FR=0.5.csv` | ⚠ mostly unread, see below |
| [A11](#A11) | Regulariser ablation | What do shrink-I, MAP and curl each do? | `ablation_<seq>_<v>`: `R --what sweep --name ablation` | `experiments/ablation/` | table_regularisers, fig_regularisers, text_numbers | tab:regularisers, fig:regularisers |
| [A12](#A12) | Image settings | Best image: dt 10 ms × curl on/off | `best_image_<seq>_<model>`: `R --what sweep --name best_image --mode grid` | `experiments/best_image/` | table_image, fig_image_settings, text_numbers | tab:image, fig:image_settings |
| [A13](#A13) | Maps on poster | Every model stopped at frame 44 | `maps_poster_<model>`: `R --what sweep --name maps_poster` | `experiments/maps_poster/` | fig_maps, fig_appendix, text_numbers (`polarity`) | fig:maps, appendix |
| [A14](#A14) | Convergence within a frame | How does I sharpen over the iterations? | `converge_{dynamic,bicycle}`: `R --what converge` | `converge_path()` npz | fig_converge | fig:converge_dynamic, appendix |
| [A15](#A15) | Reconstruction | dt 20 vs 10 ms, iterative vs exact Poisson | `recon`: `R --what recon` | `experiments/recon_figure/` | fig_recon | fig:recon |
| [A16](#A16) | Read-out fft vs dct | Which exact solver gives the better image? | `readout_{20ms,10ms}`: `R --what readout` | `experiments/readout_fft_vs_dct/`, `readout_dt10ms/` | table_recon, text_numbers (10-vs-20 ms lines of `intensity`) | tab:recon |
| [A17](#A17) | In-loop Poisson | Where to apply the exact solve: in the loop or as a read-out? | `poisson`: `R --what poisson` | `experiments/poisson_solvers/` | text_numbers (`intensity_update`) | text |
| [A18](#A18) | Lens distortion | Forward model vs OpenCV, warp check; the figure needs only raw events | `distortion_test`: `tests/test_distortion_cmax.py`; figure: fig_distortion | stdout; figure | fig_distortion | fig:distortion |
| [A19](#A19) | Adopted parameters | The step sizes the report uses | `config.py` (no run) | n/a | table_params | tab:params |

Figures drawn but not included in `bare_jrnl.tex`: `fig_cmax_baseline`,
`fig_sensitivity`. Tables written but not in the tex: `table_anchor`, `table_grid`.

## B. Research diagnostics (not in the report)

| ID | Diagnostic | Question | Entry point | Output | Walkthrough |
|----|-----------|----------|-------------|--------|-------------|
| B1 | ω-recovery oracle ladder | Where does ω recovery fail? Rungs 0 / 1a / 1b / 1c inject GT flow or GT R | `diag_omega_ladder.py --mode run / maps / plots` | `diag_ladder_*.csv`, `diag_maps_<seg>.npz`, P1–P5 plots | [diag_omega_ladder.ipynb](diag_omega_ladder.ipynb) |
| B2 | OFCE step instability | Is the explicit G step unstable above \|F\| ≈ b? 7 variants | `run_all.sh` jobs `ofce_<v>`, `ofce_summary`: `diag_omega_ladder.py --mode ofce / ofce_summary` | OFCE dir `summary.csv` + plots | none yet |
| B3 | Distortion ablation | `undistort_events` vs `C_full` | `evaluation.py --exp 9` | `results/…` | none |
| B4 | Basin of attraction | From how far off does R converge? | `evaluation.py --exp 6` | `results/…` | none |

## C. Validation checks (tests that answer a question)

| ID | Check | Question | Script |
|----|-------|----------|--------|
| C1 | CMax front end | Does CMax ω match GT and IMU? | `tests/test_cmax_frontend.py` |
| C2 | CMax gradient | Analytic gradient vs finite differences, synthetic and real | `tests/test_cmax_gradient.py` |
| C3 | CMax V1 vs V2 | Do the two CMax integrations agree? | `tests/test_cmax_v1_vs_v2.py` |
| C4 | Poisson solvers | Do iterative / fft / dct recover I from G? | `tests/test_poisson_solvers.py` |
| C5 | Golden tracking | Do the tracking numbers still match the golden values? | `tests/test_golden_tracking.py` |

## D. Model walkthroughs (existing notebooks)

| Notebook | Covers | Status |
|----------|--------|--------|
| [cook_walkthrough.ipynb](cook_walkthrough.ipynb), [thesis_walkthrough.ipynb](thesis_walkthrough.ipynb), [thesis_imu_walkthrough.ipynb](thesis_imu_walkthrough.ipynb), [cmax_v1_walkthrough.ipynb](cmax_v1_walkthrough.ipynb), [cmax_v2_walkthrough.ipynb](cmax_v2_walkthrough.ipynb) | The inside of each model, one frame | current (first three untracked) |
| [demo.ipynb](demo.ipynb) | Live animation of the maps | current; a demo, not an experiment |

Plumbing, not experiments: [experiment_tracking.py](experiment_tracking.py) (run.json,
MLflow, `find_run`), `report/report_runs.py`, `evaluation.py --exp 5` (videos).

## E. Open investigations (no code yet)

Tracked in [docs/notes/todo.md](docs/notes/todo.md) (ω recovery: clip dependence P1,
overshoot, direction error, no-clip blow-up, zig-zag residuals) and
[docs/notes/cmax_todo.md](docs/notes/cmax_todo.md) (CMax bias, IWE borders, V1/V2 t_ref, …).

## Flags found while scanning

- **A10 orphan runs:** `run_all.sh` writes ~80 `sweep_*` jobs, but only the
  `delta_FR=0.5` base point is read (text_numbers). The sens sweep (A8) replaced
  the rest. Candidate for removal from `run_all.sh`.
- **Not yet in the walkthrough:** `text_numbers.little_effect_and_window` (δ_GI
  and the reconstruction; reads the A8 sens rows) has no section yet.
- **`fig_sensitivity` has no fallback:** it reads only `experiments/sens/`, while
  `table_sweep` and `text_numbers` fall back to `report/experiments/sens_*.csv`.
- **Old dataset name in local sens csvs:** the synthetic rows say
  `bycicle_sinthetic`, so their means are nan (report code and walkthrough alike)
  until the sens csvs are re-synced.
- **Report functions that raise on a missing dataset:** `fig_dissociation`,
  `fig_recon`, `compute_maps`, `fig_regularisers`, `text_numbers.reference` and
  `image_window` (and `report_runs.image_search` when a kind has no runs). The
  walkthrough checks them only where the data is present.


### What you see, cell by cell

| Section · cell | Shows |
|---|---|
| **setup** | nothing printed; imports and paths |
| **overview** | experiments.md; each ID links to its section |
| **panel** | the old global panel, read only by sections without a form yet (A1, A5, A8–A13, A18, A19) |
| **registry** | the saved-runs lookup (model → available values → matching runs, best ω error first) |
| **forms** | nothing printed; the per-experiment form machinery |
| **helpers** | nothing printed; `check`, `need`, `redirect`, `%%section` |
| **A0** | links to the five per-model walkthroughs |
| **A1 · 1** inputs | the window (20 ms in Redraw, panel dt in Live) and the sequences with data |
| **A1 · 2** reference | the reference ω per sequence: frame count and first 3 frames, deg/s |
| **A1 · 3** rows | window_choice rows per sequence and duration: \|ω\| deg/s, rel_std, sign flips |
| **A1 · 4** scene motion | rotation (deg) and centre-pixel travel (px) in one window of 10/20/30/50 ms |
| **A1 · 5** segment speeds | gyro speed of every candidate segment over 3 s; median-speed vs reported segment |
| **A1 · 6** text lines | the image-travel line (`intensity`) and the sign-changes line (`track_length`) |
| **A1 · check** | rows vs `cmd_window` and window_choice.csv; lines vs `segment_speeds`, `intensity`, `track_length` |
| **A2 · form** | models, segment, n_frames, dt, n_iters, poisson, δ_FR; Redraw: saved values only · Live: free |
| **A2 · 1** the runs | one folder per model of the form; Live: `SAVED RUN (not rerun)` when it exists |
| **A2 · 2** reference and gyro | their shapes and the first 4 frames, deg/s |
| **A2 · 3** gyro floor | gyro error against the reference: deg/s and deg |
| **A2 · 4** rows | one `cmd_main` csv row per model: err, median, dir, β, \|r\| |
| **A2 · 5** text numbers | the lines of `poster_segments` and of the 3 s `inloop_identical` (Redraw) |
| **A2 · 6** track length | `track_length`: error and \|r\| over the first N frames of the 3 s runs |
| **A2 · check** | `[ok]`/`[DIFF]`: rows vs `main_base_*.csv`, text vs `text_numbers` (Redraw); config and row vs `cmd_main` itself (Live) |
| **A3 · form, 1** | as A2, at 75 frames; report-only steps skip away from the report configuration |
| **A3 · 2** rows | gyro floor and the 1.5 s rows of every model |
| **A3 · 5** tab:main | every sequence's cells as a table, then the LaTeX of Table III |
| **A3 · 6** fig:tracking | ω per axis over 1.5 s, poster seg_C, CMax-anchor and thesis vs reference |
| **A3 · 7** text numbers | the lines of `unanchored` and `reversal` (Redraw) |
| **A3 · 8** reference | `reference`: gyro against a one-frame and a ±50 ms reference, deg/s |
| **A3 · 9** intensity | first part of `intensity`: I std and \|r\| along the poster track |
| **A3 · 10** fig:dissociation | read-out of thesis / CMax-anchor / thesis+IMU on boxes, bicycle, street, with \|r\| and err |
| **A3 · check** | rows vs `dissoc_1p5s_*.csv`, Table III vs `table_main`, figure lines vs `fig_tracking`, images vs `fig_dissociation`, text vs `text_numbers` |
| **A4 · 1** the runs | the δ_curl 0.6 run of each model (Redraw) or the new run (Live) |
| **A4 · 2** rows | gyro floor and one `main_curl` row per model |
| **A4 · 5** tab:curl | err and \|r\| with and without curl, then the LaTeX |
| **A4 · 6** text numbers | the 1.5 s half of `inloop_identical` |
| **A4 · check** | rows vs `main_curl_*.csv`, table vs `table_curl`, text; Live: config and row vs `cmd_main` |
| **A5 · 1–4** baseline | inputs, events and CMax, the per-frame loop (ref / CMax / gyro, deg/s), metrics and csv rows |
| **A5 · 5** tab:baseline | CMax alone / gyro alone / network per sequence, then the LaTeX |
| **A5 · 6** fig_cmax_baseline | ω per axis of CMax and gyro vs reference, three IWEs (not in the tex) |
| **A5 · 7** text numbers | `maps_add` |
| **A5 · check** | rows vs `baseline_dt20ms.csv`, `table_baseline`, figure vs `fig_cmax_baseline`, text; Live: vs `cmd_baseline` |
| **A6 · 1–3** β runs | the 21 β runs; speed (deg/s) and mean β per run |
| **A6 · 4** fig:beta_speed | β against speed, three series with Pearson r and fits |
| **A6 · 5** text numbers | `beta_speed`, `synthetic_beta_original` |
| **A6 · check** | β vs `read_summary`, figure vs `fig_beta_speed`, text; Live: `cmd_beta` config |
| **A7 · 1–3** tab:anchor | runs per poster segment and model, gyro floor, the table (not in the tex) |
| **A7 · check** | `gyro_floor`, `table_anchor` |
| **A8 · defs** | nothing printed; the sweep jobs and helpers (sweep-defs) |
| **A8 · 1–2** sens runs | csv rows (Redraw) or the Live job with its run and frame totals; sweep points per axis |
| **A8 · 3** tab:sweep | err per block, value and sequence, then the LaTeX |
| **A8 · 4** fig_sensitivity | mean err and \|r\| per panel (not in the tex) |
| **A8 · 5** text numbers | `sensitivity_both`, `sweep_ends`, `anchor_ablation`, `anchors_across_weights`, `kappa_rows` |
| **A8 · check** | table, figure data, 5 text functions; Live: `cmd_sweep` config and rows |
| **A9 · 1–5** grid | err pivot δ_FR × δ_anchor, tab:grid (not in the tex), tab:params rows vs grid optimum, `grid` |
| **A9 · check** | `table_grid`, `table_params` rows and tex, text |
| **A10 · 1–3** old sweep | the 80 sweep_* jobs (2 read, 78 unread: flag), the base-point rows |
| **A10 · check** | `synthetic_beta_original` |
| **A11 · 1–4** ablation | tab:regularisers, fig:regularisers, `regularisers` |
| **A11 · check** | table, figure images, text |
| **A12 · 1–5** best image | `image_search`, tab:image, fig:image_settings, `image_window` |
| **A12 · check** | search, table, figure images, text |
| **A13 · 1–4** maps | `compute_maps` stopped at frame 44, fig:maps and appendix, `polarity` |
| **A13 · check** | stop-at-44 equals Table III, figure images, text |
| **image-defs** | nothing printed; helpers of the image-figure checks |
| **A14 · 1–3** convergence | npz snapshots (Redraw) or `cmd_converge`'s loop (Live); std of I per checkpoint; fig:converge |
| **A14 · check** | figure pixels vs `fig_converge`; Live: snapshots vs `cmd_converge` |
| **A15 · 1–4** reconstruction | `recon_cfg` runs at 20/10 ms (with commit and date), \|r\| per panel, fig:recon |
| **A15 · check** | figure pixels and \|r\| vs `fig_recon`; Live: config vs `cmd_recon` |
| **A16 · 1–4** read-out | `cmd_readout` scoring, tab:recon, the 10-vs-20 ms lines of `intensity` |
| **A16 · check** | rows vs readout.csv, `table_recon`, text; Live: vs `cmd_readout` |
| **A17 · 1–4** in-loop Poisson | iterative/fft/dct runs, five panels, `intensity_update` |
| **A17 · check** | text, stats and pixels vs `cmd_poisson` |
| **A18 · 1–4** distortion | calibration, `distortion_numbers`, events raw and undistorted, fig:distortion |
| **A18 · check** | numbers vs csv and `distortion_numbers`, pixels vs `fig_distortion` |
| **A19 · 1–3** parameters | `config.py` values, tab:params, table vs the panel's defaults |
| **A19 · check** | table vs `table_params` and the tex; panel defaults = table values |

A section whose runs are missing prints `missing: <file>` and the `run_all.sh`
job that makes it, then each of its cells prints `(skipped: <ID> inputs missing)`.

In [ ]:
# setup -- repo root, paths, imports
import os, sys, contextlib
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, 'config.py')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
os.environ['IM_MLFLOW'] = '0'                  # walkthrough runs are not logged
for p in (ROOT, os.path.join(ROOT, 'report')):
    if p not in sys.path:
        sys.path.insert(0, p)
sys.argv = sys.argv[:1]                         # evaluation.py parses argv

import csv
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as wd
from IPython.display import display
from IPython.core.magic import register_cell_magic

import evaluation as E                          # the tracking run (Live mode)
import experiment_tracking
from config import THESIS_PARAMS, ITERS_PER_FRAME, POISSON_MODE
from interacting_maps.style import to_deg_s     # every omega is shown in deg/s
from report_runs import SEGMENTS, MODELS, run_cfg

# the report scripts the sections unroll and check against
import run_experiments, make_figures, make_tables, text_numbers
get_ipython().run_line_magic('matplotlib', 'inline')   # make_figures selected Agg

In [ ]:
# the panel -- defaults are the report's settings
DELTAS = [k for k in THESIS_PARAMS if k.startswith('delta_')]
DEFAULTS = dict(mode='Redraw', model='thesis_cmax', segment='poster', dt=0.02,
                n_iters=ITERS_PER_FRAME, poisson=POISSON_MODE, n_frames=10,
                sweep='one point', **{k: THESIS_PARAMS[k] for k in DELTAS})

W = dict(mode=wd.ToggleButtons(options=['Redraw', 'Live']),
         model=wd.Dropdown(options=MODELS),
         segment=wd.Dropdown(options=[s[3] for s in SEGMENTS]),
         dt=wd.FloatText(step=0.005),
         n_iters=wd.IntText(),
         poisson=wd.Dropdown(options=['iterative', 'fft', 'dct']),
         n_frames=wd.IntText(),
         sweep=wd.ToggleButtons(options=['one point', 'one axis', 'full']),
         **{k: wd.FloatText(step=0.01) for k in DELTAS})


def reset_panel(_=None):
    for k, v in DEFAULTS.items():
        W[k].value = v


for k, w in W.items():
    w.description = k
reset_panel()
reset = wd.Button(description='reset to report defaults')
reset.on_click(reset_panel)
# the Live panel; shown with the Redraw lookup by the registry cell below
PANEL = wd.VBox([W['model'],
                 wd.HBox([W['dt'], W['n_iters'], W['poisson']]),
                 wd.HBox([W['n_frames'], W['sweep']]),
                 wd.GridBox([W[k] for k in DELTAS],
                            layout=wd.Layout(grid_template_columns='repeat(3, 1fr)')),
                 reset])

In [ ]:
# helpers shared by every section
def params():
    """The panel's current values; read inside each cell, so a changed widget
    takes effect when the section is re-run."""
    p = {k: w.value for k, w in W.items()}
    p['seg'] = next(s for s in SEGMENTS if s[3] == p['segment'])   # (ds, sid, t0, label)
    p['deltas'] = {k: p[k] for k in DELTAS}
    return p


def walk_dir(ID):
    """experiments/walkthrough/<ID>: where Live runs and check outputs go."""
    d = os.path.join('experiments', 'walkthrough', ID)
    os.makedirs(d, exist_ok=True)
    return d


def need(paths, jobs=()):
    """True if every path exists; else print what is missing and what makes it."""
    miss = [p for p in paths if not os.path.exists(p)]
    for p in miss:
        print(f'missing: {p}')
    if miss and jobs:
        print('make with run_all.sh job(s): ' + ', '.join(jobs))
    return not miss


def check(what, got, want, rtol=1e-2):
    """Loose same-code check: numbers, arrays or dicts of numbers. Never raises."""
    keys = None
    try:
        if isinstance(want, dict):
            odd = sorted(set(want) ^ set(got))
            if odd:
                raise KeyError(f'keys only on one side: {odd}')
            keys = sorted(want)
            got, want = [got[k] for k in keys], [want[k] for k in keys]
        g, w = np.asarray(got, float), np.asarray(want, float)
        if g.size == 0 and w.size == 0:
            print(f'[skip] {what}: nothing to compare')
            return None
        if g.shape != w.shape:
            raise ValueError(f'shape {g.shape} vs {w.shape}')
        ok = bool(np.allclose(g, w, rtol=rtol, atol=1e-9, equal_nan=True))
        rel = np.nanmax(np.abs(g - w) / np.maximum(np.abs(w), 1e-9)) if g.size else 0.0
        detail = f'max rel diff {rel:.3g}'
        if keys and not ok:                     # name the entries that differ
            bad = ~np.isclose(g, w, rtol=rtol, atol=1e-9, equal_nan=True)
            detail += ' in ' + ', '.join(f'{k} {x:g} vs {y:g}'
                                         for k, x, y, b in zip(keys, g, w, bad) if b)
    except Exception as e:                      # a failed comparison is a DIFF
        ok, detail = False, f'{type(e).__name__}: {e}'
    print(f'[ok] {what}' if ok else f'[DIFF] {what}: {detail}')
    return ok


def nums(lines):
    """Every number in some printed lines, in order -- to compare text output."""
    import re
    return [float(x) for l in lines
            for x in re.findall(r'[-+]?\d+\.?\d*(?:e[-+]?\d+)?', l)]


def note_redraw(p, nf):
    """Redraw shows the report's saved runs; other saved runs are in the lookup."""
    print(f"Redraw: the report's saved runs of every model, {nf} frames, report settings "
          "(other saved runs: the lookup in the panel).")


@contextlib.contextmanager
def redirect(module, **attrs):
    """Set module attributes (output folders, save) and restore them, even on error."""
    old = {k: getattr(module, k) for k in attrs}
    for k, v in attrs.items():
        setattr(module, k, v)
    try:
        yield
    finally:
        for k, v in old.items():
            setattr(module, k, v)


READY = {}                                      # section ID -> its inputs exist


@register_cell_magic
def section(line, cell):
    """%%section A2 -- run the cell only if section A2's inputs were found."""
    ID = line.strip()
    if READY.get(ID, True):
        get_ipython().run_cell(cell).raise_error()   # a failing cell fails "Run all"
    else:
        print(f'(skipped: {ID} inputs missing)')

In [ ]:
# run registry -- every saved run.json, the Redraw lookup and the Live guard
import html, io, json, re

LOOKUP_KEYS = ['segment', 'dt', 'n_frames', 'n_iters', 'poisson', 'distortion_mode', *DELTAS]
REG_COLS = ['folder', 'root', 'model', 'dataset', 'segment_id', 'segment', 't_start',
            *LOOKUP_KEYS[1:6], *DELTAS, 'err_deg_s', 'dir_err_deg', 'beta']


def scan_runs(roots=('results', 'experiments')):
    """One row per readable <root>/**/run.json. Old configs are filled like
    experiment_tracking.find_run fills them: segment_id from the folder name,
    poisson 'iterative', a step size the run did not store = 0.0."""
    rows = []
    for top in roots:
        for f in glob.glob(os.path.join(top, '**', 'run.json'), recursive=True):
            d = os.path.dirname(f)
            try:
                r = experiment_tracking.read_run(d)
                c, s = r['config'], r.get('summary') or {}
            except (OSError, ValueError, KeyError, TypeError):
                continue
            sid = c.get('segment_id') or (re.match(r'(seg_[^_]+)', os.path.basename(d)) or [None, '?'])[1]
            pr = c.get('params') or {}
            rows.append(dict(
                folder=d, root=os.path.dirname(os.path.dirname(os.path.dirname(d))),
                model=c.get('model'), dataset=c.get('dataset'), segment_id=sid,
                segment=f"{c.get('dataset')} {sid}", t_start=c.get('t_start'),
                dt=c.get('frame_duration'), n_frames=c.get('n_frames'),
                n_iters=c.get('n_iters'),
                poisson=c.get('poisson', 'iterative'), distortion_mode=c.get('distortion_mode'),
                **{k: float(pr.get(k) or 0.0) for k in DELTAS},
                err_deg_s=s.get('mean_err_deg_s', np.nan),
                dir_err_deg=s.get('mean_dir_err_deg', np.nan), beta=s.get('mean_beta', np.nan)))
    return pd.DataFrame(rows, columns=REG_COLS)


def matching(reg, chosen, skip=None):
    """Rows of reg that match every choice ('any' or absent = no filter), except `skip`."""
    m = pd.Series(True, index=reg.index)
    for k, v in chosen.items():
        if k != skip and v != 'any':
            m &= reg[k] == v
    return reg[m]


def available(reg, chosen):
    """For each lookup key, the values saved runs have given every other choice
    (a chosen dropdown keeps its alternatives)."""
    return {k: sorted(matching(reg, chosen, skip=k)[k].dropna().unique().tolist())
            for k in LOOKUP_KEYS}


def show_matches(chosen):
    """The matching runs, best mean omega error first, into the MATCH widget."""
    hit = matching(REG, chosen).sort_values('err_deg_s')
    cols = ['folder', 'segment', 'n_frames', 'err_deg_s', 'dir_err_deg', 'beta']
    MATCH.value = ('<b>no saved run matches</b>' if hit.empty else
                   f'<b>{len(hit)} run(s)</b> -- play them back in the inspection viewer'
                   + hit[cols].to_html(index=False, float_format='%.2f'))
    return 'no saved run matches' if hit.empty else hit[cols].to_string(index=False)


# --- the Redraw lookup: model first, then each choice narrows the others
LOOK = {'model': wd.Dropdown(options=MODELS, description='model'),
        **{k: wd.Dropdown(options=['any'], description=k) for k in LOOKUP_KEYS}}
INFO, MATCH, HEAD = wd.HTML(), wd.HTML(), wd.HTML()
_busy = [False]


def chosen_now():
    return {k: w.value for k, w in LOOK.items()}


def refresh(change=None, keep=False):
    """Refill every dropdown from REG; a model change resets the others and
    prints the values that model's saved runs have (keep=True: a rescan keeps them)."""
    if _busy[0]:
        return
    _busy[0] = True
    try:
        HEAD.value = f'<b>Saved runs</b> ({len(REG)} run.json under results/ and experiments/)'
        if not keep and (change is None or change['owner'] is LOOK['model']):
            for k in LOOKUP_KEYS:
                LOOK[k].options, LOOK[k].value = ['any'], 'any'
            vals = available(REG, {'model': LOOK['model'].value})
            INFO.value = '<pre>' + html.escape(repr(vals), quote=False) + '</pre>'
        av = available(REG, chosen_now())
        for k in LOOKUP_KEYS:
            keep = LOOK[k].value
            LOOK[k].options = ['any', *av[k]]
            LOOK[k].value = keep if keep in LOOK[k].options else 'any'
        show_matches(chosen_now())
    finally:
        _busy[0] = False


def rescan(_=None):
    global REG
    REG = scan_runs()
    refresh(keep=True)


for w in LOOK.values():
    w.observe(refresh, names='value')
rescan_btn = wd.Button(description='rescan runs')
rescan_btn.on_click(rescan)
REG = scan_runs()
refresh()
LOOKUP = wd.VBox([HEAD, LOOK['model'], INFO,
                  wd.GridBox([LOOK[k] for k in LOOKUP_KEYS],
                             layout=wd.Layout(grid_template_columns='repeat(3, 1fr)')),
                  rescan_btn, MATCH])


def show_mode(change=None):
    LOOKUP.layout.display = '' if W['mode'].value == 'Redraw' else 'none'
    PANEL.layout.display = 'none' if W['mode'].value == 'Redraw' else ''


W['mode'].observe(show_mode, names='value')
show_mode()
TOP = wd.HBox([W['mode'], W['segment']])          # the sections' segment, in both modes
display(wd.VBox([TOP, LOOKUP, PANEL]))


# --- saved runs of one configuration (used by saved_or_run in the forms cell)
def saved_runs(rc):
    """Complete saved runs (with a summary) of rc's configuration, by
    experiment_tracking.find_run, in every scanned root and in rc.out_root."""
    roots = sorted(set(REG.root) | {rc.out_root})
    with contextlib.redirect_stdout(io.StringIO()):      # find_run's AMBIGUOUS lines
        hits = [experiment_tracking.find_run(
                    root, rc.dataset, rc.model, rc.segment_id, rc.n_frames, rc.params,
                    dt=rc.frame_duration, t_start=rc.t_start, n_iters=rc.n_iters,
                    poisson=rc.poisson, distortion_mode=rc.distortion_mode) for root in roots]
    return [h for h in hits
            if h and 'mean_err_deg_s' in (experiment_tracking.read_summary(h) or {})]

In [ ]:
# forms -- one per experiment: its own mode and knobs; never a global value
from collections import namedtuple
from IPython.display import HTML

# kind: 'one' | 'many'. choices: a fixed list (configs, cases, solvers ...) offered in
# both modes, not looked up in the registry; the section says which have no output.
Knob = namedtuple('Knob', 'name report kind choices', defaults=('one', None))


def _pairs(choices):
    """Widget options for a choices knob: (label, value), so tuples stay values."""
    return [(str(c), c) for c in choices]
FORMS = {}                                      # ID -> {'mode', 'knobs', 'redraw', 'live', 'hints', 'box'}
LABEL = {f'{ds} {sid}': lab for ds, sid, _t0, lab in SEGMENTS}      # REG segment -> panel label
COL = {'models': 'model'}                       # knob name -> REG column
LIVE_FRAMES = 10                                # Live n_frames default: smoke size here
PRESET = globals().get('PRESET', {})            # starting values for new forms (tests set it)


def saved_view(fixed=None):
    """REG rows a form can redraw: report segments, C_full, the form's fixed values."""
    v = REG[REG.distortion_mode == 'C_full'].copy()
    v['segment'] = v['segment'].map(LABEL)
    for k, val in (fixed or {}).items():
        v = v[v[k] == val]
    return v.dropna(subset=['segment'])


def form_options(ID):
    """Redraw options of each knob: values of saved runs that fit the form's other knobs."""
    F = FORMS[ID]
    v = saved_view(F['fixed'])
    opts = {}
    for k in F['knobs']:
        if k.choices is not None:
            continue
        rows = v
        for o in F['knobs']:
            val = F['redraw'][o.name].value
            if o.name == k.name or o.choices is not None or val in (None, ()):
                continue
            col = COL.get(o.name, o.name)
            rows = rows[rows[col].isin(val)] if o.kind == 'many' else rows[rows[col] == val]
        opts[k.name] = sorted(rows[COL.get(k.name, k.name)].dropna().unique().tolist())
    return opts


def _refresh_form(ID):
    """Refill the Redraw options until they agree with the values (a value forced to
    another option changes the options of the others)."""
    F = FORMS[ID]
    if F['busy']:
        return
    F['busy'] = True
    try:
        for _ in range(len(F['knobs']) + 1):
            changed = False
            for name, vals in form_options(ID).items():
                w = F['redraw'][name]
                keep = w.value
                if list(w.options) != vals:
                    w.options = vals
                if isinstance(keep, tuple):
                    new = tuple(x for x in keep if x in vals) or tuple(vals)
                else:
                    new = keep if keep in vals else (vals[0] if vals else None)
                w.value = new
                changed |= new != keep
            if not changed:
                break
    finally:
        F['busy'] = False


def _choice_widget(k):
    w = (wd.SelectMultiple if k.kind == 'many' else wd.Dropdown)(options=_pairs(k.choices),
                                                                  description=k.name)
    w.value = tuple(k.report) if k.kind == 'many' else k.report
    return w


def _live_widget(k):
    if k.choices is not None:
        return _choice_widget(k)
    if k.name == 'models':
        return wd.SelectMultiple(options=MODELS, value=tuple(k.report), description='models')
    if k.name == 'segment':
        return wd.Dropdown(options=list(LABEL.values()), value=k.report, description='segment')
    if k.name == 'poisson':
        return wd.Dropdown(options=['iterative', 'fft', 'dct'], value=k.report, description='poisson')
    if k.name in ('n_frames', 'n_iters'):
        return wd.IntText(value=LIVE_FRAMES if k.name == 'n_frames' else k.report, description=k.name)
    return wd.FloatText(value=k.report, description=k.name, step=0.01)


def form(ID, knobs, fixed=None):
    """Show experiment ID's form: mode, one widget per knob, reset. Redraw offers only
    saved values (cascading) among runs with the `fixed` values (e.g. no curl);
    Live is free (n_frames from 10, report value as hint). Re-running keeps the values."""
    if ID not in FORMS:
        F = FORMS[ID] = dict(knobs=list(knobs), busy=False, fixed=dict(fixed or {}),
                             mode=wd.ToggleButtons(options=['Redraw', 'Live'], description=ID))
        F['redraw'] = {k.name: (_choice_widget(k) if k.choices is not None else
                                wd.SelectMultiple(description=k.name, value=()) if k.kind == 'many'
                                else wd.Dropdown(description=k.name)) for k in knobs}
        F['live'] = {k.name: _live_widget(k) for k in knobs}
        F['hints'] = {k.name: wd.HTML(f'<i>report: {k.report}</i>') for k in knobs}
        for k in knobs:                         # the report value first, where saved
            if k.choices is not None:
                continue
            F['redraw'][k.name].options = [k.report] if k.kind == 'one' else list(k.report)
            F['redraw'][k.name].value = k.report if k.kind == 'one' else tuple(k.report)
        _refresh_form(ID)
        _preset(F)
        _refresh_form(ID)
        for w in F['redraw'].values():
            w.observe(lambda _c, ID=ID: _refresh_form(ID), names='value')
        reset = wd.Button(description='reset to report')
        reset.on_click(lambda _b, ID=ID: reset_form(ID))
        red = wd.GridBox(list(F['redraw'].values()),
                         layout=wd.Layout(grid_template_columns='repeat(3, 1fr)'))
        live = wd.GridBox([wd.VBox([F['live'][k.name], F['hints'][k.name]]) for k in knobs],
                          layout=wd.Layout(grid_template_columns='repeat(3, 1fr)'))
        note = wd.HTML()
        def show(_c=None, red=red, live=live, note=note, F=F):
            r = F['mode'].value == 'Redraw'
            red.layout.display, live.layout.display = ('', 'none') if r else ('none', '')
            note.value = ('<i>Redraw: only values with saved runs</i>' if r else
                          '<i>Live: free values; a saved configuration is shown, not rerun</i>')
        F['mode'].observe(show, names='value')
        show()
        F['box'] = wd.VBox([wd.HBox([F['mode'], reset]), red, live, note],
                           layout=wd.Layout(border='1px solid #999', padding='4px'))
    display(FORMS[ID]['box'])


def _preset(F):
    """Start a new form from PRESET (mode, and knob values; 'model' sets 'models')."""
    pre = dict(PRESET)
    if 'model' in pre:
        pre.setdefault('models', (pre['model'],))
    F['mode'].value = pre.get('mode', F['mode'].value)
    for k in F['knobs']:
        if k.name not in pre:
            continue
        v = tuple(pre[k.name]) if k.kind == 'many' else pre[k.name]
        F['live'][k.name].value = v
        r = F['redraw'][k.name]
        opts = k.choices if k.choices is not None else r.options
        if (set(v) <= set(opts)) if k.kind == 'many' else (v in opts):
            r.value = v


def reset_form(ID):
    """Every knob back to the report's value (Redraw: where saved runs have it)."""
    F = FORMS[ID]
    F['busy'] = True                            # no cascade while the values are set
    try:
        for k in F['knobs']:
            rep = tuple(k.report) if k.kind == 'many' else k.report
            F['live'][k.name].value = LIVE_FRAMES if k.name == 'n_frames' else rep
            w = F['redraw'][k.name]
            if k.choices is None:
                w.options = list(rep) if k.kind == 'many' else [rep]
            w.value = rep
    finally:
        F['busy'] = False
    _refresh_form(ID)


panel_params = globals().get('panel_params', params)   # the old global panel; kept on a re-run


def params(ID=None):
    """Experiment ID's form values, with the keys of the old panel params();
    params() without an ID reads the old global panel."""
    if ID is None:
        return panel_params()
    F = FORMS[ID]
    q = {**DEFAULTS, 'mode': F['mode'].value, 'models': list(MODELS)}
    ws = F['redraw'] if q['mode'] == 'Redraw' else F['live']
    for k in F['knobs']:
        v = ws[k.name].value
        q[k.name] = list(v) if k.kind == 'many' else v
    for k in ('n_frames', 'n_iters'):           # REG columns can come back as floats
        q[k] = int(q[k]) if q[k] is not None else None
    q['form'] = ID
    if not any(k.name == 'model' for k in F['knobs']):     # a form's own model knob wins
        q['model'] = q['models'][0] if q['models'] else None
    q['seg'] = next((s for s in SEGMENTS if s[3] == q['segment']), None)
    q['deltas'] = {k: q[k] for k in DELTAS}
    return q


def is_report(ID):
    """Every knob at the report's value. The choice of models and of the segment does
    not count: the report has runs of every model on every Table II segment."""
    q = params(ID)
    same = lambda k: (set(q[k.name]) == set(k.report)) if k.kind == 'many' else q[k.name] == k.report
    return all(same(k) for k in FORMS[ID]['knobs'] if k.name not in ('models', 'segment'))


def report_skip(ID):
    """True (and says so) when the form is away from the report's configuration."""
    if is_report(ID):
        return False
    print('[skip] not the report configuration')
    return True


def roots():
    """Where a form looks for saved runs: results/ first, then every other scanned root."""
    return ['results'] + sorted(set(REG.root) - {'results'})


def report_or_saved(rc, root, need='maps_final.npz'):
    """Redraw: rc's run in the report job's root (the folder the job names, as the
    report reads it), else any complete saved run with that configuration; it must
    hold `need`. None if there is none."""
    old, rc.out_root = rc.out_root, root
    try:
        d = rc.output_dir
    finally:
        rc.out_root = old
    if os.path.exists(os.path.join(d, need)):
        return d
    hits = [h for h in saved_runs(rc) if os.path.exists(os.path.join(h, need))]
    return hits[0] if hits else None


def saved_or_run(rc):
    """The folder of rc's run: a complete saved run is used, not rerun (banner);
    otherwise run it now and save it."""
    hits = saved_runs(rc)
    if hits:
        print(f'>>> SAVED RUN (not rerun): {hits[0]}')
        display(HTML(f'<b style="color:#c00">SAVED RUN (not rerun):</b> <code>{hits[0]}</code>'))
        return hits[0]
    E.experiment_tracking(rc, save_frames=False)
    rescan()
    return rc.output_dir

<a id="A0"></a>

## A0 · Inside one tracking run (linked, not unrolled) · [↑ overview](#overview)

Every section below calls the tracking run `evaluation.py::experiment_tracking`
as one step: events → frames → network → per-frame ω → `tracking.csv` +
`run.json`. What happens inside it, model by model, is unrolled in:

- [cook_walkthrough.ipynb](cook_walkthrough.ipynb)
- [thesis_walkthrough.ipynb](thesis_walkthrough.ipynb)
- [thesis_imu_walkthrough.ipynb](thesis_imu_walkthrough.ipynb)
- [cmax_v1_walkthrough.ipynb](cmax_v1_walkthrough.ipynb) (`thesis_cmax`)
- [cmax_v2_walkthrough.ipynb](cmax_v2_walkthrough.ipynb) (`thesis_cmax_v2`)

<a id="A1"></a>

## A1 · Window choice · [↑ overview](#overview)

**Question.** How constant is ω over a segment of 0.5–3 s, and how far does the
scene move inside one accumulation window? This chose the 20 ms window and the
segment lengths before any model was run. Ground truth only: no network runs,
so both modes compute everything here (Redraw at the report's 20 ms, Live at
the panel's `dt`).

**Made by** `report/run_experiments.py::cmd_window` (`--what window`),
`run_all.sh` job `window` → `report/experiments/window_choice.csv` (one row per
sequence and duration; its `omega_mean` column stays in the code's unit, rad/s,
as stored data; every speed shown here is deg/s).

**Read by** `report/text_numbers.py`, quoted in the report text (no figure or
table): the line *image travel per 20 ms window* is in `::intensity`, the line
*sign changes of the dominant axis* is in `::track_length`. Both read the csv.
Also unrolled here, because it is the same kind of ground-truth-only question:
`::segment_speeds` (is each reported segment the one of median gyro speed?).

**Steps**: reference ω per frame → per duration |ω|, relative std, sign flips
(the csv rows) → scene motion inside one window → segment speeds → the text
lines → same-code check.

In [ ]:
# step 1 -- the inputs: ground truth of every Table II sequence (no runs needed)
p = params()
dt_A1 = 0.02 if p['mode'] == 'Redraw' else p['dt']     # cmd_window's window is 20 ms
if p['mode'] == 'Redraw':
    print("Redraw: the report's 20 ms window (cmd_window fixes it); nothing here needs a saved run.")
    if p['dt'] != 0.02:
        print(f"The panel's dt {p['dt']} is used in Live mode.")
else:
    print(f'Live: window dt {dt_A1*1000:g} ms from the panel')
DURS_A1 = (0.5, 1.0, 1.5, 2.0, 3.0)                     # cmd_window's durations
segs_A1 = [s for s in SEGMENTS if need([os.path.join('data', s[0])])]
if len(segs_A1) < len(SEGMENTS):
    print('   (datasets: download_gdrive.sh; every step below uses the sequences with data)')
READY['A1'] = bool(segs_A1)
print('sequences with data:', [s[3] for s in segs_A1])

In [ ]:
%%section A1
# step 2 -- the reference omega per frame, run_experiments.py::cmd_window
# cmd_window recomputes it for each duration from the same t0; the first n frames
# of the 3 s series are the same numbers, so one series per sequence is sliced.
def reference_series_A1(seg, dt, n):
    """cmd_window: get_reference_omega over frames [t0+k*dt, t0+(k+1)*dt), rad/s."""
    ds, _sid, t0, _lab = seg
    pp = E.get_dataset_paths(ds)
    gt, imu = E.load_groundtruth(pp['groundtruth']), E.load_imu(pp['imu'])
    om = E.load_omega_gt(pp.get('omega_gt'))
    return np.array([E.get_reference_omega(gt, imu, t0+k*dt, t0+(k+1)*dt, om)[0]
                     for k in range(n)])

w_A1 = {s[3]: reference_series_A1(s, dt_A1, int(round(max(DURS_A1) / dt_A1))) for s in segs_A1}
for lab, w in w_A1.items():
    print(f'{lab:8s} {w.shape[0]} frames of {dt_A1*1000:g} ms; first 3 frames, deg/s:',
          np.round(to_deg_s(w[:3]), 1).tolist())

In [ ]:
%%section A1
# step 3 -- per duration: mean |omega|, relative std, sign flips of the dominant axis
def window_rows_A1(lab, w, dt):
    """cmd_window's rows for one sequence: (lab, dur, |w| mean, std/mean, flips)."""
    rows = []
    for dur in DURS_A1:
        n = int(round(dur / dt))
        wn = w[:n]
        m = np.linalg.norm(wn, axis=1)
        ax = int(np.argmax(np.abs(wn).mean(0)))          # dominant rotation axis
        flips = int((np.diff(np.sign(wn[:, ax])) != 0).sum())
        rows.append((lab, dur, m.mean(), m.std() / m.mean(), flips))
    return rows

rows_A1 = [r for lab, w in w_A1.items() for r in window_rows_A1(lab, w, dt_A1)]
display(pd.DataFrame([(a, b, to_deg_s(c), d, e) for a, b, c, d, e in rows_A1],
                     columns=['sequence', 'duration_s', '|w| deg/s', 'rel_std', 'sign_flips']
                     ).round({'|w| deg/s': 1, 'rel_std': 3}))

In [ ]:
%%section A1
# step 4 -- the scene motion inside one window (cmd_window's second table)
# rotation in one window = |w| * window; image travel at the centre pixel = f * angle,
# with f = 199 px (cmd_window's constant: the DAVIS240 focal length of calib.txt)
F_PX_A1 = 199.0
ms_A1 = [10, 20, 30, 50] + ([] if dt_A1*1000 in (10, 20, 30, 50) else [round(dt_A1*1000, 3)])
motion_A1 = {lab: {f'{ms:g} ms': f'{np.degrees(wm*ms/1000):4.2f} deg / {F_PX_A1*wm*ms/1000:4.1f} px'
                   for ms in ms_A1}
             for lab, dur, wm, _rs, _fl in rows_A1 if dur == 1.0}
display(pd.DataFrame(motion_A1).T)

In [ ]:
%%section A1
# step 5 -- text_numbers.py::segment_speeds: is each reported segment the median-speed one?
# gyro |w| over 3 s (150 frames of 20 ms) of every candidate segment of config.DATASET_SEGMENTS
from config import DATASET_SEGMENTS, get_dataset_paths as _paths_A1
from interacting_maps import eval_io as _io_A1
from report_runs import SYNTHETIC as _SYN_A1

def segment_speed_lines_A1(segs, dt):
    lines = []
    real = [s for s in segs if s[3] not in _SYN_A1]
    n = int(round(3.0 / dt))
    for ds in dict.fromkeys(s[0] for s in real):
        imu = _io_A1.load_imu(_paths_A1(ds)['imu'])
        chosen = [s[1] for s in real if s[0] == ds]
        speeds = {}
        for c in DATASET_SEGMENTS[ds]:
            w = np.array([_io_A1.get_gyro_for_frame(imu, c['t_start'] + k*dt, c['t_start'] + (k+1)*dt)
                          for k in range(n)])
            m = to_deg_s(np.linalg.norm(w, axis=1))
            speeds[c['id']] = m.mean()
            lines.append(f"{ds[:14]:14s} {c['id']}{'*' if c['id'] in chosen else ' '} "
                         f"mean |w| {m.mean():6.1f} deg/s, std/mean {m.std() / m.mean():.2f}, "
                         f"range {m.min():5.1f}-{m.max():5.1f}")
        order = sorted(speeds, key=speeds.get)         # the median of the sorted speeds
        k = len(order)
        lines.append(f"   median-speed segment(s): {', '.join(order[(k - 1) // 2: k // 2 + 1])}; "
                     f"reported: {', '.join(chosen)}")
    return lines

speed_lines_A1 = segment_speed_lines_A1(segs_A1, dt_A1)
print(f'== segment speeds (* = reported), {dt_A1*1000:g} ms frames over 3 s')
print('\n'.join(speed_lines_A1) or '(no real sequence with data)')

In [ ]:
%%section A1
# step 6 -- the window lines of the text: text_numbers.py::intensity (image travel)
# and ::track_length (sign changes). Both read window_choice.csv: Redraw reads the
# report's csv as they do; Live uses this section's rows at the panel's dt.
from report_runs import SYNTHETIC as _SYN_A1
WC_A1 = os.path.join('report', 'experiments', 'window_choice.csv')

def window_text_A1(rows, dt):
    """rows: dicts with the csv's columns. Returns the two lines, as text_numbers says them."""
    real = [s[3] for s in SEGMENTS if s[3] not in _SYN_A1]
    syn = [s[3] for s in SEGMENTS if s[3] in _SYN_A1]
    travel, flips = [], []
    for name, labs in (('real', real), ('synthetic', syn)):       # intensity
        w = [float(r['omega_mean']) for r in rows
             if r['sequence'] in labs and float(r['duration_s']) == 1.0]
        if w:
            travel.append(f'image travel per {dt*1000:g} ms window, centre pixel ({name}): '
                          f'{199.0*np.mean(w)*dt:.1f} px')
    flips.append('sign changes of the dominant axis of the reference (run_experiments window): '
                 + '; '.join(f'{lab} ' + ', '.join(f'{r["duration_s"]} s {r["sign_flips"]}'
                                                  for r in rows if r['sequence'] == lab
                                                  and float(r['duration_s']) in (0.5, 1.5, 3.0))
                             for lab in real))                      # track_length
    return travel, flips

def as_csv_rows_A1(rows):
    """this section's rows as the csv stores them (same rounding, strings)"""
    return [dict(sequence=a, duration_s=str(b), omega_mean=str(round(c, 4)),
                 rel_std=str(round(d, 4)), sign_flips=str(e)) for a, b, c, d, e in rows]

if p['mode'] == 'Redraw':
    src_A1 = list(csv.DictReader(open(WC_A1))) if need([WC_A1], ['window']) else []
else:
    src_A1 = as_csv_rows_A1(rows_A1)
travel_A1, flips_A1 = window_text_A1(src_A1, dt_A1)
print('== text_numbers.intensity (window line)'); print('\n'.join(travel_A1) or '(no rows)')
print('== text_numbers.track_length (window line)'); print('\n'.join(flips_A1))

In [ ]:
%%section A1
# same-code check -- cmd_window itself (OUTDIR -> walk_dir('A1')/check, only the
# sequences with data), the report's csv, and the text_numbers functions.
# The report's code is fixed at 20 ms, so a Live run at another dt is recomputed
# at 20 ms with the unrolled functions above before comparing.
import io
if dt_A1 == 0.02:
    rows20_A1, speed20_A1 = rows_A1, speed_lines_A1
else:
    print(f'(Live dt {dt_A1*1000:g} ms: comparing a 20 ms recompute of the unrolled steps)')
    rows20_A1 = [r for s in segs_A1
                 for r in window_rows_A1(s[3], reference_series_A1(s, 0.02, 150), 0.02)]
    speed20_A1 = segment_speed_lines_A1(segs_A1, 0.02)
mine_A1 = as_csv_rows_A1(rows20_A1)
num_A1 = lambda rows: [[float(r[k]) for k in ('duration_s', 'omega_mean', 'rel_std', 'sign_flips')]
                       for r in rows]

out_A1 = os.path.join(walk_dir('A1'), 'check')
os.makedirs(out_A1, exist_ok=True)
with redirect(run_experiments, OUTDIR=out_A1, SEGMENTS=segs_A1), \
     contextlib.redirect_stdout(io.StringIO()):
    run_experiments.cmd_window(E, None)
theirs_A1 = list(csv.DictReader(open(os.path.join(out_A1, 'window_choice.csv'))))
check('cmd_window rows', num_A1(mine_A1), num_A1(theirs_A1))

if need([WC_A1], ['window']):                 # the report's csv, sequences with data
    rep_A1 = [r for r in csv.DictReader(open(WC_A1)) if r['sequence'] in {s[3] for s in segs_A1}]
    check('window_choice.csv rows', num_A1(mine_A1), num_A1(rep_A1))
    rep_all_A1 = list(csv.DictReader(open(WC_A1)))
    travel_rep_A1, flips_rep_A1 = window_text_A1(rep_all_A1, 0.02)
else:
    travel_rep_A1, flips_rep_A1 = [], []

# text_numbers: segment_speeds on the sequences with data (REAL redirected)
real_A1 = [s for s in segs_A1 if s[3] not in _SYN_A1]
with redirect(text_numbers, OUT=[], REAL=real_A1), contextlib.redirect_stdout(io.StringIO()):
    text_numbers.segment_speeds()
    ss_A1 = [l for l in text_numbers.OUT if l and not l.startswith('==')]
check('text_numbers.segment_speeds', nums(speed20_A1), nums(ss_A1))

# the window lines: their functions read the report's csv, so the 20 ms lines of that csv
for name, prefix, mine in (('intensity', 'image travel', travel_rep_A1),
                           ('track_length', 'sign changes', flips_rep_A1)):
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        getattr(text_numbers, name)()
        theirs = [l for l in text_numbers.OUT if l.startswith(prefix)]
    if theirs:
        check(f'text_numbers.{name} window line', nums(mine), nums(theirs))
    else:
        print(f'missing: the window line of text_numbers.{name} -- '
              + ('it is printed only when every 3 s run of every model is on disk '
                 '(run_all.sh jobs main_<seq>_<model>)' if name == 'track_length'
                 else 'it needs report/experiments/window_choice.csv (job window)'))

<a id="A2"></a>

## A2 · Every model, 3 s · [↑ overview](#overview)

**Question.** How does every model track ω over a full 3 s segment (150 frames
of 20 ms)?

**Made by** `report/run_experiments.py::cmd_main` with `--duration 3.0`,
`run_all.sh` jobs `main_<seq>_<model>` → runs in `results/`, one row each in
`report/experiments/main_base_<seq>_<model>.csv`.

**Read by** `report/text_numbers.py::poster_segments` and the 3 s half of
`::inloop_identical` (its 1.5 s half is A4). These numbers are quoted in the
report text; no figure or table uses the 3 s runs.

**Steps** (those of `cmd_main`, then of the two readers): reference ω and gyro
→ gyro floor → the run (found or made) → its summary row → the text numbers →
same-code check.

In [ ]:
# A2 definitions -- the unrolled pieces of cmd_main and text_numbers.run,
# reused by A3 (the same command at 1.5 s)
from interacting_maps.eval_io import (load_groundtruth, load_imu, load_omega_gt,
                                      get_reference_omega, get_gyro_for_frame)
from config import get_dataset_paths
from report_runs import BETA_SEGMENTS, ADOPTED

DEG = np.degrees(1.0)
MAIN_COLS = ['sequence', 'segment', 'model', 'err', 'median', 'dir', 'beta',
             'recon', 'recon_inloop', 'blowup', 'floor', 'floor_dir', 'secs']


def mag(v):
    return np.linalg.norm(v, axis=-1)


def deg(a, b):
    """Angle between two vectors, degrees (text_numbers.deg)."""
    c = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


def reference_and_gyro(seg, dt, nf):
    """cmd_main: reference omega (scoring) and gyro (model input) per frame.
    In the code's unit, rad/s; every print converts with to_deg_s."""
    ds, _sid, t0, _lab = seg
    p = get_dataset_paths(ds)
    gt, imu = load_groundtruth(p['groundtruth']), load_imu(p['imu'])
    om = load_omega_gt(p.get('omega_gt'))
    ref = np.array([get_reference_omega(gt, imu, t0+k*dt, t0+(k+1)*dt, om)[0]
                    for k in range(nf)])
    gy = np.array([get_gyro_for_frame(imu, t0+k*dt, t0+(k+1)*dt) for k in range(nf)])
    return ref, gy


def ref_dt(p):
    """The frame length the reference is taken over: a form's dt (in Redraw, the dt
    of the runs it selects); for the old panel, 20 ms in Redraw, its dt in Live."""
    if p.get('form'):
        return p['dt']
    return 0.02 if p['mode'] == 'Redraw' else p['dt']


def stale_note(d):
    """After a row DIFF: which code made the run on disk."""
    pv = experiment_tracking.read_run(d).get('provenance', {})
    print(f"   run on disk: commit {str(pv.get('git_commit'))[:7]} of {pv.get('timestamp')}; "
          f"the csv was written by the run_all.sh job -- re-sync results/ if they disagree")


def gyro_floor(gy, ref):
    """cmd_main: mean |gyro - reference| (deg/s) and mean direction error (deg)."""
    floor = np.degrees(np.linalg.norm(gy-ref, axis=1)).mean()
    fdir = np.degrees(np.arccos(np.clip(np.einsum('ij,ij->i', gy, ref) /
           (np.linalg.norm(gy, axis=1)*np.linalg.norm(ref, axis=1)), -1, 1))).mean()
    return floor, fdir


def changed_deltas(p):
    """The panel's deltas that differ from the report's: cmd_main's --vary."""
    return {k: v for k, v in p['deltas'].items() if v != THESIS_PARAMS[k]}


def cook_deltas(p):
    """cmd_main passes --vary deltas to cook too; cook has only the relations of
    COOK_PARAMS, so a changed delta it lacks is dropped (and named)."""
    from config import COOK_PARAMS
    d = changed_deltas(p)
    lost = sorted(k for k in d if k not in COOK_PARAMS)
    if lost:
        print(f'cook has no {lost}; ignoring them')
    return {k: v for k, v in d.items() if k in COOK_PARAMS}


def main_cfg(p, nf, out_root):
    """cmd_main's run_cfg call, with the panel's values. cmd_main passes the
    --vary deltas and poisson only; deltas a model does not have are dropped."""
    ds, sid, t0, _lab = p['seg']
    deltas = cook_deltas(p) if p['model'] == 'cook' else changed_deltas(p)
    if p['n_iters'] != ITERS_PER_FRAME:
        print(f'cmd_main does not vary n_iters; the run uses {ITERS_PER_FRAME}')
    return run_cfg(ds, sid, t0, p['model'], p['dt'], nf, out_root=out_root,
                   poisson=p['poisson'], deltas=deltas)


def main_row(ds, sid, model, s, floor, fdir, secs=0):
    """cmd_main's csv row for one run summary, same columns and rounding."""
    return [ds, sid, model, round(s['mean_err_deg_s'], 2),
            round(s['median_err_deg_s'], 2), round(s['mean_dir_err_deg'], 2),
            round(s['mean_beta'], 3), round(s.get('recon_r_mean', float('nan')), 4),
            round(s.get('recon_r_mean_inloop', float('nan')), 4),
            round(s.get('blowup_frac', float('nan')), 4), round(floor, 2),
            round(fdir, 2), round(secs)]


def run_dir(seg, model, nf, d_fr=None, d_anchor='adopted', curl=0.0, root='results', dt=0.02):
    """text_numbers.run's lookup: the run folder whose stored config matches.
    root may be a list (a form's roots(): the first root with a match wins)."""
    if not isinstance(root, str):
        with contextlib.redirect_stdout(io.StringIO()):
            hits = [run_dir(seg, model, nf, d_fr, d_anchor, curl, r, dt) for r in root]
        d = next((h for h in hits if h), None)
        if d is None:
            print(f'   (missing run: {seg[0]}/{seg[1]} {model} n{nf})')
        return d
    ds, sid, t0 = seg[:3]
    prm = {'delta_FR': ADOPTED['delta_FR'] if d_fr is None else d_fr,
           'delta_shrinkI': 0.0, 'delta_map': 0.0, 'delta_curl': curl}
    if model in ('thesis_imu', 'thesis_cmax') and d_anchor is not None:
        prm['delta_IMU'] = ADOPTED['delta_IMU'] if d_anchor == 'adopted' else d_anchor
    d = experiment_tracking.find_run(root, ds, model, sid, nf, prm, dt=dt, t_start=t0)
    if d is None:
        print(f'   (missing run: {ds}/{sid} {model} n{nf} {prm})')
    return d


def tracking(d):
    """text_numbers.run's read: tracking.csv of one run folder as arrays."""
    with open(os.path.join(d, 'tracking.csv')) as f:
        rows = list(csv.DictReader(f))
    col = lambda *k: np.array([[float(r[c]) for c in k] for r in rows])
    return {'est': col('est_wx', 'est_wy', 'est_wz'), 'gt': col('gt_wx', 'gt_wy', 'gt_wz'),
            'imu': col('imu_wx', 'imu_wy', 'imu_wz'), 'err': col('err_deg_s')[:, 0],
            'dir': col('dir_err_deg')[:, 0], 'beta': col('beta')[:, 0],
            'r': np.abs(col('recon_r')[:, 0]), 'I_std': col('I_std')[:, 0]}


def run_tn(seg, model, nf=75, **kw):
    """text_numbers.run: the arrays of the matching run's tracking.csv, or None."""
    d = run_dir(seg, model, nf, **kw)
    return tracking(d) if d else None


from report_runs import SYNTHETIC
REAL = [s for s in SEGMENTS if s[3] not in SYNTHETIC]          # text_numbers.REAL / SYN
SYN = [s for s in SEGMENTS if s[3] in SYNTHETIC]


def main_knobs(nf):
    """The knobs of a cmd_main experiment (A2: 150 frames, A3: 75), at the report's values."""
    return [Knob('models', MODELS, 'many'), Knob('segment', 'poster'), Knob('n_frames', nf),
            Knob('dt', 0.02), Knob('poisson', POISSON_MODE),
            Knob('delta_FR', ADOPTED['delta_FR'])]


MAIN_FIXED = {'delta_curl': 0.0, 'delta_shrinkI': 0.0, 'delta_map': 0.0}   # cmd_main runs have none


def form_run_dir(p, model):
    """The saved run of `model` with the form's values -- every step size as cmd_main
    would set it (main_cfg), so a sweep run with another rate never stands in --
    by experiment_tracking.find_run; results/ first, then every other scanned root."""
    ds, sid, t0, _lab = p['seg']
    with contextlib.redirect_stdout(io.StringIO()):
        prm = main_cfg({**p, 'model': model}, p['n_frames'], None).params
    for root in roots():
        with contextlib.redirect_stdout(io.StringIO()):
            d = experiment_tracking.find_run(root, ds, model, sid, p['n_frames'], prm, dt=p['dt'],
                                             t_start=t0, n_iters=p['n_iters'], poisson=p['poisson'])
        if d:
            return d
    return None


def find_runs(p, ID, duration):
    """Step 'the run(s)', one per model of the form. Redraw: the saved runs the form
    selects. Live: saved_or_run (a saved configuration is shown, not rerun), into
    experiments/walkthrough in the real folder format. Sets READY[ID];
    returns ({model: run folder}, n_frames)."""
    READY[ID], dirs = False, {}                 # until the runs are there
    if p['seg'] is None or not p['models']:
        print(f'no saved run matches the {ID} form')
        return dirs, p['n_frames']
    ds, sid, t0, lab = p['seg']
    if not need([os.path.join('data', ds)]):    # run_cfg reads the dataset's IMU
        return dirs, p['n_frames']
    if p['mode'] == 'Redraw':
        print(f'Redraw: the saved runs the {ID} form selects')
        dirs = {m: form_run_dir(p, m) for m in p['models']}
        dirs = {m: d for m, d in dirs.items() if d is not None}
        if not dirs:
            print(f'no saved run matches the {ID} form')
        READY[ID] = bool(dirs)
    else:
        dirs = {m: saved_or_run(main_cfg({**p, 'model': m}, p['n_frames'], 'experiments/walkthrough'))
                for m in p['models']}
        READY[ID] = True
    for m, d in dirs.items():
        print(f'{m:16s} {d}')
    return dirs, p['n_frames']


def check_live_main(p, ID, nf, runs, rows):
    """Live same-code check: run cmd_main itself with its run step replaced by a
    recorder. Same RunConfig as main_cfg? Same csv row as main_row?"""
    import io, argparse
    seen = []
    def record(E_, rc, save_frames, stride):
        seen.append(rc)
        return experiment_tracking.read_summary(runs[p['model']]), 0
    mc = main_cfg(p, nf, None)
    vary = [f'{k}={v}' for k, v in changed_deltas(p).items()]   # the panel's, as typed
    if p['poisson'] != POISSON_MODE:
        vary.append(f"poisson={p['poisson']}")
    args = argparse.Namespace(dt=p['dt'], duration=nf*p['dt'], sequences=p['segment'],
                              models=p['model'], vary=';'.join(vary), out=f'{ID}_check.csv',
                              no_frames=True, frame_stride=1)
    out = os.path.join(walk_dir(ID), 'check')
    os.makedirs(out, exist_ok=True)
    said = io.StringIO()
    with redirect(run_experiments, _run=record, OUTDIR=out), contextlib.redirect_stdout(said):
        run_experiments.cmd_main(E, args)
    if not seen:                                # cmd_main catches its own errors
        print('[DIFF] config: cmd_main made no run; its last output:')
        for line in said.getvalue().strip().splitlines()[-5:]:
            print('   ' + line)
        return
    a, b = seen[0].to_dict(), mc.to_dict()
    differ = sorted(k for k in a if k != 'out_root' and a[k] != b[k])
    print('[ok] config' if not differ else f'[DIFF] config: {differ}')
    with open(os.path.join(out, f'{ID}_check.csv')) as f:
        got = list(csv.reader(f))[1]
    check('cmd_main row', dict(zip(MAIN_COLS[3:12], map(float, got[3:12]))),
          dict(zip(MAIN_COLS[3:12], rows[p['model']][3:12])))

In [ ]:
# the A2 form: Redraw browses saved runs, Live computes new ones
form('A2', main_knobs(150), fixed=MAIN_FIXED)

In [ ]:
# step 1 -- the run(s): found (Redraw) or made now (Live)
p = params('A2')
runs_A2, nf_A2 = find_runs(p, 'A2', duration=3.0)
rep_A2 = is_report('A2')                        # the report-only steps run only then

In [ ]:
%%section A2
# step 2 -- reference omega (Vicon / omega_gt) and the gyro, per 20 ms frame
ref_A2, gy_A2 = reference_and_gyro(p['seg'], ref_dt(p), nf_A2)
print('reference', ref_A2.shape, ' gyro', gy_A2.shape, ' (frames x xyz); first 4 frames, deg/s:')
print(pd.DataFrame(to_deg_s(np.hstack([ref_A2[:4], gy_A2[:4]])),
                   columns=['ref_x', 'ref_y', 'ref_z', 'gyro_x', 'gyro_y', 'gyro_z']).round(2))

In [ ]:
%%section A2
# step 3 -- the gyro floor: what the gyro scores against the reference
floor_A2, fdir_A2 = gyro_floor(gy_A2, ref_A2)
print(f'gyro floor {floor_A2:.2f} deg/s, direction {fdir_A2:.2f} deg')

In [ ]:
%%section A2
# step 4 -- one cmd_main row per run, from its run.json summary
rows_A2 = {m: main_row(*p['seg'][:2], m, experiment_tracking.read_summary(d), floor_A2, fdir_A2)
           for m, d in runs_A2.items()}
display(pd.DataFrame(rows_A2.values(), columns=MAIN_COLS).drop(columns='secs'))

In [ ]:
%%section A2
# step 5 -- the text numbers (report configuration only: they read the report's runs)
lines_A2 = {'poster_segments': [], 'inloop_identical': []}
if rep_A2:
    # text_numbers.poster_segments: fastest-segment gap, error relative to speed
    for seg in BETA_SEGMENTS:
        dc = run_dir(seg, 'thesis_cmax', 150, d_fr=0.1, d_anchor=0.5)   # BETA_CONFIGS series
        di = run_dir(seg, 'thesis_imu', 150) if seg[0] == 'poster_rotation' else None
        if dc is None:
            continue
        c, i = tracking(dc), tracking(di) if di else None
        w = mag(c['gt']).mean()
        floor = mag(c['imu'] - c['gt']).mean() * DEG
        line = (f'{seg[0][:6]} {seg[1]}: |w| {w*DEG:.1f} deg/s; CMax-anchor {c["err"].mean():5.2f} deg/s '
                f'= {100*c["err"].mean()/(w*DEG):4.1f}% of |w|; gyro floor {floor:4.2f} '
                f'= {100*floor/(w*DEG):4.1f}%')
        if i is not None:
            line += f'; thesis+IMU {i["err"].mean():5.2f}, gap {c["err"].mean()-i["err"].mean():4.2f}'
        lines_A2['poster_segments'].append(line)
    # text_numbers.inloop_identical, 3 s half: CMax-inloop omega at two delta_FR
    for seg in [b for b in BETA_SEGMENTS if b in [s[:3] for s in SEGMENTS]]:
        da = run_dir(seg, 'thesis_cmax_v2', 150, d_fr=0.5, d_anchor=None)
        db = run_dir(seg, 'thesis_cmax_v2', 150)
        if da and db:
            lines_A2['inloop_identical'].append(
                f'{seg[0][:6]} {seg[1]}: dFR 0.5 vs {ADOPTED["delta_FR"]} (3 s): '
                f'max |dw| = {np.abs(tracking(da)["est"]-tracking(db)["est"]).max()*DEG:.2e} deg/s')
    for k, ls in lines_A2.items():
        print(f'== {k}'); print('\n'.join(ls) or '(no runs)')
else:
    print('[skip] report-only step')

In [ ]:
%%section A2
# step 6 -- text_numbers.track_length: the 3 s runs cut after N frames (report configuration only).
# The network is causal, so their first 75 frames are the 1.5 s runs of A3.
L = lines_A2['track_length'] = []
if not rep_A2:
    print('[skip] report-only step')
else:
    Ns = (25, 50, 75, 100, 150)
    FIVE = ('cook', 'thesis', 'thesis_imu', 'thesis_cmax', 'thesis_cmax_v2')
    same = []
    for model in FIVE:
        L.append(model)
        for s in SEGMENTS:
            long_, short = run_tn(s, model, nf=150), run_tn(s, model)
            if long_ is None:
                continue
            if short is not None:
                same.append(np.array_equal(long_['err'][:75], short['err'])
                            and np.array_equal(long_['r'][:75], short['r']))
            L.append(f'  {s[3]:8s} ' + '  '.join(
                f'{n * 0.02:.1f}s {long_["err"][:n].mean():6.1f} {long_["r"][:n].mean():.2f}' for n in Ns))
    L.append(f'first 75 frames of the 3 s runs identical to the 1.5 s runs: {sum(same)} of {len(same)}')
    T_ = {(m, s[3]): run_tn(s, m, nf=150) for m in FIVE for s in SEGMENTS}
    if all(v is not None for v in T_.values()):
        Em = lambda m, lab, n: T_[m, lab]['err'][:n].mean()
        Rr = lambda m, lab, n: T_[m, lab]['r'][:n].mean()
        real, labs, anch = [s[3] for s in REAL], [s[3] for s in SEGMENTS], FIVE[2:]
        L.append('anchored (thesis_imu, thesis_cmax), largest change of the error from 1.5 to 3 s: '
                 f'{max(abs(Em(m, l, 150) - Em(m, l, 75)) for m in anch[:2] for l in labs):.1f} deg/s')
        order = lambda l, n: sorted(anch, key=lambda m: Em(m, l, n))
        L.append('order of the three anchored models the same at 0.5, 1.5 and 3 s on every sequence: '
                 + str(all(order(s[3], n) == order(s[3], 75) for s in SEGMENTS for n in (25, 150))))
        for n in (25, 75, 150):
            v = [Em(m, l, n) for m in FIVE[:2] for l in real]
            L.append(f'unanchored on the real sequences over {n * 0.02:g} s: {min(v):.1f}-{max(v):.1f} deg/s')
        for n in (25, 75):
            v = [T_[m, l]['dir'][:n].mean() for m in FIVE[:2] for l in real]
            L.append(f'unanchored direction error on the real sequences over {n * 0.02:g} s: {min(v):.0f}-{max(v):.0f} deg')
        L.append('unanchored error over 1.5 s as a share of the 3 s value, real sequences: at least '
                 f'{min(Em(m, l, 75) / Em(m, l, 150) for m in FIVE[:2] for l in real):.0%}')
        L.append('anchored read-out lowest over 0.5 s on every real sequence: '
                 + str(all(Rr(m, l, 25) < min(Rr(m, l, 75), Rr(m, l, 150)) for m in anch for l in real))
                 + f'; largest change from 1.5 to 3 s {max(abs(Rr(m, l, 150) - Rr(m, l, 75)) for m in anch for l in labs):.2f}')
        L.append('best read-out over 3 s per sequence: ' + ', '.join(
            f'{s[3]} {max(T_, key=lambda k: Rr(*k, 150) if k[1] == s[3] else -1)[0]} '
            f'{max(Rr(m, s[3], 150) for m in FIVE):.2f}' for s in SEGMENTS))
        wc = os.path.join('report', 'experiments', 'window_choice.csv')
        if os.path.exists(wc):
            with open(wc) as f:
                wrows = list(csv.DictReader(f))
            L.append('sign changes of the dominant axis of the reference (run_experiments window): ' + '; '.join(
                f'{lab} ' + ', '.join(f'{r["duration_s"]} s {r["sign_flips"]}' for r in wrows
                                      if r['sequence'] == lab and float(r['duration_s']) in (0.5, 1.5, 3.0))
                for lab in real))
    print('== track_length')
    for line in L:
        print(line)

In [ ]:
%%section A2
# same-code check -- against cmd_main (Live, Redraw) and text_numbers (Redraw)
import io
if p['mode'] == 'Live':
    for m_ in runs_A2:
        check_live_main({**p, 'model': m_}, 'A2', nf_A2, runs_A2, rows_A2)
if not rep_A2:
    print('[skip] not the report configuration')
else:
    # the rows against report/experiments/main_base_<seq>_<model>.csv
    for m, row in rows_A2.items():
        path = os.path.join('report', 'experiments', f'main_base_{p["segment"]}_{m}.csv')
        if need([path], [f'main_{p["segment"]}_{m}']):
            with open(path) as f:
                rep = list(csv.reader(f))[1]
            if not check(f'main_base row {m}', dict(zip(MAIN_COLS[3:12], row[3:12])),
                         dict(zip(MAIN_COLS[3:12], map(float, rep[3:12])))):
                stale_note(runs_A2[m])
    # the text numbers against text_numbers' own functions
    for name in lines_A2:
        with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
            getattr(text_numbers, name)()
            theirs = [l for l in text_numbers.OUT if not l.startswith(('==', '   (missing'))
                      and l and (name != 'inloop_identical' or '(3 s)' in l)]
        check(f'text_numbers.{name}', nums(lines_A2[name]), nums(theirs))

<a id="A3"></a>

## A3 · Main table (Table III), 1.5 s · [↑ overview](#overview)

**Question.** Every model on every Table II segment over 1.5 s (75 frames of
20 ms): ω error, direction error, β and |r|. This is the report's main result.

**Made by** `report/run_experiments.py::cmd_main` with `--duration 1.5`,
`run_all.sh` jobs `dissoc_<seq>_<model>` → runs in `results/`, one row each in
`report/experiments/dissoc_1p5s_<seq>_<model>.csv`. The steps up to the row are
those of A2 (its definitions cell is reused).

**Read by** `report/make_tables.py::table_main` (tab:main),
`report/make_figures.py::fig_tracking` (fig:tracking), and
`report/text_numbers.py::unanchored` / `::reversal`. Also read, but unrolled
where their second input is: `fig_dissociation`, `fig_maps`/`fig_appendix`
(A13) and `fig_image_settings` (A12); the network column of `table_baseline`
(A5).

In [ ]:
# the A3 form: Redraw browses saved runs, Live computes new ones
form('A3', main_knobs(75), fixed=MAIN_FIXED)

In [ ]:
# steps 1-4 of A2 at 1.5 s: the run(s), reference and gyro, floor, the rows
p = params('A3')
runs_A3, nf_A3 = find_runs(p, 'A3', duration=1.5)
rep_A3 = is_report('A3')                        # the report-only steps run only then

In [ ]:
%%section A3
ref_A3, gy_A3 = reference_and_gyro(p['seg'], ref_dt(p), nf_A3)
floor_A3, fdir_A3 = gyro_floor(gy_A3, ref_A3)
rows_A3 = {m: main_row(*p['seg'][:2], m, experiment_tracking.read_summary(d), floor_A3, fdir_A3)
           for m, d in runs_A3.items()}
print(f'gyro floor {floor_A3:.2f} deg/s, direction {fdir_A3:.2f} deg')
display(pd.DataFrame(rows_A3.values(), columns=MAIN_COLS).drop(columns='secs'))

### tab:main -- `make_tables.py::table_main`

Reads every `dissoc_1p5s_*.csv` (all sequences, all models), keeps four
columns, marks the best model per sequence in bold (lowest err and dir, β
closest to 1, highest |r|) and adds the gyro floor as a last row. The speed
under each sequence name comes from the A5 baseline npz.

In [ ]:
%%section A3
# step 5 -- table_main, unrolled (fmt / mark_best / tabular are its formatters).
# It reads the report's csv rows of every segment: report configuration only.
from make_tables import fmt, mark_best, tabular, floor_label, SEQS, MODELS as TAB_MODELS
csv_rows = {}
if not rep_A3:
    print('[skip] report-only step')
for path in sorted(glob.glob(os.path.join('report', 'experiments', 'dissoc_1p5s_*.csv'))) if rep_A3 else []:
    with open(path) as f:
        for r in csv.DictReader(f):
            csv_rows[(r['sequence'], r['model'])] = r
print(f'{len(csv_rows)} rows from dissoc_1p5s_*.csv')

def seq_speed(lab):
    """mean |omega| of the reference over the segment, deg/s, from the A5 npz"""
    path = os.path.join('report', 'experiments', f'baseline_dt20ms_{lab}.npz')
    if not need([path], ['baseline']):
        return None
    return float(to_deg_s(np.linalg.norm(np.load(path)['ref'], axis=1).mean()))

body, cells = [], []
for seq, disp, lab, extra in SEQS:
    sp = seq_speed(lab)
    head_ = [disp] + ([f'${sp:.0f}$\\,$^\\circ$/s'] if sp else []) + extra
    body += [r'\midrule', r'\multirow{6}{*}{\shortstack[l]{' + r'\\'.join(head_) + '}}']
    floor = None
    cols = {c: {} for c in ('err', 'dir', 'beta', 'recon')}
    for mdisp, key in TAB_MODELS:
        r = csv_rows.get((seq, key))
        if r is None:
            continue
        floor = (r['floor'], r['floor_dir'])
        cols['err'][key], cols['dir'][key] = fmt(r['err']), fmt(r['dir'])
        cols['beta'][key] = fmt(r['beta'], 0 if float(r['beta']) >= 100 else 2)
        cols['recon'][key] = fmt(r['recon'], 2)
        cells.append([lab, key, r['err'], r['dir'], r['beta'], r['recon']])
    cols = {'err': mark_best(cols['err'], lambda v: v),
            'dir': mark_best(cols['dir'], lambda v: v),
            'beta': mark_best(cols['beta'], lambda v: abs(v - 1.0)),
            'recon': mark_best(cols['recon'], lambda v: -v)}
    for mdisp, key in TAB_MODELS:
        if key not in cols['err']:
            body.append(f' & {mdisp:<22}& --- & --- & --- & --- \\\\')
            continue
        body.append(f' & {mdisp:<22}& {cols["err"][key]} & {cols["dir"][key]} & '
                    f'{cols["beta"][key]} & {cols["recon"][key]} \\\\')
    if floor:
        nd = 2 if float(floor[0]) < 1 else 1
        body.append(f' & {floor_label(seq):<22}& \\emph{{{fmt(floor[0], nd)}}} & '
                    f'\\emph{{{fmt(floor[1])}}} & --- & --- \\\\')
table_A3 = tabular('@{}llrrrr@{}',
                   [r'Sequence & Model & err ($^\circ$/s) & dir ($^\circ$) & $\beta$ & $|r|$ \\'],
                   body)
if rep_A3:
    display(pd.DataFrame(cells, columns=['sequence', 'model', 'err', 'dir', 'beta', 'recon']))
    print(table_A3)

### fig:tracking -- `make_figures.py::fig_tracking`

ω per axis over the 1.5 s of poster seg_C: CMax-anchor and thesis against the
reference. Redraw draws exactly the report's figure (its segment and models are
fixed in the code); Live draws the panel's run against its own reference.

In [ ]:
%%section A3
# step 6 -- fig_tracking, unrolled
from interacting_maps.style import use_style, save, frame_times, PAPER_WIDTH
from interacting_maps.viz import draw_omega
if rep_A3:
    seg_fig = ('poster_rotation', 'seg_C', 8.816)
    pp = get_dataset_paths(seg_fig[0])
    ref_fig, _ = reference_and_gyro(seg_fig + ('poster',), 0.02, 75)
    ref_src = get_reference_omega(load_groundtruth(pp['groundtruth']), load_imu(pp['imu']),
                                  8.816, 8.836, load_omega_gt(pp.get('omega_gt')))[1]
    fig_runs = [(m, run_dir(seg_fig, m, 75)) for m in ('thesis_cmax', 'thesis')]
else:
    fig_runs = list(runs_A3.items())
    ref_fig, ref_src = tracking(fig_runs[0][1])['gt'], 'groundtruth'
t = frame_times(0.02 if rep_A3 else p['dt'], len(ref_fig))

with use_style('paper'):
    fig_A3, axes = plt.subplots(3, 1, figsize=(PAPER_WIDTH, 3.6), sharex=True)
    first = True
    for model, d in fig_runs:
        if d is None:
            continue
        est = tracking(d)['est'][:len(ref_fig)]
        for i in range(3):
            draw_omega(axes[i], t[:len(est)], est[:, i], axis=i, model=model,
                       ref=ref_fig[:len(est), i] if first else None, ref_src=ref_src)
        first = False
    for ax in axes:
        ax.label_outer()
    fig_A3.legend(*axes[0].get_legend_handles_labels(), loc='outside upper center', ncol=3)
    lines_fig = [l.get_ydata() for ax in fig_A3.axes for l in ax.lines]
    if first:                                   # no run was found: nothing drawn
        plt.close(fig_A3)
        print('fig_tracking: its runs are missing (jobs dissoc_poster_thesis, dissoc_poster_thesis_cmax)')
    else:
        save(fig_A3, os.path.join(walk_dir('A3'), 'fig_tracking.png'), show=True)

In [ ]:
%%section A3
# step 7 -- the text numbers (report configuration only: they read the report's runs)
from report_runs import SYNTHETIC
REAL = [s for s in SEGMENTS if s[3] not in SYNTHETIC]
SYN = [s for s in SEGMENTS if s[3] in SYNTHETIC]
lines_A3 = {'unanchored': [], 'reversal': []}
if rep_A3:
    L = lines_A3['unanchored']          # text_numbers.unanchored: cook and thesis
    for seg in REAL + SYN:
        for m in ('cook', 'thesis'):
            d = run_dir(seg, m, 75)
            if d is None:
                continue
            tr = tracking(d)
            e = tr['est']
            m0 = np.linalg.norm(e[0])
            L.append(f'{seg[3]:8s} {m:7s} |w| within {100*np.max(np.abs(mag(e)/m0 - 1)):5.1f}% '
                     f'of frame 0, direction within {max(deg(x, e[0]) for x in e):5.1f} deg; '
                     f'frame 0 |w|/gyro = {m0/np.linalg.norm(tr["imu"][0]):.3f}; '
                     f'last/first |w| = {mag(e)[-1]/m0:.3f}')
        d = run_dir(seg, 'thesis', 75)
        if d is not None:
            g = tracking(d)
            L.append(f'{seg[3]:8s} reference: axis turns up to {max(deg(x, g["gt"][0]) for x in g["gt"]):5.1f} '
                     f'deg, speed {mag(g["gt"]).min()*DEG:.1f}-{mag(g["gt"]).max()*DEG:.1f} deg/s')
    L = lines_A3['reversal']            # text_numbers.reversal: worst CMax-anchor frame
    for seg in REAL:
        dc, di = run_dir(seg, 'thesis_cmax', 75), run_dir(seg, 'thesis_imu', 75)
        if dc is None or di is None:
            continue
        c, i = tracking(dc), tracking(di)
        k = int(np.argmax(c['err']))
        keep = np.arange(len(c['err'])) != k
        dom = int(np.argmax(np.abs(c['gt']).mean(0)))     # dominant rotation axis
        near = c['gt'][max(k - 3, 0):k + 4, dom]
        flip = bool((np.diff(np.sign(near)) != 0).any())
        L.append(f'{seg[3]:8s} frame {k}: |w_ref| = {mag(c["gt"])[k]*DEG:.1f} deg/s '
                 f'({"dominant axis reverses within 3 frames" if flip else "no reversal within 3 frames"}; '
                 f'{int((mag(c["gt"]) <= 0.2).sum())} frames of the track at <= {0.2*DEG:.1f} deg/s); '
                 f'err CMax-anchor {c["err"][k]:6.1f}, thesis+IMU {i["err"][k]:4.1f} deg/s; '
                 f'mean without it: CMax-anchor {c["err"][keep].mean():5.2f}, '
                 f'thesis+IMU {i["err"][keep].mean():5.2f}, gap {c["err"][keep].mean()-i["err"][keep].mean():5.2f} '
                 f'(all frames: gap {c["err"].mean()-i["err"].mean():5.2f})')
    for k_, ls in lines_A3.items():
        print(f'== {k_}'); print('\n'.join(ls) or '(no runs)')
else:
    print('[skip] report-only step')

In [ ]:
%%section A3
# step 8 -- text_numbers.reference: the gyro against a one-frame and a +-50 ms reference.
# No runs: ground truth and IMU only, 75 frames of 20 ms on every Table II and beta segment.
from interacting_maps import eval_io as eio_A3
L = lines_A3['reference'] = []
if not rep_A3:
    print('[skip] report-only step')
else:
    segs = REAL + [s + ('',) for s in BETA_SEGMENTS if s not in [r[:3] for r in REAL]] + SYN
    for seg in segs:
        ds_, sid_, t0_ = seg[:3]
        pp = get_dataset_paths(ds_)
        if not os.path.exists(pp['imu']):
            print(f'missing: {pp["imu"]}')
            continue
        imu, gt, om = (eio_A3.load_imu(pp['imu']), eio_A3.load_groundtruth(pp['groundtruth']),
                       eio_A3.load_omega_gt(pp.get('omega_gt')))
        one, wide = [], []
        for k in range(75):
            lo, hi = t0_ + k * 0.02, t0_ + (k + 1) * 0.02
            g = eio_A3.get_gyro_for_frame(imu, lo, hi)
            if om is not None:                      # exact omega over the frame itself
                msk = (om[:, 0] >= lo) & (om[:, 0] < hi)
                ref1 = om[msk, 1:4].mean(0) if msk.any() else eio_A3.omega_gt_at(om, lo, hi)
            else:
                ref1 = eio_A3.gt_omega_body(gt, lo, hi)
            one.append(np.linalg.norm(g - ref1) * DEG)
            wide.append(np.linalg.norm(g - eio_A3.get_reference_omega(gt, imu, lo, hi, om)[0]) * DEG)
        L.append(f'{ds_[:14]:14s} {sid_}: gyro vs one-frame reference {np.mean(one):5.2f} deg/s, '
                 f'vs +-50 ms {np.mean(wide):5.2f}')
    print('== reference'); [print(l) for l in L]

In [ ]:
%%section A3
# step 9 -- text_numbers.intensity, its first part: I std and |r| along the poster track
# of the anchored and the unanchored network (the rest of that function reads A8, A1, A16)
L = lines_A3['intensity'] = []
if not rep_A3:
    print('[skip] report-only step')
else:
    FIVE = ('cook', 'thesis', 'thesis_imu', 'thesis_cmax', 'thesis_cmax_v2')
    poster = [s for s in SEGMENTS if s[3] == 'poster'][0]
    c, t_ = run_tn(poster, 'thesis_cmax'), run_tn(poster, 'thesis')
    if c is not None and t_ is not None:
        k = int(np.argmax(c['r']))
        L.append(f'CMax-anchor I std {c["I_std"].min():.2f}-{c["I_std"].max():.2f}; '
                 f'thesis (no anchor) {t_["I_std"].min():.2f}-{t_["I_std"].max():.2f}')
        b = c['I_std'] > 5 * np.median(c['I_std'])
        L.append(f'CMax-anchor omega error on the {b.sum()} frames with I std above 5x its median '
                 f'{c["err"][b].mean():.1f} deg/s, on the other {(~b).sum()} frames {c["err"][~b].mean():.1f} deg/s')
        L.append(f'CMax-anchor I std: median {np.median(c["I_std"]):.2f}, above 5x the median at frames '
                 f'{np.where(b)[0].min()}-{np.where(b)[0].max()}, peak {c["I_std"].max():.2f} at frame {int(np.argmax(c["I_std"]))}')
        L.append(f'CMax-anchor |r|: highest at frame {k} ({c["r"][k]:.2f}), track mean {c["r"].mean():.2f}')
    L.append('frames with I std above 5x its median, per model (Table main runs, 1.5 s):')
    for s in SEGMENTS:
        n = {m: run_tn(s, m) for m in FIVE}
        L.append(f'  {s[3]:8s} ' + ', '.join(f'{m} {int((r["I_std"] > 5 * np.median(r["I_std"])).sum())}/{len(r["I_std"])}'
                                            for m, r in n.items() if r is not None))
    allr = [run_tn(poster, m) for m in FIVE]
    if all(a is not None for a in allr):
        mr = np.mean([a['r'] for a in allr], axis=0)
        L.append(f'mean |r| over the five models: highest at frame {int(np.argmax(mr))}')
    print('== intensity (first part)'); [print(l) for l in L]

### fig:dissociation -- `make_figures.py::fig_dissociation`

The read-out (fft solve of the final G) of the Table III runs of thesis,
CMax-anchor and thesis+IMU on boxes, bicycle and street, next to the APS frame;
each labelled with its track-mean |r| and ω error. The run folder is the one
`run_cfg` names (not a search).

In [ ]:
%%section A3
# step 10 -- fig_dissociation, unrolled (Redraw: the report's runs)
from interacting_maps.network_dissertation import solve_poisson_exact
from interacting_maps.viz import draw_map
from report_runs import aps_frame, track_stats
fig_dis_A3 = None
if rep_A3:
    D = make_figures.DISSOC
    nf_ = int(round(D['duration'] / 0.02))
    seqs = [s for s in SEGMENTS if s[3] in D['sequences']]
    with use_style('paper'):
        fig_dis_A3, axes = plt.subplots(len(seqs), len(D['models']) + 1,
                                        figsize=make_figures.grid_size(len(seqs), len(D['models']) + 1,
                                                                       below=1, label_lines=1))
        axes = np.atleast_2d(axes)
        for r_i, (ds_, sid_, t0_, lab) in enumerate(seqs):
            if not need([os.path.join('data', ds_)]):     # run_cfg reads the dataset's IMU
                for a in axes[r_i]:
                    a.set_axis_off()
                continue
            panels = [('APS', aps_frame(ds_, t0_ + nf_ * 0.02), None, 'aps')]
            for m in D['models']:
                d = run_cfg(ds_, sid_, t0_, m, 0.02, nf_).output_dir
                npz = os.path.join(d, 'maps_final.npz')
                img = st = None
                if need([npz], [f'dissoc_{lab}_{m}']):
                    img = solve_poisson_exact(np.load(npz)['G'], 'fft')   # the fft read-out
                    st = track_stats(d)                                    # (track mean |r|, err deg/s)
                panels.append((make_figures.MODELS[m][0], img, st, 'I'))
            for c_i, (title, img, st, kind) in enumerate(panels):
                a = axes[r_i, c_i]
                if img is None:
                    a.set_axis_off()
                    continue
                draw_map(a, kind, img, title=title if r_i == 0 else '')
                if c_i == 0:
                    a.set_ylabel(lab)
                if st is not None:
                    a.set_xlabel(f"$|r|={st[0]:.2f}$,  $\\omega$ err $={st[1]:.1f}$ °/s")
        save(fig_dis_A3, os.path.join(walk_dir('A3'), 'fig_dissociation.png'), show=True)
else:
    print('[skip] report-only step')

In [ ]:
%%section A3
# same-code check
import io
if p['mode'] == 'Live':
    for m_ in runs_A3:
        check_live_main({**p, 'model': m_}, 'A3', nf_A3, runs_A3, rows_A3)
if not rep_A3:
    print('[skip] not the report configuration')
else:
    for m, row in rows_A3.items():      # the rows against dissoc_1p5s_<seq>_<model>.csv
        path = os.path.join('report', 'experiments', f'dissoc_1p5s_{p["segment"]}_{m}.csv')
        if need([path], [f'dissoc_{p["segment"]}_{m}']):
            with open(path) as f:
                rep = list(csv.reader(f))[1]
            if not check(f'dissoc row {m}', dict(zip(MAIN_COLS[3:12], row[3:12])),
                         dict(zip(MAIN_COLS[3:12], map(float, rep[3:12])))):
                stale_note(runs_A3[m])
    check('table_main', nums(table_A3.splitlines()),
          nums(make_tables.table_main().splitlines()))
    captured = []
    def keep(fig, path, show=False):
        captured.append(fig)
    with redirect(make_figures, FIG=os.path.join(walk_dir('A3'), 'check'), save=keep), \
         contextlib.redirect_stdout(io.StringIO()):
        make_figures.fig_tracking(E)
    theirs = [l.get_ydata() for ax in captured[0].axes for l in ax.lines] if captured else []
    for f_ in captured:
        plt.close(f_)
    flat = lambda ls: np.concatenate(ls) if ls else np.array([])
    check('fig_tracking lines', flat(lines_fig), flat(theirs))
    captured = []
    dis_data = [os.path.join('data', s[0]) for s in SEGMENTS if s[3] in make_figures.DISSOC['sequences']]
    if all(os.path.exists(d) for d in dis_data):     # fig_dissociation needs every dataset
        with redirect(make_figures, FIG=os.path.join(walk_dir('A3'), 'check'), save=keep), \
             contextlib.redirect_stdout(io.StringIO()):
            make_figures.fig_dissociation(E)
        imgs = lambda fig: [np.asarray(im.get_array(), float).ravel()
                            for ax in fig.axes for im in ax.images] if fig else []
        check('fig_dissociation images', flat(imgs(fig_dis_A3)), flat(imgs(captured[0] if captured else None)))
    else:
        print('[skip] fig_dissociation: datasets missing here: '
              + ', '.join(d for d in dis_data if not os.path.exists(d)))
    for f_ in captured:
        plt.close(f_)
    for name in lines_A3:
        try:
            with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
                getattr(text_numbers, name)()
                theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
        except FileNotFoundError as e:          # the report's function needs a dataset absent here
            print(f'[skip] text_numbers.{name}: {e}')
            continue
        if name == 'intensity':                 # only its first part is unrolled here
            cut = [i for i, l in enumerate(theirs) if l.startswith('curl share')]
            theirs = theirs[:cut[0]] if cut else theirs
        check(f'text_numbers.{name}', nums(lines_A3[name]), nums(theirs))

<a id="A4"></a>

## A4 · Curl regulariser (tab:curl), 1.5 s · [↑ overview](#overview)

**Question.** What does the integrability prior on G, `delta_curl = 0.6`, change
in every model over 1.5 s: ω error and read-out |r|, against the runs of
Table III (A3) without it?

**Made by** `report/run_experiments.py::cmd_main` with `--duration 1.5 --vary
delta_curl=0.6`, `run_all.sh` jobs `curl_<seq>_<model>` → runs in `results/`,
one row each in `report/experiments/main_curl_<seq>_<model>.csv`. The steps up
to the row are those of A2 (its definitions cell is reused) with `delta_curl`
at 0.6.

**Read by** `report/make_tables.py::table_curl` (tab:curl; its *none* half is
the A3 csvs) and the 1.5 s half of `report/text_numbers.py::inloop_identical`
(quoted in the text; its 3 s half is A2). Also read, but unrolled where their
other input is: `make_tables.table_image` (A12) and the curl-share lines of
`text_numbers.regularisers` (A11).

In [ ]:
# the A4 form: as A3, with the curl step size (report: 0.6)
form('A4', main_knobs(75) + [Knob('delta_curl', 0.6)],
     fixed={'delta_shrinkI': 0.0, 'delta_map': 0.0})

In [ ]:
# step 1 -- the runs with the form's delta_curl: found (Redraw) or made now (Live)
p = params('A4')
runs_A4, nf_A4 = find_runs(p, 'A4', duration=1.5)
q_A4, rep_A4 = p, is_report('A4')               # the report-only steps run only then

In [ ]:
%%section A4
# steps 2-4 of A2 (cmd_main): reference and gyro, gyro floor, one row per run
ref_A4, gy_A4 = reference_and_gyro(q_A4['seg'], ref_dt(q_A4), nf_A4)
floor_A4, fdir_A4 = gyro_floor(gy_A4, ref_A4)
rows_A4 = {m: main_row(*q_A4['seg'][:2], m, experiment_tracking.read_summary(d), floor_A4, fdir_A4)
           for m, d in runs_A4.items()}
print(f'gyro floor {floor_A4:.2f} deg/s, direction {fdir_A4:.2f} deg')
display(pd.DataFrame(rows_A4.values(), columns=MAIN_COLS).drop(columns='secs'))

### tab:curl -- `make_tables.py::table_curl`

Two inputs: the `dissoc_1p5s_*.csv` rows (A3, *none*) and the `main_curl_*_*.csv`
rows (this section, *curl*), keyed by (sequence, model). Per sequence it shows
err and |r| of every model with and without the curl term, and marks in bold the
lowest err and the highest |r| among the ten runs of that sequence.

In [ ]:
%%section A4
# step 5 -- table_curl, unrolled (load / fmt / mark_best / tabular are its helpers)
# It reads the report's csv rows of every segment: report configuration only.
from make_tables import load, fmt, mark_best, tabular, SEQS, MODELS as TAB_MODELS
none_A4, curl_A4 = (load(['dissoc_1p5s_*.csv']), load(['main_curl_*_*.csv'])) if rep_A4 else ({}, {})
if not rep_A4:
    print('[skip] report-only step')
print(f'{len(none_A4)} rows without curl (dissoc_1p5s_*.csv), {len(curl_A4)} with (main_curl_*_*.csv)')


def table_curl_A4(none, curl):
    """make_tables.table_curl on the two row sets: (LaTeX, the numbers shown)."""
    body, cells = [], []
    for ds, disp, lab, _extra in SEQS:
        body.append(r'\midrule')
        g = lambda r, k, nd: fmt(r[k], nd) if r else '---'
        # bold: the lowest err and the highest |r| of the ten runs of this sequence
        err = mark_best({(key, w): g(src.get((ds, key)), 'err', 1) for _, key in TAB_MODELS
                         for w, src in (('none', none), ('curl', curl))}, lambda v: v)
        rec = mark_best({(key, w): g(src.get((ds, key)), 'recon', 2) for _, key in TAB_MODELS
                         for w, src in (('none', none), ('curl', curl))}, lambda v: -v)
        for j, (mdisp, key) in enumerate(TAB_MODELS):
            body.append(f'{disp if j == 0 else "":<22}& {mdisp:<22}& '
                        f'{err[key, "none"]} & {err[key, "curl"]} & '
                        f'{rec[key, "none"]} & {rec[key, "curl"]} \\\\')
            a, b = none.get((ds, key)), curl.get((ds, key))
            cells.append([lab, key, a and a['err'], b and b['err'], a and a['recon'], b and b['recon']])
    head = [r' & & \multicolumn{2}{c}{$\wvec$ err [$^\circ$/s]} & \multicolumn{2}{c}{$|r|$} \\',
            r'\cmidrule(lr){3-4}\cmidrule(lr){5-6}',
            r'Sequence & Model & none & curl & none & curl \\']
    return tabular('@{}llrrrr@{}', head, body), cells


table_A4, cells_A4 = table_curl_A4(none_A4, curl_A4)
if rep_A4:
    display(pd.DataFrame(cells_A4, columns=['sequence', 'model', 'err none (deg/s)', 'err curl (deg/s)',
                                            '|r| none', '|r| curl']))
    print(table_A4)

In [ ]:
%%section A4
# step 6 -- text_numbers.py::inloop_identical, 1.5 s half (report configuration only:
# it reads the report's runs). CMax-inloop omega with and without the curl term: the maps
# change, the anchor's omega should not.
lines_A4 = []
if rep_A4:
    for seg in SEGMENTS:
        da = run_dir(seg, 'thesis_cmax_v2', 75)
        db = run_dir(seg, 'thesis_cmax_v2', 75, curl=0.6)
        if da and db:
            lines_A4.append(f'{seg[3]:8s} curl 0 vs 0.6 (1.5 s): max |dw| = '
                            f'{np.abs(tracking(da)["est"]-tracking(db)["est"]).max()*DEG:.2e} deg/s')
    print('== inloop_identical (1.5 s)'); print('\n'.join(lines_A4) or '(no runs)')
else:
    print('[skip] report-only step')

In [ ]:
%%section A4
# same-code check
import io
if p['mode'] == 'Live':                         # cmd_main --vary delta_curl=...
    for m_ in runs_A4:
        check_live_main({**q_A4, 'model': m_}, 'A4', nf_A4, runs_A4, rows_A4)
if not rep_A4:
    print('[skip] not the report configuration')
else:
    for m, row in rows_A4.items():      # the rows against main_curl_<seq>_<model>.csv
        path = os.path.join('report', 'experiments', f'main_curl_{p["segment"]}_{m}.csv')
        if need([path], [f'curl_{p["segment"]}_{m}']):
            with open(path) as f:
                rep = list(csv.reader(f))[1]
            if not check(f'main_curl row {m}', dict(zip(MAIN_COLS[3:12], row[3:12])),
                         dict(zip(MAIN_COLS[3:12], map(float, rep[3:12])))):
                stale_note(runs_A4[m])
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        text_numbers.inloop_identical()
        theirs = [l for l in text_numbers.OUT if '(1.5 s)' in l]
    check('text_numbers.inloop_identical', nums(lines_A4), nums(theirs))
    check('table_curl', nums(table_A4.splitlines()), nums(make_tables.table_curl().splitlines()))

<a id="A5"></a>

## A5 · Anchor alone (baseline) · [↑ overview](#overview)

**Question.** What do the two anchor signals score on their own, with no
network? CMax-anchor feeds the network a per-frame CMax ω, thesis+IMU feeds it
the gyro; if the network cannot beat its own anchor, the maps add nothing to ω.

**Made by** `report/run_experiments.py::cmd_baseline` (`--duration 1.5`),
`run_all.sh` job `baseline` → `report/experiments/baseline_dt20ms.csv` (one row
per sequence and source, cmax / gyro) and `baseline_dt20ms_<seq>.npz` (the
per-frame ω series of CMax, gyro and reference, and the CMax IWE at the first,
middle and last frame).

**Read by** `report/make_tables.py::table_baseline` (tab:baseline; its
*network* column is the CMax-anchor row of the A3 csvs `dissoc_1p5s_*`),
`report/make_figures.py::fig_cmax_baseline` (**not in the tex**: drawn, but no
`\includegraphics` uses it) and `report/text_numbers.py::maps_add`. The npz
reference also gives the speed under each sequence name of tab:main (A3).

**Steps** (those of `cmd_baseline`'s per-frame loop, then of the readers): the
segment's events and the CMax estimator → per frame: reference, CMax with a warm
start from the previous frame, gyro → `compute_metrics` per frame and source →
the csv rows → table, figure, text → same-code check. Redraw shows the stored
series (it does not rerun CMax); Live runs the loop now on the panel's segment.
The panel's model and step sizes do not enter: there is no network here.

In [ ]:
# step 1 -- the inputs: the stored baseline (Redraw) or the segment's data (Live)
import time
p = params()
ds_A5, sid_A5, t0_A5, lab_A5 = p['seg']
BASE_A5 = os.path.join('report', 'experiments')
if p['mode'] == 'Redraw':
    note_redraw(p, 75)
    nf_A5, dt_A5, src_A5 = 75, 0.02, BASE_A5
    READY['A5'] = need([os.path.join(BASE_A5, 'baseline_dt20ms.csv'),
                        os.path.join(BASE_A5, f'baseline_dt20ms_{lab_A5}.npz')], ['baseline'])
else:
    nf_A5, dt_A5, src_A5 = min(p['n_frames'], 75), p['dt'], walk_dir('A5')
    print(f"Live: cmd_baseline's loop on {lab_A5}, {nf_A5} frames of {dt_A5*1000:g} ms "
          f"(no network: the panel's model and step sizes are not used)")
    READY['A5'] = need([os.path.join('data', ds_A5)])
csv_A5 = os.path.join(src_A5, f'baseline_dt{int(dt_A5*1000)}ms.csv')
npz_A5 = csv_A5.replace('.csv', f'_{lab_A5}.npz')
print(f'series: {npz_A5}')

In [ ]:
%%section A5
# step 2 -- run_experiments.py::cmd_baseline: the segment's events and the CMax estimator
from interacting_maps.data_loader import load_events_fast, undistort_events, CameraCalibration
from interacting_maps.angular_velocity import CMaxAngularVelocity
if p['mode'] == 'Live':
    pp_A5 = E.get_dataset_paths(ds_A5)
    calib_A5 = CameraCalibration(pp_A5['calib'])
    gt_A5, imu_A5 = E.load_groundtruth(pp_A5['groundtruth']), E.load_imu(pp_A5['imu'])
    om_A5 = E.load_omega_gt(pp_A5.get('omega_gt'))
    # the same undistorted events experiment_tracking gives CMax
    ev_A5 = undistort_events(load_events_fast(pp_A5['events'], t_start=t0_A5,
                                              duration=nf_A5 * dt_A5 + 0.1), calib_A5)
    est_A5 = CMaxAngularVelocity(180, 240, calib_A5.fx, calib_A5.fy,
                                 calib_A5.cx, calib_A5.cy, use_polarity=True)
    print(f'{len(ev_A5)} undistorted events from t = {t0_A5} s over {nf_A5*dt_A5 + 0.1:.2f} s; '
          f'CMax on a 180 x 240 sensor, polarity used')
else:
    print(f'Redraw: the events were read when the baseline job ran; '
          f'the series it stored are in {npz_A5}')

In [ ]:
%%section A5
# step 3 -- cmd_baseline's per-frame loop: reference, CMax (warm start), gyro; the npz
if p['mode'] == 'Live':
    w_prev = np.zeros(3)                                # CMax starts from zero at frame 0
    keep_A5 = (0, nf_A5 // 2, nf_A5 - 1)                # frames whose IWE is saved
    ser_A5, iwes = {'cmax': [], 'gyro': [], 'ref': []}, []
    t_wall = time.time()
    for k in range(nf_A5):
        t_lo, t_hi = t0_A5 + k * dt_A5, t0_A5 + (k + 1) * dt_A5
        w_ref, ref_src_A5 = E.get_reference_omega(gt_A5, imu_A5, t_lo, t_hi, om_A5)
        win = ev_A5[(ev_A5[:, 0] >= t_lo) & (ev_A5[:, 0] < t_hi)]
        if len(win) > 10:                               # warm start: last frame's CMax
            w_prev = est_A5.estimate(win, t_ref=0.5 * (t_lo + t_hi), omega_init=w_prev)
        w_gyro = E.get_gyro_for_frame(imu_A5, t_lo, t_hi)
        for src, w in (('cmax', w_prev), ('gyro', w_gyro), ('ref', w_ref)):
            ser_A5[src].append(w)
        if k in keep_A5:
            iwes.append(est_A5.last_iwe if len(win) > 10 else np.zeros((180, 240)))
    secs_A5 = time.time() - t_wall
    np.savez_compressed(npz_A5, frames=keep_A5, iwe=np.array(iwes), ref_src=ref_src_A5,
                        **{s: np.array(v) for s, v in ser_A5.items()})
    print(f'{nf_A5} frames in {secs_A5:.0f} s -> {npz_A5}')
else:
    secs_A5 = None
d_A5 = np.load(npz_A5)
series_A5 = {s: d_A5[s] for s in ('cmax', 'gyro', 'ref')}
print(f"series (frames x xyz): {series_A5['ref'].shape}, reference from {d_A5['ref_src']}; "
      f"IWE kept at frames {d_A5['frames'].tolist()}. First 4 frames, deg/s:")
print(pd.DataFrame(to_deg_s(np.hstack([series_A5[s][:4] for s in ('ref', 'cmax', 'gyro')])),
                   columns=[f'{s}_{a}' for s in ('ref', 'cmax', 'gyro') for a in 'xyz']).round(2))

In [ ]:
%%section A5
# step 4 -- compute_metrics per frame and source (evaluation.compute_metrics), then the csv rows
BL_COLS_A5 = ['err', 'median', 'dir', 'beta']
acc_A5 = {src: np.array([E.compute_metrics(w, r) for w, r in zip(series_A5[src], series_A5['ref'])])
          for src in ('cmax', 'gyro')}                  # (n, 3): err deg/s, dir deg, beta
rows_A5 = {src: dict(zip(BL_COLS_A5, (round(a[:, 0].mean(), 2), round(np.median(a[:, 0]), 2),
                                      round(a[:, 1].mean(), 2), round(a[:, 2].mean(), 3))))
           for src, a in acc_A5.items()}
print('per frame, first 5 (err deg/s, dir deg, beta):')
print(pd.DataFrame(np.hstack([acc_A5['cmax'][:5], acc_A5['gyro'][:5]]),
                   columns=['cmax_err', 'cmax_dir', 'cmax_beta',
                            'gyro_err', 'gyro_dir', 'gyro_beta']).round(3))
display(pd.DataFrame([{'sequence': ds_A5, 'segment': sid_A5, 'source': s, **r}
                      for s, r in rows_A5.items()]))
if p['mode'] == 'Live':                                  # cmd_baseline's csv, this segment only
    with open(csv_A5, 'w', newline='', encoding='utf-8') as fh:
        wr = csv.writer(fh)
        wr.writerow(['sequence', 'segment', 'source', 'dt_ms', 'n_frames',
                     'err', 'median', 'dir', 'beta', 'secs'])
        for src, r in rows_A5.items():
            wr.writerow([ds_A5, sid_A5, src, int(dt_A5 * 1000), nf_A5,
                         *[r[c] for c in BL_COLS_A5], round(secs_A5)])
    print(f'-> {csv_A5}')

### tab:baseline -- `make_tables.py::table_baseline`

Per sequence: the mean ω error of CMax alone and of the gyro alone (this
section's csv), and of the CMax-anchored network (the `thesis_cmax` row of A3's
`dissoc_1p5s_*.csv`). Bold marks the lower of CMax alone and network. In Live
the CMax and gyro columns hold only the panel's segment, from the Live csv.

In [ ]:
%%section A5
# step 5 -- make_tables.py::table_baseline, unrolled (fmt / tabular are its formatters)
base_A5 = {}
if os.path.exists(csv_A5):
    for r in csv.DictReader(open(csv_A5)):
        base_A5[(r['sequence'], r['source'])] = float(r['err'])
net_A5 = make_tables.load(['dissoc_1p5s_*.csv'])        # the network column: A3's csvs
body_A5, cells_A5 = [], []
for i, (ds, disp, _lab, _extra) in enumerate(make_tables.SEQS):
    if i and ds in make_tables.SYNTHETIC and make_tables.SEQS[i - 1][0] not in make_tables.SYNTHETIC:
        body_A5.append(r'\midrule')
    cm, gy = base_A5.get((ds, 'cmax')), base_A5.get((ds, 'gyro'))
    nw = net_A5.get((ds, 'thesis_cmax'))
    nw = float(nw['err']) if nw else None
    best = min(v for v in (cm, nw) if v is not None) if (cm or nw) else None
    cell = lambda v: (rf'\textbf{{{make_tables.fmt(v, 2)}}}' if v is not None and v == best
                      else make_tables.fmt(v, 2))
    body_A5.append(f'{disp:<18}& {cell(cm)} & {make_tables.fmt(gy, 2)} & {cell(nw)} \\\\')
    cells_A5.append([_lab, cm, gy, nw])
head_A5 = [r'sequence & CMax alone & gyro alone & network \\',
        r' & [$^\circ$/s] & [$^\circ$/s] & [$^\circ$/s] \\', r'\midrule']
table_A5 = make_tables.tabular('@{}lccc@{}', head_A5, body_A5)
display(pd.DataFrame(cells_A5, columns=['sequence', 'CMax alone (deg/s)', 'gyro alone (deg/s)',
                                     'network (deg/s)']))
print(table_A5)

### fig_cmax_baseline -- `make_figures.py::fig_cmax_baseline` (not in the tex)

CMax alone and the gyro against the reference, ω per axis, with the CMax IWE at
the three stored frames (dashed lines). The report draws boxes and bicycle;
Live draws the panel's segment from the Live npz.

In [ ]:
%%section A5
# step 6 -- make_figures.py::fig_cmax_baseline, unrolled
from interacting_maps.style import use_style, save, frame_times, PAPER_WIDE, NEUTRAL_COLOR
from interacting_maps.viz import draw_map, draw_omega
labels_A5 = ('boxes', 'bicycle') if p['mode'] == 'Redraw' else (lab_A5,)
with use_style('paper'):
    fig_A5 = plt.figure(figsize=(PAPER_WIDE, 5.2))
    for sub, lab in zip(fig_A5.subfigures(1, len(labels_A5)), labels_A5):
        path = os.path.join(src_A5, f'baseline_dt{int(dt_A5*1000)}ms_{lab}.npz')
        if not need([path], ['baseline']):
            continue
        d = np.load(path)
        t = frame_times(dt_A5, len(d['ref']))
        gs = sub.add_gridspec(4, 3, height_ratios=(1, 1, 1, 1.3))
        for i in range(3):
            ax = sub.add_subplot(gs[i, :])
            draw_omega(ax, t, d['cmax'][:, i], ref=d['ref'][:, i], imu=d['gyro'][:, i],
                       axis=i, model='thesis_cmax', ref_src=str(d['ref_src']),
                       label='CMax alone')
            for k in d['frames']:
                ax.axvline(t[k], color=NEUTRAL_COLOR, lw=0.6, ls='--')
            if i < 2:
                ax.tick_params(labelbottom=False)
                ax.set_xlabel('')
        # one IWE per stored image (a Live run of < 3 frames stores fewer than 3 frames)
        for j, k in enumerate(d['frames'][:len(d['iwe'])]):
            draw_map(sub.add_subplot(gs[3, j]), 'IWE', d['iwe'][j],
                     title=f'IWE  t = {t[k]:.2f} s')
        sub.suptitle(lab)
        sub.legend(*sub.axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=3)
    lines_fig_A5 = [l.get_ydata() for ax in fig_A5.axes for l in ax.lines]
    imgs_fig_A5 = [np.asarray(im.get_array(), float) for ax in fig_A5.axes for im in ax.images]
    save(fig_A5, os.path.join(walk_dir('A5'), 'fig_cmax_baseline.png'), show=True)

In [ ]:
%%section A5
# step 7 -- text_numbers.py::maps_add (Redraw only: it reads the report's csvs)
from text_numbers import rows_of, sens_rows, axis as tn_axis, f as tn_f
lines_A5 = []
if p['mode'] == 'Redraw':
    L = lines_A5
    bl_A5 = {(r['sequence'], r['source']): r for r in rows_of(os.path.join(BASE_A5, 'baseline_dt20ms.csv'))}
    nt_A5 = {r['sequence']: r for r in rows_of(os.path.join(BASE_A5, 'dissoc_1p5s_*_thesis_cmax.csv'))}
    for seg in SEGMENTS:                       # network against its own anchor, mean and median
        b, n = bl_A5.get((seg[0], 'cmax')), nt_A5.get(seg[0])
        if b and n:
            L.append(f'{seg[3]:8s} CMax alone {tn_f(b):.2f} (median {tn_f(b, "median"):.2f}), '
                     f'network {tn_f(n):.2f} (median {tn_f(n, "median"):.2f}): '
                     f'mean {100*(1-tn_f(n)/tn_f(b)):+.0f}%, '
                     f'median {100*(1-tn_f(n, "median")/tn_f(b, "median")):+.0f}%')
    sens_A5 = sens_rows()                         # the delta_VFG axis of the A8 sweep
    L.append('delta_VFG (CMax-anchor):')
    for seg in SEGMENTS:
        a, b = tn_axis(sens_A5, 'delta_VFG', seg[0]), bl_A5.get((seg[0], 'cmax'))
        if not a or not b:
            continue
        lo, hi = a[min(a)], a[max(a)]
        dirs = [tn_f(r, 'dir') for r in a.values()]
        L.append(f'  {seg[3]:8s} at {min(a):g}: err {tn_f(lo):.2f} vs CMax alone {tn_f(b):.2f} '
                 f'(diff {tn_f(lo)-tn_f(b):+.2f}); dir range {max(dirs)-min(dirs):.2f} deg; '
                 f'beta {tn_f(lo, "beta"):.2f} -> {tn_f(hi, "beta"):.2f}; at {max(a):g} '
                 f'{100*(1-tn_f(hi)/tn_f(b)):+.0f}% vs CMax alone')
    print('== maps_add (err in deg/s)')
    print('\n'.join(lines_A5) or '(no rows)')
else:
    print('Live: the text numbers read the report csvs; switch to Redraw to see them.')

In [ ]:
%%section A5
# same-code check -- cmd_baseline itself (Live); the csv, table_baseline,
# fig_cmax_baseline and text_numbers.maps_add (Redraw)
import io, argparse
if p['mode'] == 'Live':
    # cmd_baseline on the panel's segment only (SEGMENTS redirected), same frames,
    # its csv and npz into walk_dir('A5')/check
    out = os.path.join(walk_dir('A5'), 'check')
    os.makedirs(out, exist_ok=True)
    args = argparse.Namespace(dt=dt_A5, duration=nf_A5 * dt_A5)
    with redirect(run_experiments, SEGMENTS=[p['seg']], OUTDIR=out), \
         contextlib.redirect_stdout(io.StringIO()):
        run_experiments.cmd_baseline(E, args)
    theirs = {r['source']: r for r in csv.DictReader(open(os.path.join(out, os.path.basename(csv_A5))))}
    for src, r in rows_A5.items():
        check(f'cmd_baseline row {src}', r, {c: float(theirs[src][c]) for c in BL_COLS_A5})
    d2 = np.load(os.path.join(out, os.path.basename(npz_A5)))
    for s in ('cmax', 'gyro', 'ref'):
        check(f'cmd_baseline series {s}', series_A5[s], d2[s])
else:
    # the rows recomputed from the stored series against the csv the job wrote
    rep = {r['source']: r for r in csv.DictReader(open(csv_A5)) if r['sequence'] == ds_A5}
    for src, r in rows_A5.items():
        check(f'baseline row {src}', r, {c: float(rep[src][c]) for c in BL_COLS_A5})
    check('table_baseline', nums(table_A5.splitlines()),
          nums(make_tables.table_baseline().splitlines()))
    captured = []
    def keep(fig, path, show=False):
        captured.append(fig)
    with redirect(make_figures, FIG=os.path.join(walk_dir('A5'), 'check'), save=keep), \
         contextlib.redirect_stdout(io.StringIO()):
        make_figures.fig_cmax_baseline(E)
    flat = lambda ls: np.concatenate([np.ravel(x) for x in ls]) if ls else np.array([])
    th_lines = [l.get_ydata() for f_ in captured for ax in f_.axes for l in ax.lines]
    th_imgs = [np.asarray(im.get_array(), float) for f_ in captured for ax in f_.axes for im in ax.images]
    for f_ in captured:
        plt.close(f_)
    check('fig_cmax_baseline lines', flat(lines_fig_A5), flat(th_lines))
    check('fig_cmax_baseline IWE images', flat(imgs_fig_A5), flat(th_imgs))
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        text_numbers.maps_add()
        theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
    check('text_numbers.maps_add', nums(lines_A5), nums(theirs))

<a id="A6"></a>

## A6 · β vs speed · [↑ overview](#overview)

**Question.** Does the scale β = |ω_ref| / |ω_est| depend on the rotation speed?
Seven segments of the two real sequences (boxes, poster), 3 s each, under three
anchor settings: CMax-anchor at δ_FR 0.5 / anchor 0.3 (the original step sizes),
CMax-anchor at δ_FR 0.1 / anchor 0.5 (the adopted ones), CMax-inloop at δ_FR 0.5.

**Made by** `report/run_experiments.py::cmd_beta`, one run per
(`report_runs.BETA_SEGMENTS` × `BETA_CONFIGS`) pair, `run_all.sh` jobs
`beta_<i>` (i = 3·segment + config) → runs in `results/` (150 frames). Jobs 4
and 16 are skipped there while config.py adopts δ_FR 0.1 / anchor 0.5: those runs
are the `main_<seq>_thesis_cmax` runs of A2.

**Read by** `report/make_figures.py::fig_beta_speed` (fig:beta_speed) and
`report/text_numbers.py::beta_speed`. `text_numbers.py::synthetic_beta_original`
(the synthetic β at the original step sizes, from the A10 sweep base point) is
quoted next to it. The 3 s runs are also read by A2's `poster_segments`.

**Steps:** the runs (found, or one made now) → per run: speed and mean β →
the speed as the figure takes it → the figure → the text numbers → same-code check.
Live makes the panel's segment only, if it is one of the seven: `one point` with
the adopted config, `one axis` / `full` with the three `BETA_CONFIGS`.

In [ ]:
# the A6 form: which anchor configurations on which beta segments
from report_runs import BETA_CONFIGS
form('A6', [Knob('configs', BETA_CONFIGS, 'many', choices=BETA_CONFIGS),
            Knob('segments', BETA_SEGMENTS, 'many', choices=BETA_SEGMENTS),
            Knob('n_frames', 150), Knob('dt', 0.02)])

In [ ]:
# step 1 -- the runs of the form's (segment x config) pairs: found (Redraw) or made (Live)
p = params('A6')
rep_A6 = is_report('A6')                        # the report-only steps run only then
PAIRS_A6 = [(s, c) for s in BETA_SEGMENTS for c in BETA_CONFIGS]   # cmd_beta's order: index i
nf_A6, runs_A6, miss = p['n_frames'], {}, []


def beta_run_dir_A6(seg, model, d_fr, d_a):
    """The saved run of one (segment, config): every step size as cmd_beta's run_cfg sets
    it (config.py's, with the config's delta_FR / anchor), so a sweep run with another
    rate never stands in; results/ first."""
    prm = {**make_tables.expected_params(model), 'delta_FR': d_fr,
           **({'delta_IMU': d_a} if d_a is not None else {})}
    with contextlib.redirect_stdout(io.StringIO()):
        return next((h for h in (experiment_tracking.find_run(r, seg[0], model, seg[1], nf_A6, prm,
                                                               dt=p['dt'], t_start=seg[2])
                                 for r in roots()) if h), None)


for seg in p['segments']:
    for model, d_fr, d_a in p['configs']:
        if p['mode'] == 'Redraw':
            d = beta_run_dir_A6(seg, model, d_fr, d_a)
        elif need([os.path.join('data', seg[0])]):      # cmd_beta's run_cfg call
            d = saved_or_run(run_cfg(*seg, model, p['dt'], nf_A6, out_root='experiments/walkthrough',
                                     delta_FR=d_fr, delta_IMU=d_a))
        else:
            d = None
        if d is None:
            miss.append((seg, (model, d_fr, d_a)))
        else:
            runs_A6[seg, (model, d_fr, d_a)] = d
print(f'{len(runs_A6)} of {len(runs_A6) + len(miss)} runs found')
for seg, cfg in miss:
    i = PAIRS_A6.index((seg, cfg)) if (seg, cfg) in PAIRS_A6 else None
    print(f'no saved run for {seg[0][:6]} {seg[1]} {cfg}' + (f' (run_all.sh job beta_{i})' if i is not None else ''))
READY['A6'] = bool(runs_A6)
for (seg, cfg), d in runs_A6.items():
    print(f'{seg[0][:6]} {seg[1]} {cfg[0]:15s} dFR {cfg[1]} anchor {cfg[2]}: {d}')

In [ ]:
%%section A6
# step 2 -- per run: speed |w_ref| and mean beta, from its tracking.csv (text_numbers.run)
pts_A6 = []
for (seg, (model, d_fr, d_a)), d in runs_A6.items():
    tr = tracking(d)
    pts_A6.append(dict(segment=f'{seg[0][:6]} {seg[1]}', model=model, dFR=d_fr, anchor=d_a,
                       speed_deg_s=float(mag(tr['gt']).mean() * DEG), beta=float(tr['beta'].mean()),
                       beta_summary=experiment_tracking.read_summary(d)['mean_beta'],
                       err_deg_s=float(tr['err'].mean()), dir_deg=float(tr['dir'].mean())))
display(pd.DataFrame(pts_A6).round(3))

In [ ]:
%%section A6
# step 3 -- the speed as fig_beta_speed takes it: mean |reference| over the
# segment's 150 frames, from the dataset (make_figures.reference_omega); where
# the dataset is not on this machine, the same reference from the runs' tracking.csv
speed_A6, speed_src_A6 = {}, {}
for seg in BETA_SEGMENTS:
    have = [d for (s, _c), d in runs_A6.items() if s == seg]
    if not have:
        continue
    if os.path.isdir(os.path.join('data', seg[0])):
        ref, _ = reference_and_gyro(seg + ('',), p['dt'], nf_A6)
        speed_A6[seg], speed_src_A6[seg] = float(to_deg_s(mag(ref).mean())), 'dataset'
    elif have:
        speed_A6[seg] = float(to_deg_s(mag(tracking(have[0])['gt']).mean()))
        speed_src_A6[seg] = 'tracking.csv'
for seg, w in speed_A6.items():
    print(f'{seg[0][:6]} {seg[1]}: |w| = {w:6.1f} deg/s ({speed_src_A6[seg]})')

In [ ]:
%%section A6
# step 4 -- make_figures.py::fig_beta_speed, unrolled: beta against speed, one
# series per BETA_CONFIGS entry, Pearson r and a straight-line fit
from interacting_maps.style import (use_style, save, PAPER_WIDTH, NEUTRAL_COLOR, LABEL_BETA,
                                    MODELS as STYLE_MODELS_A6)
series_A6 = {'default': ('thesis_cmax', 0.5, 0.3, False),     # fixed in fig_beta_speed
             'tuned':   ('thesis_cmax', 0.1, 0.5, True),
             'inloop':  ('thesis_cmax_v2', 0.5, None, True)}
data_A6 = {k: ([], []) for k in series_A6}
for seg in BETA_SEGMENTS:
    for key, (model, d_fr, d_a, _) in series_A6.items():
        d = runs_A6.get((seg, (model, d_fr, d_a)))
        if d is None:
            continue
        data_A6[key][0].append(speed_A6[seg])
        data_A6[key][1].append(experiment_tracking.read_summary(d)['mean_beta'])
with use_style('paper'):
    fig_A6, ax = plt.subplots(figsize=(PAPER_WIDTH, 2.5))
    for key, (model, d_fr, _, filled) in series_A6.items():
        x, y = np.array(data_A6[key][0]), np.array(data_A6[key][1])
        name, col = STYLE_MODELS_A6[model]
        if len(x) < 2:                                  # Live: a point, no fit
            if len(x):
                ax.scatter(x, y, s=26, zorder=3, color=col if filled else 'none', edgecolors=col,
                           linewidths=0.8, label=rf'{name}, $\delta_{{FR}}={d_fr}$ (no fit: 1 point)')
            continue
        r = np.corrcoef(x, y)[0, 1]
        ax.scatter(x, y, s=26, zorder=3, color=col if filled else 'none', edgecolors=col,
                   linewidths=0.8, label=rf'{name}, $\delta_{{FR}}={d_fr}$  ($r={r:+.2f}$)')
        xs = np.linspace(x.min(), x.max(), 20)
        ax.plot(xs, np.polyval(np.polyfit(x, y, 1), xs), color=col, lw=1.0, alpha=0.55,
                ls='-' if filled else '--', zorder=2)
        print(f'{key:8s} {name} dFR {d_fr}: n = {len(x)}, r = {r:+.2f}')
    ax.axhline(1.0, color=NEUTRAL_COLOR, lw=0.8, ls=':', zorder=1)
    ax.set_xlabel(r'rotation speed $\|\omega\|$ (°/s)')
    ax.set_ylabel(LABEL_BETA)
    ax.legend(loc='upper left')
    pts_fig_A6 = [np.asarray(c.get_offsets(), float) for c in ax.collections]
    lines_fig_A6 = [l.get_ydata() for l in ax.lines]
    save(fig_A6, os.path.join(walk_dir('A6'), 'fig_beta_speed.png'), show=True)

In [ ]:
%%section A6
# step 5 -- the text numbers (report configuration only: they read the report's runs)
from scipy import stats
from text_numbers import rows_of, f as tn_f
lines_A6 = {'beta_speed': [], 'synthetic_beta_original': []}
if rep_A6:
    L = lines_A6['beta_speed']          # text_numbers.beta_speed
    tn_data = {}
    for cfg in BETA_CONFIGS:
        model, d_fr, d_a = cfg
        pts = []
        for seg in BETA_SEGMENTS:
            d = runs_A6.get((seg, cfg))
            if d is not None:
                t_ = tracking(d)
                pts.append((mag(t_['gt']).mean(), t_['beta'].mean(), t_['err'].mean(),
                            t_['dir'].mean(), t_['est']))
        tn_data[cfg] = pts
        if len(pts) < 3:
            continue
        w, b = np.array([q[0] for q in pts]), np.array([q[1] for q in pts])
        r, pv = stats.pearsonr(w, b)
        L.append(f'{model} dFR {d_fr} anchor {d_a}: r = {r:+.2f} (p = {pv:.3g}, n = {len(w)}), '
                 f'beta {b.min():.2f}-{b.max():.2f}, beta at slowest {b[np.argmin(w)]:.2f}, '
                 f'at fastest {b[np.argmax(w)]:.2f}; speeds {w.min()*DEG:.1f}-{w.max()*DEG:.1f} deg/s')
    orig, adopt, inl = (tn_data.get(c, []) for c in BETA_CONFIGS)
    if len(orig) >= 2:                  # the original fit, extrapolated to the synthetic speed
        fit = np.polyfit([q[0] for q in orig], [q[1] for q in orig], 1)
        syn = [run_dir(s, 'thesis_cmax', 75) for s in SEGMENTS if s[3] in ('bicycle', 'street')]
        ws = np.mean([mag(tracking(d)['gt']).mean() for d in syn if d is not None])
        L.append(f'original fit predicts beta {np.polyval(fit, ws):.2f} at the synthetic {ws*DEG:.1f} deg/s')
    if len(inl) == len(orig) == len(adopt):
        for (o, a, v, seg) in zip(orig, adopt, inl, BETA_SEGMENTS):
            L.append(f'{seg[0][:6]} {seg[1]}: dir CMax-anchor {o[3]:.1f} / {a[3]:.1f}, CMax-inloop {v[3]:.1f}; '
                     f'err CMax-anchor {o[2]:.1f} / {a[2]:.1f}, CMax-inloop {v[2]:.1f} (original / adopted)')
    L = lines_A6['synthetic_beta_original']   # text_numbers.synthetic_beta_original (A10's sweep csv)
    for s in [s for s in SEGMENTS if s[3] in ('bicycle', 'street')]:
        path = os.path.join('experiments', 'sweep', f'sweep_{s[3]}_cmax_delta_FR=0.5.csv')
        need([path], [f'sweep_{s[3]}_cmax_delta_FR=0.5'])
        for r in rows_of(path):
            L.append(f'{s[3]:8s} dFR {r["delta_FR"]} anchor {r["delta_IMU"]}: beta {tn_f(r, "beta"):.3f}, '
                     f'err {tn_f(r):.2f}')
    for k, ls in lines_A6.items():
        print(f'== {k}'); print('\n'.join(ls) or '(no runs)')
else:
    print('[skip] report-only step')

In [ ]:
%%section A6
# same-code check -- beta per run vs read_summary; cmd_beta's config (Live);
# fig_beta_speed and text_numbers (Redraw)
import io, argparse
check('beta per run vs read_summary', {f"{q['segment']} {q['model']} {q['dFR']}": q['beta'] for q in pts_A6},
      {f"{q['segment']} {q['model']} {q['dFR']}": q['beta_summary'] for q in pts_A6})
both = [s for s in speed_A6 if speed_src_A6[s] == 'dataset' and any(k[0] == s for k in runs_A6)]
if both:                                # the run's stored reference is the dataset's
    check('speed: dataset vs tracking.csv',
          [speed_A6[s] for s in both],
          [float(to_deg_s(mag(tracking(next(d for (s_, _c), d in runs_A6.items() if s_ == s))['gt']).mean()))
           for s in both])
if p['mode'] == 'Live':
    # cmd_beta with its run step replaced by a recorder: same RunConfig per index?
    for (seg, cfg), d in runs_A6.items():
        if (seg, cfg) not in PAIRS_A6:
            print(f'[skip] cmd_beta config: {cfg} is not a BETA_CONFIGS entry')
            continue
        seen = []
        def record(E_, rc, save_frames, stride):
            seen.append(rc)
            return experiment_tracking.read_summary(d), 0
        args = argparse.Namespace(dt=p['dt'], duration=nf_A6 * p['dt'],
                                  index=PAIRS_A6.index((seg, cfg)), no_frames=True, frame_stride=1)
        with redirect(run_experiments, _run=record), contextlib.redirect_stdout(io.StringIO()):
            run_experiments.cmd_beta(E, args)
        mine = run_cfg(*seg, cfg[0], p['dt'], nf_A6, delta_FR=cfg[1], delta_IMU=cfg[2]).to_dict()
        a = seen[0].to_dict() if seen else {}
        differ = sorted(k for k in mine if k != 'out_root' and a.get(k) != mine[k])
        print(f'[ok] cmd_beta config {cfg}' if seen and not differ
              else f'[DIFF] cmd_beta config {cfg}: {differ or "no run"}')
if not rep_A6:
    print('[skip] not the report configuration')
else:
    # fig_beta_speed reads the reference from the dataset (make_figures.reference_omega).
    # A dataset not on this machine: its reference is taken from the segment's run
    # (tracking.csv stores the same reference, see the speed check above).
    ref_fn = make_figures.reference_omega
    def ref_or_run(E_, ds, t0, n=150):
        if os.path.isdir(os.path.join('data', ds)):
            return ref_fn(E_, ds, t0, n)
        d = next((d for (s, _c), d in runs_A6.items() if s[0] == ds and abs(s[2] - t0) < 1e-9), None)
        return (tracking(d)['gt'][:n] if d else np.full((n, 3), np.nan)), 'groundtruth'
    gone = sorted({s[0] for s in BETA_SEGMENTS if not os.path.isdir(os.path.join('data', s[0]))})
    if gone:
        print(f'fig_beta_speed check: {", ".join(gone)} not here, its reference read from the runs')
    captured = []
    def keep(fig, path, show=False):
        captured.append(fig)
    with redirect(make_figures, FIG=os.path.join(walk_dir('A6'), 'check'), save=keep,
                  reference_omega=ref_or_run), contextlib.redirect_stdout(io.StringIO()):
        make_figures.fig_beta_speed(E)
    ax_ = captured[0].axes[0]
    flat = lambda ls: np.concatenate([np.ravel(x) for x in ls]) if ls else np.array([])
    check('fig_beta_speed points', flat(pts_fig_A6),
          flat([np.asarray(c.get_offsets(), float) for c in ax_.collections]))
    check('fig_beta_speed lines', flat(lines_fig_A6), flat([l.get_ydata() for l in ax_.lines]))
    for f_ in captured:
        plt.close(f_)
    for name in lines_A6:
        with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
            getattr(text_numbers, name)()
            theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
        check(f'text_numbers.{name}', nums(lines_A6[name]), nums(theirs))

<a id="A7"></a>

## A7 · Anchor across poster segments · [↑ overview](#overview)

**Question.** Does the ranking of the models hold on the other poster segments?
Every model on poster seg_A (fastest), seg_C and seg_D (slowest), 3 s each,
with the gyro floor of each segment.

**Made by** `evaluation.py --exp 2 --dataset poster_rotation --segment <seg>
--model <model> --no-frames`, `run_all.sh` jobs `anchor_<seg>_<model>` (seg_A /
seg_D × cook, thesis, thesis_imu, thesis_cmax_v2) → runs in
`results/poster_rotation/<model>/` (150 frames, config.py's step sizes). The
other cells are runs made elsewhere: seg_C is the 3 s main run of A2
(`main_poster_<model>`), thesis_cmax on seg_A / seg_D the adopted β run of A6
(`beta_<i>`).

**Read by** `report/make_tables.py::table_anchor` (tab:anchor), including
`make_tables.py::gyro_floor`. **Not in the tex**: `make_tables.py` writes
`table_anchor.tex`, but `bare_jrnl.tex` does not input it.

**Steps:** the runs (found, or made now) → the gyro floor per segment → the
table → same-code check. Live runs the panel's model on poster seg_A
(`one point`) or on all three segments (`one axis` / `full`).

In [ ]:
# the A7 form: which models on which poster segments
SEGS_A7 = make_tables.ANCHOR_SEGMENTS                      # seg_A, seg_C, seg_D
form('A7', [Knob('models', MODELS, 'many'),
            Knob('segments', list(SEGS_A7), 'many', choices=list(SEGS_A7)),
            Knob('n_frames', 150), Knob('dt', 0.02), Knob('poisson', POISSON_MODE),
            Knob('delta_FR', ADOPTED['delta_FR'])])

In [ ]:
# step 1 -- the runs: table_anchor's lookup for the form's values (Redraw) or made now (Live)
from report_runs import BETA_CONFIGS
p = params('A7')
rep_A7 = is_report('A7')                        # the report-only steps run only then
T0_A7 = {s['id']: s['t_start'] for s in E.DATASET_SEGMENTS['poster_rotation']}


def anchor_job(sid, model):
    """The run_all.sh job that makes table_anchor's run of (sid, model)."""
    if sid == 'seg_C':
        return f'main_poster_{model}'
    if model == 'thesis_cmax':                              # the adopted beta run
        seg = ('poster_rotation', sid, T0_A7[sid])
        cfg = ('thesis_cmax', ADOPTED['delta_FR'], ADOPTED['delta_IMU'])
        return f'beta_{3 * BETA_SEGMENTS.index(seg) + BETA_CONFIGS.index(cfg)}'
    return f'anchor_{sid}_{model}'


def anchor_deltas_A7(model):
    """The step sizes the form changes (evaluation.py --exp 2 takes config.py's); cook
    has its own and keeps them."""
    d = {} if model == 'cook' else {'delta_FR': p['delta_FR']}
    return {k: v for k, v in d.items() if v != THESIS_PARAMS[k]}


runs_A7, miss, nf_A7 = {}, [], p['n_frames']
if p['mode'] == 'Live':
    print(f"{len(p['segments']) * len(p['models'])} run(s) of {nf_A7} frames: "
          f"{', '.join(p['models'])} on poster {', '.join(p['segments'])}")
for sid in p['segments']:
    for k in p['models']:
        if p['mode'] == 'Redraw':
            # table_anchor: find_run, every step size as in config.py, the form's delta_FR
            prm = {**make_tables.expected_params(k), **anchor_deltas_A7(k)}
            d = next((h for h in (experiment_tracking.find_run(
                     r, 'poster_rotation', k, sid, nf_A7, prm, dt=p['dt'], poisson=p['poisson'])
                     for r in roots()) if h), None)
        elif need([os.path.join('data', 'poster_rotation')]):
            # evaluation.py --exp 2's RunConfig with the form's values, out_root walkthrough
            d = saved_or_run(E.RunConfig(dataset='poster_rotation', model=k, segment=sid,
                                         n_frames=nf_A7, frame_duration=p['dt'],
                                         poisson=p['poisson'], deltas=anchor_deltas_A7(k) or None,
                                         out_root='experiments/walkthrough'))
        else:
            d = None
        if d is None:
            print(f'no saved run for poster {sid} {k}' + (f' (run_all.sh job {anchor_job(sid, k)})'
                                                          if rep_A7 else ''))
        else:
            runs_A7[sid, k] = d
READY['A7'] = bool(runs_A7)
for (sid, k), d in runs_A7.items():
    print(f'{sid} {k:16s} {d}')

In [ ]:
%%section A7
# step 2 -- make_tables.py::gyro_floor: the gyro against the reference, from the
# thesis+IMU run's tracking.csv (Live: from the panel's run; every run stores both)
floors_A7 = {}
for sid in SEGS_A7:
    d = runs_A7.get((sid, 'thesis_imu')) or next((d for (s, _k), d in runs_A7.items() if s == sid), None)
    if d is None:
        continue
    tr = tracking(d)
    g, t_ = tr['imu'], tr['gt']                             # gyro, reference (code unit)
    cos = (g * t_).sum(1) / (np.linalg.norm(g, axis=1) * np.linalg.norm(t_, axis=1))
    floors_A7[sid] = (np.degrees(np.linalg.norm(g - t_, axis=1)).mean(),
                      np.degrees(np.arccos(np.clip(cos, -1, 1))).mean(),
                      np.linalg.norm(t_, axis=1).mean())
    print(f'{sid}: gyro floor {floors_A7[sid][0]:.2f} deg/s, direction {floors_A7[sid][1]:.2f} deg; '
          f'mean |w_ref| {to_deg_s(floors_A7[sid][2]):.1f} deg/s  ({os.path.basename(d)})')

### tab:anchor -- `make_tables.py::table_anchor` (not in the tex)

One block per segment, slowest last: err, dir and β of every model from its
run summary, and the gyro floor in italics. The row label is the segment's mean
|ω| in deg/s.

In [ ]:
%%section A7
# step 3 -- table_anchor, unrolled (fmt / tabular / floor_label are its formatters)
body_A7, cells_A7 = [], []
for sid in SEGS_A7:
    fl = floors_A7.get(sid) if (sid, 'thesis_imu') in runs_A7 or p['mode'] == 'Live' else None
    body_A7 += [r'\midrule',
                rf'\multirow{{6}}{{*}}{{${to_deg_s(fl[2]):.0f}$\,$^\circ$/s}}' if fl else r'\multirow{6}{*}{---}']
    for mdisp, key in make_tables.MODELS:
        d = runs_A7.get((sid, key))
        if d is None:
            body_A7.append(f' & {mdisp:<22}& --- & --- & --- \\\\')
            continue
        s = experiment_tracking.read_summary(d)
        body_A7.append(f' & {mdisp:<22}& {make_tables.fmt(s["mean_err_deg_s"])} & '
                       f'{make_tables.fmt(s["mean_dir_err_deg"])} & '
                       f'{make_tables.fmt(s["mean_beta"], 1 if s["mean_beta"] >= 100 else 2)} \\\\')
        cells_A7.append([sid, key, s['mean_err_deg_s'], s['mean_dir_err_deg'], s['mean_beta']])
    if fl:
        body_A7.append(rf' & {make_tables.floor_label("poster_rotation")}  & '
                       rf'\emph{{{make_tables.fmt(fl[0])}}} & \emph{{{make_tables.fmt(fl[1])}}} & --- \\')
        cells_A7.append([sid, 'gyro vs ref.', fl[0], fl[1], np.nan])
table_A7 = make_tables.tabular('@{}llrrr@{}',
                               [r'$\|\wvec\|$ & Model & err ($^\circ$/s) & dir ($^\circ$) & $\beta$ \\'],
                               body_A7)
df_A7 = pd.DataFrame(cells_A7, columns=['segment', 'model', 'err (deg/s)', 'dir (deg)', 'beta'])
df_A7.insert(1, '|w_ref| (deg/s)', [to_deg_s(floors_A7[s][2]) if s in floors_A7 else np.nan
                                    for s in df_A7['segment']])
display(df_A7.round(2))
print(table_A7)

In [ ]:
%%section A7
# same-code check -- gyro_floor and table_anchor themselves (Redraw);
# the run summaries against their tracking.csv and gyro_floor (Live)
import io
for sid, fl in floors_A7.items():
    d = runs_A7.get((sid, 'thesis_imu')) or next(d for (s, _k), d in runs_A7.items() if s == sid)
    check(f'gyro_floor {sid}', fl, make_tables.gyro_floor(d))
if p['mode'] == 'Live':
    for (sid, k), d in runs_A7.items():
        s, tr = experiment_tracking.read_summary(d), tracking(d)
        check(f'summary vs tracking.csv {sid} {k}',
              [s['mean_err_deg_s'], s['mean_dir_err_deg'], s['mean_beta']],
              [tr['err'].mean(), tr['dir'].mean(), tr['beta'].mean()])
if not rep_A7:
    print('[skip] not the report configuration')
else:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        theirs = make_tables.table_anchor()
    check('table_anchor', nums(table_A7.splitlines()), nums(theirs.splitlines()))

<a id="A8"></a>

## A8 · Sensitivity sweep (one parameter at a time) · [↑ overview](#overview)

**Question.** Around the adopted configuration, how do the ω error, the
direction error, β and the read-out |r| move when one parameter changes: the
anchor weight (CMax-anchor and thesis+IMU), δ_FR, the four relaxation rates,
the iterations per frame and the window? CMax-anchor, 1.5 s, every sequence.

**Made by** `report/run_experiments.py::cmd_sweep`
(`--what sweep --name sens --mode coord --duration 1.5`), `run_all.sh` jobs
`sens_cmax_<seq>`, `sens_imu_<seq>`, `sens_rates_<seq>` → runs and
`sens_*.csv` in `experiments/sens/`. `table_sweep` and `text_numbers` fall back
to `report/experiments/sens_*.csv` when that folder is empty;
`fig_sensitivity` does not (flagged below).

**Read by** `report/make_tables.py::table_sweep` (tab:sweep),
`report/make_figures.py::fig_sensitivity` (**not in the tex**: drawn, never
`\includegraphics`-ed), and `report/text_numbers.py::sensitivity_both`,
`::sweep_ends`, `::anchor_ablation`, `::anchors_across_weights`,
`::kappa_rows`. The κ panel of `fig_sensitivity` and the grid half of
`sensitivity_both` read A9's grid rows.

**Sweep extent** (panel, Live only; sections A8–A13 share it): `one point`
runs the job's base point, `one axis` every value of the axis the panel
changed (else the job's first axis), `full` the job's whole `--vary`; always on
the panel's segment, and `n_frames` caps the track (n_frames × 20 ms, the
duration `cmd_sweep` holds fixed, so a 5 ms point runs 4× the frames). The
panel's model picks the job (`thesis_imu` → `sens_imu`; a changed rate →
`sens_rates`).

In [ ]:
# sweep definitions -- the unrolled plumbing of run_experiments.py::cmd_sweep,
# shared by A8-A13 (the counterpart of A2-defs)
import io, json, argparse
from run_experiments import ALL_DELTAS, _parse_vary, _points, _split
from interacting_maps.style import use_style, save as style_save

#: cmd_sweep's csv columns, in its order
SWEEP_COLS = (['sequence', 'segment', 'model', 'dt_ms', 'n_frames', 'n_iters', 'poisson']
              + ALL_DELTAS + ['err', 'median', 'dir', 'beta', 'low_freq', 'contrast', 'curl',
                              'recon_mean', 'recon_min', 'recon_inloop', 'recon_inloop_min',
                              'blowup', 'secs'])
_ALL = 'boxes,poster,dynamic,bicycle,street'
_S = lambda m, seqs: tuple(f'sweep_{m}_{s}.csv' for s in seqs.split(','))


def _job(name, mode, model, duration, vary, csv_, run_all, base='', seqs=_ALL):
    return dict(name=name, mode=mode, model=model, duration=duration, vary=vary, base=base,
                csv=csv_, run_all=run_all, seqs=seqs)


#: run_all.sh's sweep jobs, copied. Where run_all.sh splits one sweep into one
#: job per value (grid: one delta_FR row per job; sweep_*: one value per job),
#: the entry is their union, which _points turns into the same set of points.
SWEEP_JOBS = {
    'sens_cmax': _job('sens', 'coord', 'thesis_cmax', 1.5,
                      'delta_IMU=0,0.1,0.3,0.5,0.7,0.9;delta_FR=0.02,0.05,0.1,0.3,0.5,0.7,0.9;'
                      'n_iters=25,50,75,150;dt=0.005,0.01,0.02,0.04',
                      ('sens_cmax_*.csv',), 'sens_cmax_<seq>'),
    'sens_imu': _job('sens', 'coord', 'thesis_imu', 1.5, 'delta_IMU=0.1,0.3,0.5,0.7,0.9',
                     ('sens_imu_*.csv',), 'sens_imu_<seq>'),
    'sens_rates': _job('sens', 'coord', 'thesis_cmax', 1.5,
                       'delta_VFG=0.03,0.06,0.12,0.24,0.48;delta_IG=0.03,0.1,0.2,0.4;'
                       'delta_GI=0.02,0.05,0.1,0.2;delta_RF=0.01,0.02,0.05,0.1,0.2',
                       ('sens_rates_*.csv',), 'sens_rates_<seq>'),
    'grid': _job('grid', 'grid', 'thesis_cmax', 1.5,
                 'delta_FR=0.02,0.05,0.10,0.30,0.50;delta_IMU=0.1,0.3,0.5,0.7,0.9',
                 ('grid_*.csv',), 'grid_<seq>_FR<x>'),
    'sweep_cmax': _job('sweep', 'coord', 'thesis_cmax', 3.0,
                       'delta_IMU=0,0.1,0.5;delta_FR=0.1,0.3,0.5,0.7,0.9;n_iters=25,50,150;dt=0.01,0.05',
                       ('sweep_*_cmax_*.csv',), 'sweep_<seq>_cmax_<v>',
                       base='delta_FR=0.5,delta_IMU=0.3'),
    'sweep_imu': _job('sweep', 'coord', 'thesis_imu', 3.0, 'delta_IMU=0.1,0.3,0.5',
                      ('sweep_*_imu_*.csv',), 'sweep_<seq>_imu_<v>',
                      base='delta_FR=0.5,delta_IMU=0.3'),
    'ablation': _job('ablation', 'coord', 'thesis_cmax', 1.5,
                     'delta_shrinkI=0,1;delta_map=0.01,0.03;delta_curl=0.3,0.6',
                     ('*.csv',), 'ablation_<seq>_<v>', seqs='boxes,poster,dynamic,bicycle'),
    **{f'best_image_{m}': _job('best_image', 'grid', m, 1.5, 'dt=0.01;delta_curl=0,0.6',
                               _S(m, _ALL), f'best_image_<seq>_{m}')
       for m in ('thesis', 'thesis_cmax')},
    **{f'maps_poster_{m}': _job('maps_poster', 'coord', m, 0.88, 'n_iters=75',
                                (f'sweep_{m}.csv',), f'maps_poster_{m}', seqs='poster')
       for m in MODELS},
}
LIVE_JOB, LIVE_RUNS = {}, {}                    # section ID -> the job / runs Live made


def job_axes(job):
    return _parse_vary(SWEEP_JOBS[job]['vary'])


def job_base(job):
    """cmd_sweep's parsing of --base."""
    base = {}
    for part in filter(None, (s.strip() for s in SWEEP_JOBS[job]['base'].split(','))):
        k, _, v = part.partition('=')
        base[k.strip()] = v.strip() if k.strip() == 'poisson' else float(v)
    return base


def panel_changes(p):
    """The panel's sweepable values that differ from the report's defaults."""
    return [k for k in DELTAS + ['n_iters', 'dt'] if p[k] != DEFAULTS[k]]


def pick_job(p, jobs):
    """Live: the job of the panel's model whose axes hold a changed value, else its first."""
    mine = [j for j in jobs if SWEEP_JOBS[j]['model'] == p['model']] or list(jobs)
    ch = panel_changes(p)
    return next((j for j in mine if any(a in ch for a, _ in job_axes(j))), mine[0])


def _held(name, vals, base):
    """grid mode: an axis not being varied sits at the base, else at the adopted
    value if the axis has it, else at its first value."""
    if name in base:
        return base[name]
    adopted = {**THESIS_PARAMS, 'n_iters': ITERS_PER_FRAME, 'dt': 0.02}.get(name)
    return adopted if adopted in vals else vals[0]


def sweep_points(job, p):
    """cmd_sweep's points (run_experiments._points) cut to the panel's sweep extent."""
    J, axes, base = SWEEP_JOBS[job], job_axes(job), job_base(job)
    if p['sweep'] == 'full':
        return _points(axes, J['mode'], base)
    held = {n: _held(n, v, base) for n, v in axes} if J['mode'] == 'grid' else {}
    if p['sweep'] == 'one point':
        return [{**base, **held}]
    ch = panel_changes(p)
    name, vals = next(((n, v) for n, v in axes if n in ch), axes[0])
    return [{**base, **held, name: v} for v in vals]


def sweep_duration(p, J):
    """Live track length: the job's duration, capped at n_frames frames of 20 ms."""
    return min(J['duration'], p['n_frames'] * 0.02)


def sweep_seg(p, J):
    labels = J['seqs'].split(',')
    if p['segment'] in labels:
        return p['seg']
    print(f"the job runs {J['seqs']} only; using {labels[0]} instead of {p['segment']}")
    return next(s for s in SEGMENTS if s[3] == labels[0])


def sweep_rows(name, patterns=('*.csv',), fallback=True):
    """The job csvs in experiments/<name>/ (for sens, failing that in
    report/experiments/, as make_tables._sens_rows and text_numbers.sens_rows do),
    read with csv.DictReader. A file missing a cmd_sweep column is a [DIFF]."""
    find = lambda top: sorted(q for pat in patterns for q in glob.glob(os.path.join(top, pat)))
    paths = find(os.path.join('experiments', name))
    if not paths and name == 'sens' and fallback:
        paths = find(os.path.join('report', 'experiments'))
    rows = []
    for path in paths:
        with open(path, newline='', encoding='utf-8') as f:
            rd = csv.DictReader(f)
            gone = [c for c in SWEEP_COLS if c not in (rd.fieldnames or [])]
            if gone:
                print(f'[DIFF] {os.path.basename(path)}: missing column(s) {", ".join(gone)} '
                      '(an older csv; rerun its job)')
            rows += list(rd)
    if paths:
        print(f'   {len(rows)} rows from {len(paths)} csv(s) in {os.path.dirname(paths[0])}')
    return rows


def have_cols(rows, cols, ID):
    """Gate a section on the columns it reads; a missing one skips it (the
    [DIFF] naming it was printed by sweep_rows)."""
    gone = sorted({c for r in rows for c in cols if c not in r})
    if gone:
        print(f'({ID}: rows lack column(s) {", ".join(gone)}; the section is skipped)')
        READY[ID] = False
    return not gone


def sweep_row(rc, s, secs=0):
    """cmd_sweep's csv row for one run summary: same columns, same rounding."""
    g = lambda k: s.get(k, float('nan'))
    vals = ([rc.dataset, rc.segment_id, rc.model, round(rc.frame_duration * 1000, 1),
             rc.n_frames, rc.n_iters, rc.poisson] + [rc.params.get(d, '') for d in ALL_DELTAS]
            + [round(s['mean_err_deg_s'], 2), round(s['median_err_deg_s'], 2),
               round(s['mean_dir_err_deg'], 2), round(s['mean_beta'], 3),
               round(g('low_freq_share'), 4), round(g('contrast'), 4), round(g('curl_share'), 4),
               round(g('recon_r_mean'), 4), round(g('recon_r_min'), 4),
               round(g('recon_r_mean_inloop'), 4), round(g('recon_r_min_inloop'), 4),
               round(g('blowup_frac'), 4), round(secs)])
    return dict(zip(SWEEP_COLS, vals))


def _cfg_key(rc):
    return json.dumps({k: v for k, v in rc.to_dict().items() if k != 'out_root'},
                      sort_keys=True, default=str)


def note_sweep(p, jobs, nf):
    """Redraw: say that the saved rows are shown, and which panel values need Live."""
    off = [f'{k} {p[k]}' for k in panel_changes(p) + ['poisson'] if p[k] != DEFAULTS[k]]
    if p['n_frames'] != nf:
        off.append(f"n_frames {p['n_frames']} (saved: {nf})")
    if p['sweep'] != DEFAULTS['sweep']:
        off.append(f"sweep {p['sweep']!r}")
    print(f"Redraw: the saved rows of {', '.join(jobs)}: every point, every sequence, "
          f"report settings.")
    if off:
        print(f"No saved run has the panel's {', '.join(off)}: rerun it in Live mode.")


def run_sweep(p, ID, jobs):
    """Step 'the runs' of a sweep section. Redraw: the saved rows of the jobs.
    Live: the panel's job (pick_job), its points cut to the sweep extent, run on
    the panel's segment into walk_dir(ID), one cmd_sweep row each. Sets READY[ID]."""
    jobs = [jobs] if isinstance(jobs, str) else list(jobs)
    if p['mode'] == 'Redraw':
        J0 = SWEEP_JOBS[jobs[0]]
        note_sweep(p, jobs, int(round(J0['duration'] / 0.02)))
        rows = []
        for j in jobs:
            J = SWEEP_JOBS[j]
            got = sweep_rows(J['name'], J['csv'])
            if not got:
                need([os.path.join('experiments', J['name'], J['csv'][0])], [J['run_all']])
            rows += got
        READY[ID] = bool(rows)
        return rows
    job = pick_job(p, jobs)
    J = SWEEP_JOBS[job]
    ds, sid, t0, lab = sweep_seg(p, J)
    READY[ID], LIVE_JOB[ID], LIVE_RUNS[ID] = False, job, {}
    if not need([os.path.join('data', ds)]):
        return []
    pts, dur = sweep_points(job, p), sweep_duration(p, J)
    nfs = [int(round(dur / (pt.get('dt') or 0.02))) for pt in pts]
    print(f"Live: job {job} ({J['model']}) on {lab}, extent {p['sweep']!r}: "
          f"{len(pts)} runs, {sum(nfs)} frames in total ({dur:g} s each)")
    if p['sweep'] == 'one point' and panel_changes(p):
        print(f"   'one point' runs the base point; the panel's {', '.join(panel_changes(p))} "
              "pick the axis of 'one axis'")
    root = walk_dir(ID)
    path = os.path.join(root, f'{job}.csv')
    with open(path, 'w', newline='', encoding='utf-8') as f:
        wr = csv.writer(f)
        wr.writerow(SWEEP_COLS)
        for pt, nf in zip(pts, nfs):
            poisson, dt, n_iters, deltas = _split(pt)
            rc = run_cfg(ds, sid, t0, J['model'], dt or 0.02, nf, out_root=root,
                         poisson=poisson, deltas=deltas, n_iters=n_iters)
            E.experiment_tracking(rc, save_frames=False)
            row = sweep_row(rc, experiment_tracking.read_summary(rc.output_dir))
            LIVE_RUNS[ID][_cfg_key(rc)] = rc.output_dir
            wr.writerow([row[c] for c in SWEEP_COLS])
            f.flush()
            print('   ' + ' '.join(f'{k}={v}' for k, v in sorted(pt.items()))
                  + f"  err {row['err']} deg/s, beta {row['beta']}")
    with open(path, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    READY[ID] = True
    return rows


def check_live_sweep(p, ID, job, rows):
    """Live same-code check: run cmd_sweep itself on the job, with its run step
    replaced by a recorder that hands back the Live runs' summaries (a point
    the Live run did not make is refused). Same configs? Same csv rows?"""
    J = SWEEP_JOBS[job]
    runs = LIVE_RUNS.get(ID, {})
    seen = []
    def record(E_, rc, save_frames, stride):
        k = _cfg_key(rc)
        if k not in runs:
            raise LookupError('not a point of the Live run')
        seen.append(k)
        return experiment_tracking.read_summary(runs[k]), 0
    name = os.path.join('walkthrough', ID, 'check')
    args = argparse.Namespace(vary=J['vary'], base=J['base'], mode=J['mode'], model=J['model'],
                              duration=sweep_duration(p, J), dt=0.02,
                              sequences=sweep_seg(p, J)[3], name=name, out=f'{job}_check.csv',
                              no_frames=True, frame_stride=1)
    said = io.StringIO()
    with redirect(run_experiments, _run=record), contextlib.redirect_stdout(said), \
         contextlib.redirect_stderr(said):
        run_experiments.cmd_sweep(E, args)
    miss = len(set(runs) - set(seen))
    print(f'[ok] config ({len(runs)} of {len(runs)} Live points are cmd_sweep points)' if not miss
          else f'[DIFF] config: {miss} of {len(runs)} Live points were not made by cmd_sweep')
    with open(os.path.join('experiments', name, f'{job}_check.csv'), newline='') as f:
        theirs = {tuple(r[c] for c in SWEEP_COLS[3:16]): r for r in csv.DictReader(f)}
    num = SWEEP_COLS[16:-1]
    for r in rows:
        t = theirs.get(tuple(r[c] for c in SWEEP_COLS[3:16]))
        what = 'cmd_sweep row ' + ' '.join(f'{c}={r[c]}' for c in SWEEP_COLS[3:16]
                                           if c in ('dt_ms', 'n_iters') or r[c] != str(THESIS_PARAMS.get(c)))
        if t is None:
            print(f'[DIFF] {what}: no such row in the cmd_sweep csv')
        else:
            check(what, {c: float(r[c]) for c in num}, {c: float(t[c]) for c in num})


def report_figs(fn, ID, show=False, **attrs):
    """Call a make_figures function with FIG/save redirected (never report/figures/);
    every figure it saves is kept, and shown + saved to walk_dir(ID) if show."""
    figs = []
    def keep(fig, path, show_=False):
        figs.append(fig)
        if show:
            style_save(fig, path, show=True)
    out = walk_dir(ID) if show else os.path.join(walk_dir(ID), 'check')
    quiet = contextlib.nullcontext() if show else contextlib.redirect_stdout(io.StringIO())
    with redirect(make_figures, FIG=out, save=keep, **attrs), quiet:
        fn(E)
    if not show:
        for fg in figs:
            plt.close(fg)
    return figs


def fig_images(figs):
    """Every image drawn in the figures, flattened in axes order (loose image check)."""
    arrs = [np.ma.filled(np.ma.asarray(im.get_array(), float), np.nan).ravel()
            for fg in figs for ax in fg.axes for im in ax.images]
    return np.concatenate(arrs) if arrs else np.array([])

In [ ]:
# step 1 -- the runs: the saved sens rows (Redraw) or the panel's sens job (Live)
p = params()
JOBS_A8 = ['sens_cmax', 'sens_imu', 'sens_rates']
rows_A8 = run_sweep(p, 'A8', JOBS_A8)
if READY['A8']:
    have_cols(rows_A8, SWEEP_COLS[:-1], 'A8')

In [ ]:
%%section A8
# step 2 -- the rows as cells: make_tables.py::_sens_rows keys every row by
# (model, the 8 SENS_KEYS); a point varies one key away from the adopted base
from make_tables import SENS_KEYS, SEQS as TAB_SEQS, fmt, mark_best, tabular
from text_numbers import BASE as TN_BASE, axis as tn_axis   # one-at-a-time selection
cell_A8 = {}
for r in rows_A8:
    key = (r['model'],) + tuple(round(float(r[k]), 6) for k in SENS_KEYS)
    cell_A8.setdefault(key, {})[r['sequence']] = r
base_A8 = dict({k: THESIS_PARAMS[k] for k in SENS_KEYS[:6]}, n_iters=ITERS_PER_FRAME, dt_ms=20.0)
pts_A8 = []
for key, by_seq in cell_A8.items():
    off = [k for k, v in zip(SENS_KEYS, key[1:]) if abs(v - base_A8[k]) > 1e-9]
    pts_A8.append([key[0], ','.join(off) or '(base)',
                   ','.join(f'{v:g}' for k, v in zip(SENS_KEYS, key[1:]) if k in off) or '-',
                   len(by_seq), ' '.join(sorted(s.split('_')[0] for s in by_seq))])
print(f'{len(cell_A8)} points; adopted base: {base_A8}')
display(pd.DataFrame(pts_A8, columns=['model', 'varied', 'value', 'n seq', 'sequences'])
        .sort_values(['model', 'varied']).reset_index(drop=True))

### tab:sweep -- `make_tables.py::table_sweep`

One block per parameter. The adopted value of each block is bold; the lowest
error of each block on each sequence is underlined (the δ_anchor = 0 ablation
row excluded). err in deg/s, dir in degrees.

In [ ]:
%%section A8
# step 3 -- table_sweep, unrolled (its block() is copied; fmt/mark_best/tabular reused)
seen_A8 = []
def block_A8(title, model, name, values, fmt_label=lambda v: f'{v:g}'):
    out = [r'\midrule', rf'\multicolumn{{12}}{{@{{}}l}}{{{title}}}\\']
    rows = []
    for v in values:
        pt = dict(base_A8, **{name: v})
        rows.append((v, cell_A8.get((model,) + tuple(round(float(pt[k]), 6) for k in SENS_KEYS), {})))
    err = {}
    for ds, *_ in TAB_SEQS:
        err[ds] = mark_best({v: fmt(rs[ds]['err']) if ds in rs else '---'
                             for v, rs in rows if not (name == 'delta_IMU' and v == 0)},
                            lambda x: x, mark=r'\underline')
    for v, rs in rows:
        label = fmt_label(v)
        lab = rf'$\mathbf{{{label}}}$' if abs(v - base_A8[name]) < 1e-9 else f'${label}$'
        vals = ' & '.join((f'{err[ds].get(v, fmt(rs[ds]["err"]))} & {fmt(rs[ds]["dir"])}')
                          if ds in rs else '--- & ---' for ds, *_ in TAB_SEQS)
        out.append(f'& {lab:<20}& {vals} \\\\')
        seen_A8.append([model, name, v] + [float(rs[ds]['err']) if ds in rs else np.nan
                                           for ds, *_ in TAB_SEQS])
    return out

ms_A8 = lambda v: rf'{v:g}\,\mathrm{{ms}}'
body_A8 = (block_A8(r'\emph{anchor weight} $\delta_{\text{anchor}}$, contrast anchor', 'thesis_cmax',
                    'delta_IMU', [0.0, 0.1, 0.3, 0.5, 0.7, 0.9],
                    lambda v: r'0\ \text{(abl.)}' if v == 0 else f'{v:g}')
           + block_A8(r'\emph{anchor weight} $\delta_{\text{anchor}}$, gyroscope', 'thesis_imu',
                      'delta_IMU', [0.1, 0.3, 0.5, 0.7, 0.9])
           + block_A8(r'\emph{kinematic weight} $\delta_{F\!R}$', 'thesis_cmax',
                      'delta_FR', [0.02, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9])
           + block_A8(r'\emph{flow constraint} $\delta_{V\!F\!G}$', 'thesis_cmax',
                      'delta_VFG', [0.03, 0.06, 0.12, 0.24, 0.48])
           + block_A8(r'\emph{flow from} $\wvec$, $\delta_{R\!F}$', 'thesis_cmax',
                      'delta_RF', [0.01, 0.02, 0.05, 0.1, 0.2])
           + block_A8(r'\emph{gradient from intensity} $\delta_{I\!G}$', 'thesis_cmax',
                      'delta_IG', [0.03, 0.1, 0.2, 0.4])
           + block_A8(r'\emph{intensity from gradient} $\delta_{G\!I}$', 'thesis_cmax',
                      'delta_GI', [0.02, 0.05, 0.1, 0.2])
           + block_A8(r'\emph{iterations per frame}', 'thesis_cmax', 'n_iters', [25, 50, 75, 150])
           + block_A8(r'\emph{accumulation window} $\Delta t$', 'thesis_cmax', 'dt_ms',
                      [5.0, 10.0, 20.0, 40.0], ms_A8))
head_A8 = ['& & ' + '\n& '.join(rf'\multicolumn{{2}}{{c}}{{{d}}}' for _, d, *_ in TAB_SEQS) + r' \\',
           ''.join(rf'\cmidrule(lr){{{3 + 2*i}-{4 + 2*i}}}' for i in range(len(TAB_SEQS))),
           '& & ' + ' & '.join(['err & dir'] * len(TAB_SEQS)) + r' \\']
table_A8 = tabular('@{}ll' + 'rr' * len(TAB_SEQS) + '@{}', head_A8, body_A8)
print('err (deg/s) per block, value and sequence:')
display(pd.DataFrame(seen_A8, columns=['model', 'parameter', 'value'] + [s for _, _, s, _ in TAB_SEQS]))
print(table_A8)

### fig_sensitivity -- `make_figures.py::fig_sensitivity` (not in the tex)

`sensitivity_data` averages err and read-out |r| over the three real and the
two synthetic sequences per swept value; its κ panel pools A9's grid by
κ = δ_FR/δ_anchor, and the δ_anchor = 0 row is the "no anchor" point. **Flag:**
it reads `experiments/sens/` only, with no fallback to `report/experiments/`
(unlike table_sweep and text_numbers), so on a machine holding only the
fallback csvs its one-at-a-time panels are empty while tab:sweep is full.
The data are unrolled; the drawing is the report's own code fed these data.

In [ ]:
%%section A8
# step 4 -- make_figures.py::sensitivity_data, unrolled; then fig_sensitivity draws it
from make_figures import SENS_REAL, SENS_SYN, SENS_BASE, SENS_AXES
if p['mode'] == 'Redraw':
    grid_rows_A8 = sweep_rows('grid', ('*.csv',))                 # A9's grid: the kappa panel
    fig_rows_A8 = sweep_rows('sens', ('sens_*.csv',), fallback=False)   # experiments/sens only
    if not fig_rows_A8:
        print('   fig_sensitivity: no csv in experiments/sens/ (it has no fallback): '
              'its one-at-a-time panels stay empty')
else:
    grid_rows_A8, fig_rows_A8 = [], rows_A8

def sensitivity_data_A8(grid_rows, sens_rows):
    """make_figures.sensitivity_data on the given rows: {panel: {value: {kind: (err, |r|)}}}."""
    def mean(rows):
        out = {}
        for kind, seqs in (('real', SENS_REAL), ('syn', SENS_SYN)):
            sel = [r for r in rows if r['sequence'] in seqs]
            if sel:
                out[kind] = (np.mean([float(r['err']) for r in sel]),
                             np.mean([float(r['recon_mean']) for r in sel]))
        return out
    data, by_k = {}, {}
    for r in (r for r in grid_rows if r['model'] == 'thesis_cmax'):
        by_k.setdefault(round(float(r['delta_FR']) / float(r['delta_IMU']), 4), []).append(r)
    data['kappa'] = {k: mean(v) for k, v in sorted(by_k.items())}
    sens = [r for r in sens_rows if r['model'] == 'thesis_cmax']
    for key, _ in SENS_AXES + [('delta_IMU', '')]:
        by_v = {}
        for r in sens:
            if all(abs(float(r[k]) - v) < 1e-9 for k, v in SENS_BASE.items() if k != key):
                by_v.setdefault(float(r[key]), []).append(r)
        data[key] = {v: mean(rs) for v, rs in sorted(by_v.items())}
    return data, data.pop('delta_IMU').get(0.0)          # delta_anchor = 0 is thesis

data_A8, no_anchor_A8 = sensitivity_data_A8(grid_rows_A8, fig_rows_A8)
display(pd.DataFrame([[pn, v, kind, e, rr] for pn, d in data_A8.items() for v, kk in d.items()
                      for kind, (e, rr) in kk.items()],
                     columns=['panel', 'value', 'kind', 'err deg/s', '|r| read-out']))
print('no anchor (delta_anchor = 0):', {k: tuple(round(x, 2) for x in v)
                                        for k, v in (no_anchor_A8 or {}).items()})
figs_A8 = report_figs(make_figures.fig_sensitivity, 'A8', show=True,
                      sensitivity_data=lambda: (data_A8, no_anchor_A8))

In [ ]:
%%section A8
# step 5 -- the text numbers (Redraw only: they read the report's rows)
from text_numbers import REAL as TN_REAL, SYN as TN_SYN
def text_numbers_A8(rows, grid_rows):
    """The five text_numbers functions on the sens rows (= text_numbers.sens_rows())."""
    fl = lambda r, k='err': float(r[k])
    lines = {k: [] for k in ('sensitivity_both', 'sweep_ends', 'anchor_ablation',
                             'anchors_across_weights', 'kappa_rows')}
    # text_numbers.sensitivity_both: real/synthetic means per swept value, grid by kappa
    L = lines['sensitivity_both']
    L.append('value: err real/synthetic (deg/s) | read-out |r| real/synthetic | in-loop |r| '
             'real/synthetic | fraction of frames in an excursion real/synthetic')
    def line_A8(label, rs):
        cells = []
        for k, fm in (('err', '5.1f'), ('recon_mean', '.2f'), ('recon_inloop', '.2f'), ('blowup', '.3f')):
            m = [np.mean([fl(r, k) for r in rs if r['sequence'] in [s[0] for s in g]])
                 for g in (TN_REAL, TN_SYN)]
            cells.append(f'{m[0]:{fm}} {m[1]:{fm}}')
        L.append(f'  {label:<12s} ' + ' | '.join(cells))
    bk = {}
    for r in (r for r in grid_rows if r['model'] == 'thesis_cmax'):
        an = r.get('delta_IMU', r.get('delta_anchor'))
        bk.setdefault(round(fl(r, 'delta_FR') / float(an), 4), []).append(r)
    if bk:
        L.append('grid, by kappa = delta_FR / delta_anchor:')
        for k in sorted(bk):
            line_A8(f'kappa {k:g}', bk[k])
        spread = 0.0
        for rs in bk.values():
            for s in SEGMENTS:
                v = [fl(r, 'recon_mean') for r in rs if r['sequence'] == s[0]]
                if len(v) > 1:
                    spread = max(spread, max(v) - min(v))
        L.append(f'  largest |r| spread between settings of equal kappa: {spread:.3f}')
    for model, keys in (('thesis_cmax', ('delta_IMU', 'delta_FR', 'delta_VFG', 'delta_RF', 'delta_IG',
                                         'delta_GI', 'n_iters', 'dt_ms')), ('thesis_imu', ('delta_IMU',))):
        for key in keys:
            by_v = {}
            for s in SEGMENTS:
                for v, r in tn_axis(rows, key, s[0], model).items():
                    by_v.setdefault(v, []).append(r)
            if by_v:
                L.append(f'{model} {key}:')
                for v in sorted(by_v):
                    line_A8(f'{v:g}', by_v[v])
    lo, hi = 0.01, TN_BASE['delta_RF']
    cells = []
    for s in TN_REAL:
        a = tn_axis(rows, 'delta_RF', s[0])
        if lo in a and hi in a:
            cells.append(f'{s[3]} err {fl(a[lo]):.1f} vs {fl(a[hi]):.1f}, |r| '
                         f'{fl(a[lo], "recon_mean"):.2f} vs {fl(a[hi], "recon_mean"):.2f}')
    if cells:
        L.append(f'delta_RF {lo:g} vs {hi:g} per real sequence: ' + '; '.join(cells))
    # text_numbers.sweep_ends: the best value of each axis per sequence, and its range
    L = lines['sweep_ends']
    for key in ('delta_FR', 'delta_IMU', 'delta_VFG', 'delta_RF', 'delta_IG', 'delta_GI', 'n_iters', 'dt_ms'):
        cells = []
        for s in SEGMENTS:
            a = {k: v for k, v in tn_axis(rows, key, s[0]).items() if not (key == 'delta_IMU' and k == 0)}
            if a:
                e = {k: fl(v) for k, v in a.items()}
                cells.append(f'{s[3]} best {min(e, key=e.get):g} (range {max(e.values())-min(e.values()):.2f})')
        L.append(f'{key:10s} ' + '; '.join(cells))
    # text_numbers.anchor_ablation: err over the anchor weight, both anchors
    L = lines['anchor_ablation']
    for model in ('thesis_cmax', 'thesis_imu'):
        for s in SEGMENTS:
            a = tn_axis(rows, 'delta_IMU', s[0], model)
            L.append(f'{model:11s} {s[3]:8s} ' + ', '.join(f'{k:g}: {fl(v):.1f}' for k, v in a.items()))
    for model in ('thesis_cmax', 'thesis_imu'):
        syn = [tn_axis(rows, 'delta_IMU', s[0], model) for s in TN_SYN]
        ws = sorted(set.intersection(*[set(a) for a in syn])) if syn else []
        if ws:
            L.append(f'{model} synthetic mean: ' + ', '.join(
                f'{w:g}: {np.mean([fl(a[w]) for a in syn]):.1f}' for w in ws))
    # text_numbers.anchors_across_weights: CMax minus gyro anchor at each weight
    L = lines['anchors_across_weights']
    worst = 0.0
    for seg in SEGMENTS:
        c, i = tn_axis(rows, 'delta_IMU', seg[0]), tn_axis(rows, 'delta_IMU', seg[0], 'thesis_imu')
        diffs = {w: fl(c[w]) - fl(i[w]) for w in c if w in i and w > 0}
        if seg in TN_REAL:
            worst = max([worst] + [abs(v) for v in diffs.values()])
        L.append(f'{seg[3]:8s} CMax minus gyro: ' + ', '.join(f'{w:g}: {v:+.1f}' for w, v in diffs.items())
                 + (f';  no anchor: {fl(c[0.0]):.1f}' if 0.0 in c else ''))
    L.append(f'largest |CMax - gyro| on the real sequences: {worst:.1f} deg/s')
    # text_numbers.kappa_rows: anchor 0.1 vs dFR 0.5 (both kappa 1); beta over dFR
    L = lines['kappa_rows']
    for seg in SEGMENTS:
        an, fr = tn_axis(rows, 'delta_IMU', seg[0]), tn_axis(rows, 'delta_FR', seg[0])
        if 0.1 in an and 0.5 in fr:
            b = [fl(r, 'beta') for r in fr.values()]
            mono = all(x < y for x, y in zip(b, b[1:]))
            L.append(f'{seg[3]:8s} anchor 0.1 vs dFR 0.5 (kappa 1): {fl(an[0.1]):.2f} vs {fl(fr[0.5]):.2f}; '
                     f'beta over dFR {min(fr):g}-{max(fr):g}: {b[0]:.2f} -> {b[-1]:.2f}, '
                     f'{"rises at every step" if mono else "NOT monotonic"}')
    return lines


lines_A8 = {}
if p['mode'] == 'Redraw':
    lines_A8 = text_numbers_A8(rows_A8, grid_rows_A8)
    for k, ls in lines_A8.items():
        print(f'== {k}'); print('\n'.join(ls) or '(no rows)')
else:
    print('Live: the text numbers read the report rows; switch to Redraw to see them.')

In [ ]:
%%section A8
# same-code check -- cmd_sweep (Live); table_sweep, sensitivity_data, text_numbers (Redraw)
if p['mode'] == 'Live':
    check_live_sweep(p, 'A8', LIVE_JOB['A8'], rows_A8)
else:
    check('table_sweep', nums(table_A8.splitlines()), nums(make_tables.table_sweep().splitlines()))
    flat_A8 = lambda d, na: {**{f'{pn}|{v:g}|{kind}|{j}': x for pn, dd in d.items() for v, kk in dd.items()
                                for kind, xs in kk.items() for j, x in enumerate(xs)},
                             **{f'no_anchor|{kind}|{j}': x for kind, xs in (na or {}).items()
                                for j, x in enumerate(xs)}}
    theirs_A8 = make_figures.sensitivity_data()
    check('fig_sensitivity data', flat_A8(data_A8, no_anchor_A8), flat_A8(*theirs_A8))
    for name in lines_A8:
        with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
            getattr(text_numbers, name)()
            theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
        check(f'text_numbers.{name}', nums(lines_A8[name]), nums(theirs))

<a id="A9"></a>

## A9 · The (δ_FR, δ_anchor) grid · [↑ overview](#overview)

**Question.** δ_FR against the anchor weight on a full 5×5 grid
(δ_FR ∈ {0.02, 0.05, 0.1, 0.3, 0.5} × δ_anchor ∈ {0.1, 0.3, 0.5, 0.7, 0.9}),
CMax-anchor, 1.5 s, every sequence: where is each sequence best, and does the
error depend on the two only through κ = δ_FR/δ_anchor?

**Made by** `report/run_experiments.py::cmd_sweep`
(`--what sweep --name grid --mode grid --duration 1.5`), `run_all.sh` jobs
`grid_<seq>_FR<x>` (one δ_FR row per job; here as one job, the union) → runs and
`grid_<seq>_FR<x>.csv` in `experiments/grid/`.

**Read by** `report/make_tables.py::table_grid` (**not in the tex**: written
to `table_grid.tex`, never pasted), `::table_params` (tab:params: its δ_FR and
δ_anchor rows are the values "set after a parameter sweep"; the table is typed
into the tex by hand, so it is compared with the tex here),
`report/text_numbers.py::grid`. Also the κ panel of `fig_sensitivity` and the
grid half of `text_numbers.sensitivity_both`, unrolled in A8.

In [ ]:
# step 1 -- the runs: the saved grid rows (Redraw) or the panel's grid points (Live)
p = params()
rows_A9 = run_sweep(p, 'A9', 'grid')
if READY['A9']:
    have_cols(rows_A9, SWEEP_COLS[:-1], 'A9')

In [ ]:
%%section A9
# step 2 -- the grid per sequence: err (deg/s) over delta_FR (rows) x delta_anchor (columns)
grid_A9 = {}
for r in rows_A9:
    grid_A9.setdefault(r['sequence'], {})[(float(r['delta_FR']), float(r['delta_IMU']))] = float(r['err'])
for ds_, g in grid_A9.items():
    print(f'{ds_}: err deg/s, {len(g)} cells')
    display(pd.Series(g).unstack().rename_axis(index='delta_FR', columns='delta_anchor').round(2))

### tab:grid -- `make_tables.py::table_grid` (not in the tex)

Each sequence reduced to its adopted cell and its best cell, with κ.

In [ ]:
%%section A9
# step 3 -- table_grid, unrolled (every csv row, any model, keyed by sequence)
from make_tables import fmt, tabular, SEQS as TAB_SEQS, ADOPTED as TAB_ADOPTED
body_A9, opt_A9 = [], {}
for ds_, disp, *_ in TAB_SEQS:
    g = grid_A9.get(ds_)
    if not g:
        continue
    (fr, an), e = min(g.items(), key=lambda kv: kv[1])
    ad = g.get(TAB_ADOPTED)
    opt_A9[ds_] = (e, fr, an)
    body_A9.append(f'{disp:<18}& {fmt(ad, 2)} & {fmt(TAB_ADOPTED[0] / TAB_ADOPTED[1], 2)} & '
                   f'{fmt(e, 2)} & {fr:g} & {an:g} & {fmt(fr / an, 2)} \\\\')
head_A9 = [r' & \multicolumn{2}{c}{adopted} & \multicolumn{4}{c}{grid optimum} \\',
           r'\cmidrule(lr){2-3}\cmidrule(lr){4-7}',
           r'sequence & err & $\kappa$ & err & $\delta_{F\!R}$ & $\delta_{\text{anchor}}$ & $\kappa$ \\',
           r'\midrule']
table_A9 = tabular('@{}lrrrrrr@{}', head_A9, body_A9)
print(table_A9)

### tab:params -- the grid's part of `make_tables.py::table_params`

table_params prints config.py's step sizes; two of them, δ_FR and δ_anchor
(`delta_IMU`), are the ones the report says were set after a parameter sweep.
Below: those two rows, next to the grid's optimum per sequence and per kind of
data (the cell with the lowest mean error over the kind's sequences).

In [ ]:
%%section A9
# step 4 -- table_params' delta_FR and delta_anchor rows, against the grid optimum
from make_tables import COOK_PARAMS as TAB_COOK, THESIS_PARAMS as TAB_THESIS
from report_runs import SYNTHETIC as RR_SYN
pcell_A9 = lambda d, k: f'{d[k]:.2f}' if k in d else '---'
rows_params_A9 = {k: [pcell_A9(TAB_COOK, k), pcell_A9(TAB_THESIS, k)] for k in ('delta_FR', 'delta_IMU')}
for k, (c, m) in rows_params_A9.items():
    print(f'table_params {k:9s}: Cook {c}, Martel {m}')
def grid_optimum_A9(grid):
    """The cell of lowest mean error over each kind's sequences (cells all of them have)."""
    out = {}
    for kind in ('real', 'synthetic'):
        seqs = [s[0] for s in SEGMENTS if (s[3] in RR_SYN) == (kind == 'synthetic') and s[0] in grid]
        cells = set.intersection(*[set(grid[s]) for s in seqs]) if seqs else set()
        if cells:
            mean = {c: np.mean([grid[s][c] for s in seqs]) for c in cells}
            best = min(mean, key=mean.get)
            out[kind] = best
            print(f'grid optimum, {kind} mean over {len(seqs)}: dFR {best[0]:g}, anchor {best[1]:g} '
                  f'(kappa {best[0]/best[1]:.2f}), {mean[best]:.2f} deg/s; adopted '
                  f'({TAB_ADOPTED[0]:g}, {TAB_ADOPTED[1]:g}): {mean.get(TAB_ADOPTED, np.nan):.2f} deg/s')
    return out

kinds_A9 = grid_optimum_A9(grid_A9)
for ds_, (e, fr, an) in opt_A9.items():
    print(f'   {ds_:18s} optimum dFR {fr:g}, anchor {an:g}: {e:.2f} deg/s')

In [ ]:
%%section A9
# step 5 -- text_numbers.grid, unrolled (Redraw only: it reads the report's rows)
def text_grid_A9(rows):
    from text_numbers import REAL as TN_REAL, SYN as TN_SYN
    from report_runs import ADOPTED as RR_ADOPTED
    L = []
    if not rows:
        return ['   (no grid csv)']
    tab = {}
    for r in rows:
        an = r.get('delta_IMU', r.get('delta_anchor'))     # older grid csv: delta_anchor
        tab[(r['sequence'], round(float(r['delta_FR']), 4), round(float(an), 4))] = float(r['err'])
    seqs = [s for s in SEGMENTS if any(k[0] == s[0] for k in tab)]
    spread = 0.0
    for s in seqs:
        by_k = {}
        for (q, fr, an), e in tab.items():
            if q == s[0]:
                by_k.setdefault(round(fr / an, 3), []).append(e)
        own = max([0.0] + [max(v) - min(v) for v in by_k.values() if len(v) > 1])
        spread = max(spread, own)
        ks = sorted(by_k)
        mean = {k: np.mean(by_k[k]) for k in ks}
        best = min(ks, key=mean.get)
        errs = [mean[k] for k in ks]
        mono = all(x <= y for x, y in zip(errs, errs[1:]))
        ad = tab.get((s[0], RR_ADOPTED['delta_FR'], RR_ADOPTED['delta_IMU']))
        L.append(f'{s[3]:8s} best kappa {best:g} ({mean[best]:.2f}); smallest kappa {ks[0]:g} '
                 f'({mean[ks[0]]:.2f}){" error rises with kappa throughout" if mono else ""}; '
                 f'adopted {ad:.2f}; kappa 0.6 (0.3/0.5) {tab.get((s[0], 0.3, 0.5), np.nan):.2f}; '
                 f'start (0.5/0.3) {tab.get((s[0], 0.5, 0.3), np.nan):.2f}; '
                 f'equal-kappa spread {own:.2f}')
    L.append(f'largest spread between settings of equal kappa: {spread:.2f} deg/s')
    for name, group in (('real', TN_REAL), ('synthetic', TN_SYN)):
        m = lambda fr, an: np.mean([tab[(s[0], fr, an)] for s in group if (s[0], fr, an) in tab])
        L.append(f'{name} mean: adopted {m(RR_ADOPTED["delta_FR"], RR_ADOPTED["delta_IMU"]):.2f}, '
                 f'start {m(0.5, 0.3):.2f}, kappa 0.6 {m(0.3, 0.5):.2f}')
    return L

lines_A9 = []
if p['mode'] == 'Redraw':
    lines_A9 = text_grid_A9(rows_A9)
    print('== grid'); print('\n'.join(lines_A9))
else:
    print('Live: the text numbers read the report rows; switch to Redraw to see them.')

In [ ]:
%%section A9
# same-code check -- cmd_sweep (Live); table_grid, table_params, text_numbers.grid (Redraw)
if p['mode'] == 'Live':
    check_live_sweep(p, 'A9', LIVE_JOB['A9'], rows_A9)
else:
    check('table_grid', nums(table_A9.splitlines()), nums(make_tables.table_grid().splitlines()))
    tp_A9 = make_tables.table_params().splitlines()
    theirs = {k: nums([l for l in tp_A9 if tag in l][0].split('&')[2:]) for k, tag in
              (('delta_FR', r'from $\Fmap$'), ('delta_IMU', 'anchor~'))}
    mine = {k: nums([v for v in vals if v != '---']) for k, vals in rows_params_A9.items()}
    check('table_params rows delta_FR, delta_anchor', np.concatenate(list(mine.values())),
          np.concatenate(list(theirs.values())))
    tex_A9 = open(os.path.join('report', 'bare_jrnl.tex'), encoding='utf-8').read()
    blk = tex_A9[tex_A9.index(r'\label{tab:params}'):]
    blk = blk[blk.index(r'\begin{tabular}'):blk.index(r'\end{tabular}')]
    check('table_params vs tab:params typed in the tex', nums(blk.splitlines()), nums(tp_A9[:-2]))
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        text_numbers.grid()
        theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
    check('text_numbers.grid', nums(lines_A9), nums(theirs))

<a id="A10"></a>

## A10 · Old coordinate sweep (mostly unread) · [↑ overview](#overview)

**Question.** One axis at a time around the step sizes the work started from
(δ_FR 0.5, δ_anchor 0.3), 3 s: δ_anchor, δ_FR, iterations, window (CMax-anchor)
and δ_anchor (thesis+IMU). The sens sweep (A8) replaced it.

**Made by** `report/run_experiments.py::cmd_sweep`
(`--what sweep --name sweep --mode coord --base delta_FR=0.5,delta_IMU=0.3`,
3 s), `run_all.sh` jobs `sweep_<seq>_cmax_<v>` (13 per sequence) and
`sweep_<seq>_imu_<v>` (3 per sequence) → runs and
`sweep_<seq>_{cmax,imu}_<v>.csv` in `experiments/sweep/`.

**Read by** only `report/text_numbers.py::synthetic_beta_original`, which
reads the base point of the two synthetic sequences,
`sweep_{bicycle,street}_cmax_delta_FR=0.5.csv` (that function is unrolled in
A6; here it is only checked). **⚠ Flag:** the other 78 of the 80 `sweep_*`
jobs are read by no table, figure or text number: candidates for removal from
`run_all.sh`.

In [ ]:
# step 1 -- the runs: the saved sweep rows (Redraw) or the panel's job (Live; 3 s capped by n_frames)
p = params()
rows_A10 = run_sweep(p, 'A10', ['sweep_cmax', 'sweep_imu'])
if READY['A10']:
    have_cols(rows_A10, SWEEP_COLS[:-1], 'A10')

In [ ]:
%%section A10
# step 2 -- run_all.sh's sweep_* jobs (one per axis value) and the ones anything reads
READ_A10 = [f'sweep_{lab}_cmax_delta_FR=0.5.csv' for lab in ('bicycle', 'street')]  # synthetic_beta_original
jobs_A10 = [f'sweep_{s[3]}_{kind}_{k}={v:g}' for s in SEGMENTS
            for kind, job in (('cmax', 'sweep_cmax'), ('imu', 'sweep_imu'))
            for k, vals in job_axes(job) for v in vals]
on_disk_A10 = {os.path.basename(q) for q in glob.glob(os.path.join('experiments', 'sweep', 'sweep_*.csv'))}
unread_A10 = [j for j in jobs_A10 if f'{j}.csv' not in READ_A10]
print(f'{len(jobs_A10)} sweep_* jobs in run_all.sh; {len(on_disk_A10)} of their csvs on disk here; '
      f'read: {len(READ_A10)} ({", ".join(READ_A10)})')
print(f'unread, candidates for removal from run_all.sh ({len(unread_A10)}):')
for lab in [s[3] for s in SEGMENTS]:
    print(f'   {lab:8s} ' + ' '.join(j.split(f'sweep_{lab}_')[1] for j in unread_A10
                                  if j.startswith(f'sweep_{lab}_')))

In [ ]:
%%section A10
# step 3 -- the base point (delta_FR 0.5, anchor 0.3, 75 iterations, 20 ms) of every sequence
is_base_A10 = lambda r: (r['model'] == 'thesis_cmax' and abs(float(r['delta_FR']) - 0.5) < 1e-9
                         and abs(float(r['delta_IMU']) - 0.3) < 1e-9 and float(r['dt_ms']) == 20.0
                         and int(float(r['n_iters'])) == ITERS_PER_FRAME)
base_A10 = {r['sequence']: r for r in rows_A10 if is_base_A10(r)}
display(pd.DataFrame([{k: r[k] for k in ('sequence', 'n_frames', 'delta_FR', 'delta_IMU', 'err',
                                         'median', 'dir', 'beta', 'recon_mean', 'blowup')}
                      for r in base_A10.values()]))
print('err, median in deg/s, dir in deg; read by the text: the synthetic rows only')

In [ ]:
%%section A10
# same-code check -- cmd_sweep (Live); text_numbers.synthetic_beta_original (Redraw)
if p['mode'] == 'Live':
    check_live_sweep(p, 'A10', LIVE_JOB['A10'], rows_A10)
else:
    from text_numbers import SYN as TN_SYN
    mine = []
    for s in TN_SYN:                      # its rows: the whole base-point file of each synthetic sequence
        path = os.path.join('experiments', 'sweep', f'sweep_{s[3]}_cmax_delta_FR=0.5.csv')
        if need([path], [f'sweep_{s[3]}_cmax_delta_FR=0.5']):
            for r in sweep_rows('sweep', (os.path.basename(path),)):
                mine.append(f'{s[3]:8s} dFR {r["delta_FR"]} anchor {r["delta_IMU"]}: '
                            f'beta {float(r["beta"]):.3f}, err {float(r["err"]):.2f}')
    print('\n'.join(mine))
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        text_numbers.synthetic_beta_original()
        theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
    check('text_numbers.synthetic_beta_original', nums(mine), nums(theirs))

<a id="A11"></a>

## A11 · Regulariser ablation · [↑ overview](#overview)

**Question.** What does each regulariser do on its own — shrink-I
(δ_shrinkI = 1), the MAP prior (δ_map 0.01, 0.03) and the curl term (δ_curl 0.3,
0.6) — to the ω error, the read-out and in-loop |r|, and the excursions?
CMax-anchor, 1.5 s, four sequences (street repeats bicycle's trajectory).

**Made by** `report/run_experiments.py::cmd_sweep`
(`--what sweep --name ablation --mode coord --duration 1.5`), `run_all.sh`
jobs `ablation_<seq>_<v>` (one value per job; `delta_shrinkI=0` is the "none"
point) → runs and `sweep_<seq>_<v>.csv` in `experiments/ablation/`.

**Read by** `report/make_tables.py::table_regularisers` (tab:regularisers),
`report/make_figures.py::fig_regularisers` (fig:regularisers),
`report/text_numbers.py::regularisers` (its curl-share half reads the Table III
and curl runs in `results/`, A3/A4).

In [ ]:
# step 1 -- the runs: the saved ablation rows (Redraw) or the panel's ablation points (Live)
p = params()
rows_A11 = run_sweep(p, 'A11', 'ablation')
if READY['A11']:
    have_cols(rows_A11, SWEEP_COLS[:-1], 'A11')

In [ ]:
%%section A11
# step 2 -- table_regularisers, unrolled: each term alone, mean over the four sequences
from make_tables import REGULARISERS as TAB_REG, ABLATION_SEQS, fmt, tabular
def table_regularisers_A11(rows):
    terms = ('delta_shrinkI', 'delta_map', 'delta_curl')
    stats = []
    for label, on in TAB_REG:
        want = {k: on.get(k, 0.0) for k in terms}
        hit = {r['sequence']: r for r in rows
               if r['model'] == 'thesis_cmax' and int(float(r['n_iters'])) == ITERS_PER_FRAME
               and all(abs(float(r[k]) - v) < 1e-9 for k, v in want.items())}
        got = [hit[ds] for ds in ABLATION_SEQS if ds in hit]
        mean = (lambda k: float(np.mean([abs(float(r[k])) for r in got]))
                if len(got) == len(ABLATION_SEQS) else None)     # all four, or '---'
        stats.append((label, [mean('err'), mean('recon_mean'), mean('recon_inloop'), mean('blowup')],
                      len(got)))
    pick = [min, max, max, min]                 # bold: lowest err and blow-up, highest |r|
    best = [f([s[1][c] for s in stats if s[1][c] is not None], default=None)
            for c, f in enumerate(pick)]
    body = []
    for label, vals, _n in stats:
        cells = []
        for c, (v, nd) in enumerate(zip(vals, (2, 3, 3, 3))):
            t = fmt(v, nd)
            cells.append(rf'\textbf{{{t}}}' if v is not None and v == best[c] else t)
        body.append(f'{label:<38} & ' + ' & '.join(cells) + r' \\')
    head = [r' & $\wvec$ err & $|r|$ read-out & $|r|$ in-loop & blow-up \\',
            r' & [$^\circ$/s] & & & \\', r'\midrule']
    return stats, tabular('@{}lcccc@{}', head, body)

stats_A11, table_A11 = table_regularisers_A11(rows_A11)
display(pd.DataFrame([[lab, n, *v] for lab, v, n in stats_A11],
                     columns=['term', 'n seq', 'err deg/s', '|r| read-out', '|r| in-loop', 'blow-up']))
print(table_A11)

### fig:regularisers -- `make_figures.py::fig_regularisers`

The read-out of each ablation run's final G (Poisson solve, fft), one row per
sequence, next to the APS frame; |r| under each panel. Redraw draws the
sequences whose data are here; Live draws the panel's segment from the Live runs.

In [ ]:
%%section A11
# step 3 -- fig_regularisers, unrolled
from interacting_maps.network_dissertation import solve_poisson_exact
from interacting_maps.viz import draw_map
from interacting_maps.metrics import abs_corr, crop_interior
from report_runs import aps_frame
from make_figures import REGULARISERS as FIG_REG, grid_size
cols_A11 = [('APS', None), ('none', {}),
            (r'$\delta_{\mathrm{shrink}I}{=}1$', {'delta_shrinkI': 1.0}),
            (r'$\delta_{\mathrm{map}}{=}0.03$', {'delta_map': 0.03}),
            (r'$\delta_{\mathrm{curl}}{=}0.6$', {'delta_curl': 0.6})]
if p['mode'] == 'Redraw':
    seqs_A11 = [s for s in SEGMENTS if s[3] in FIG_REG['sequences']]
    seqs_A11 = [s for s in seqs_A11 if need([os.path.join('data', s[0])], [])]   # aps + run_cfg need them
    root_A11, nf_A11 = FIG_REG['root'], int(round(FIG_REG['duration'] / 0.02))
else:
    seqs_A11 = [sweep_seg(p, SWEEP_JOBS['ablation'])]
    root_A11 = walk_dir('A11')
    nf_A11 = int(round(sweep_duration(p, SWEEP_JOBS['ablation']) / 0.02))
img_A11 = []
if seqs_A11:
    with use_style('paper'):
        fig_A11, axes = plt.subplots(len(seqs_A11), len(cols_A11),
                                     figsize=grid_size(len(seqs_A11), len(cols_A11), below=1, label_lines=1))
        axes = np.atleast_2d(axes)
        for row, (ds, sid, t0, lab) in enumerate(seqs_A11):
            aps = aps_frame(ds, t0 + nf_A11 * 0.02)
            ref = np.log(aps.astype(np.float64) + 1.0) if aps is not None else None
            for col, (title, deltas) in enumerate(cols_A11):
                a = axes[row, col]
                kind, img = 'aps', aps
                if col > 0:
                    kind, img = 'I', None
                    rc = run_cfg(ds, sid, t0, FIG_REG['model'], 0.02, nf_A11, out_root=root_A11,
                                 deltas=dict(deltas))
                    npz = os.path.join(rc.output_dir, 'maps_final.npz')
                    if os.path.exists(npz):
                        img = solve_poisson_exact(np.load(npz)['G'], 'fft')
                    else:
                        print(f'   (no run: {rc.output_dir})')
                if img is None:
                    a.set_axis_off()
                    continue
                draw_map(a, kind, img, title=title if row == 0 else '')
                if col == 0:
                    a.set_ylabel(lab)
                elif ref is not None and ref.shape == img.shape:
                    a.set_xlabel(f'$|r|={abs_corr(crop_interior(img), crop_interior(ref)):.2f}$')
        img_A11 = fig_images([fig_A11])
        style_save(fig_A11, os.path.join(walk_dir('A11'), 'fig_regularisers.png'), show=True)
else:
    print('fig_regularisers: no sequence has its data here')

In [ ]:
%%section A11
# step 4 -- text_numbers.regularisers, unrolled (Redraw only: it reads the report's runs)
def text_regularisers_A11(rows):
    L = []
    terms = (('none', {}), ('shrinkI=1', {'delta_shrinkI': 1.0}),
             ('map=0.01', {'delta_map': 0.01}), ('map=0.03', {'delta_map': 0.03}),
             ('curl=0.3', {'delta_curl': 0.3}), ('curl=0.6', {'delta_curl': 0.6}))
    reg = ('delta_shrinkI', 'delta_map', 'delta_curl')
    for s in SEGMENTS:
        mine = [r for r in rows if r['sequence'] == s[0]]
        for name, on in terms:
            hit = [r for r in mine if all(abs(float(r[k]) - on.get(k, 0.0)) < 1e-9 for k in reg)]
            if hit:
                r = hit[0]
                L.append(f'{s[3]:8s} {name:10s} err {float(r["err"]):6.2f}  |r| read-out '
                         f'{float(r["recon_mean"]):.3f}  in-loop {float(r["recon_inloop"]):.3f}  '
                         f'blow-up {float(r["blowup"]):.2f}  curl share {float(r["curl"]):.2f}')
    L.append('curl share of the final G, CMax-anchor, Table curl runs (none -> curl 0.6):')
    for s in SEGMENTS:                  # text_numbers.run_dir = A2's run_dir at 75 frames
        a, b = run_dir(s, 'thesis_cmax', 75, curl=0.0), run_dir(s, 'thesis_cmax', 75, curl=0.6)
        if a and b:
            L.append(f'  {s[3]:8s} {experiment_tracking.read_summary(a)["curl_share"]:.2f} -> '
                     f'{experiment_tracking.read_summary(b)["curl_share"]:.2f}')
    return L

lines_A11 = []
if p['mode'] == 'Redraw':
    # text_numbers reads experiments/ablation/sweep_*.csv (the table: *.csv)
    lines_A11 = text_regularisers_A11(sweep_rows('ablation', ('sweep_*.csv',)))
    print('== regularisers'); print('\n'.join(lines_A11))
else:
    print('Live: the text numbers read the report runs; switch to Redraw to see them.')

In [ ]:
%%section A11
# same-code check -- cmd_sweep (Live); table_regularisers, fig_regularisers, text (Redraw)
if p['mode'] == 'Live':
    check_live_sweep(p, 'A11', LIVE_JOB['A11'], rows_A11)
else:
    check('table_regularisers', nums(table_A11.splitlines()),
          nums(make_tables.table_regularisers().splitlines()))
    if seqs_A11:                        # the report's figure on the same sequences
        figs = report_figs(make_figures.fig_regularisers, 'A11',
                           REGULARISERS={**FIG_REG, 'sequences': tuple(s[3] for s in seqs_A11)})
        check(f'fig_regularisers images ({", ".join(s[3] for s in seqs_A11)})',
              img_A11, fig_images(figs))
    else:
        need([os.path.join('data', s[0]) for s in SEGMENTS if s[3] in FIG_REG['sequences']],
             ['(raw data: download_gdrive.sh)'])
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        text_numbers.regularisers()
        theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
    check('text_numbers.regularisers', nums(lines_A11), nums(theirs))

<a id="A12"></a>

## A12 · Settings for the best image · [↑ overview](#overview)

**Question.** Set for the image instead of for ω: which of
{thesis (no anchor), CMax-anchor} × Δt {20, 10} ms × δ_curl {0, 0.6} gives the
highest mean read-out |r|, separately for real and synthetic data, and what
does it cost in ω error? 1.5 s runs.

**Made by** `report/run_experiments.py::cmd_sweep`
(`--what sweep --name best_image --mode grid --vary 'dt=0.01;delta_curl=0,0.6'
--duration 1.5`), `run_all.sh` jobs `best_image_<seq>_<model>` (thesis,
thesis_cmax) → the 10 ms runs and `sweep_<model>_<seq>.csv` in
`experiments/best_image/`. The 20 ms points are not run here: they are the
Table III runs (A3, jobs `dissoc_<seq>_<model>`) and the curl runs (A4, jobs
`curl_<seq>_<model>`, which also write `main_curl_<seq>_<model>.csv`), in `results/`.

**Read by** `report/report_runs.py::image_search` (the search, from the run
folders, not the csvs), used by `report/make_tables.py::table_image`
(tab:image), `report/make_figures.py::fig_image_settings` (fig:image_settings;
its ω column is the Table III CMax-anchor run, A3) and
`report/text_numbers.py::image_window`.

In [ ]:
# step 1 -- the runs: the saved 10 ms rows (Redraw) or the panel's best_image points (Live)
p = params()
rows_A12 = run_sweep(p, 'A12', ['best_image_thesis', 'best_image_thesis_cmax'])
if READY['A12']:
    have_cols(rows_A12, SWEEP_COLS[:-1], 'A12')
if p['mode'] == 'Redraw' and READY['A12']:       # the 20 ms half: A3's and A4's runs
    need([os.path.join('report', 'experiments', f'{pre}_{s[3]}_{m}.csv') for s in SEGMENTS
          for m in ('thesis', 'thesis_cmax') for pre in ('dissoc_1p5s', 'main_curl')],
         ['dissoc_<seq>_<model>', 'curl_<seq>_<model>'])

In [ ]:
%%section A12
# step 2 -- report_runs.py::image_search, unrolled: mean (|r|, err) per kind and configuration
from report_runs import IMG_SEARCH, SYNTHETIC as RR_SYN, track_stats
def image_search_A12(duration, img_root):
    def rdir(ds, sid, t0, model, dt, curl):   # the folder of one search point (run_cfg's output_dir)
        return run_cfg(ds, sid, t0, model, dt, int(round(duration / dt)),
                       out_root=None if dt == 0.02 else img_root,
                       deltas={'delta_curl': curl} if curl else None).output_dir
    kinds = {k: [s for s in SEGMENTS if (s[3] in RR_SYN) == (k == 'synthetic')]
             for k in ('real', 'synthetic')}
    score, best = {}, {}
    for kind, seqs in kinds.items():
        for cfg in IMG_SEARCH:
            try:
                score[kind, cfg] = np.mean([track_stats(rdir(ds, sid, t0, *cfg))
                                            for ds, sid, t0, _ in seqs], axis=0)
            except OSError:
                print(f'    missing runs for {kind} {cfg}')
        # the report's max() raises when a kind has no point at all; here None
        best[kind] = max((c for c in IMG_SEARCH if (kind, c) in score),
                         key=lambda c: score[kind, c][0], default=None)
    return score, best, rdir

if p['mode'] == 'Redraw':
    score_A12, best_A12, rdir_A12 = image_search_A12(1.5, os.path.join('experiments', 'best_image'))
    display(pd.DataFrame([[kind, *cfg[:2], cfg[2], *sc] for (kind, cfg), sc in score_A12.items()],
                         columns=['kind', 'model', 'dt s', 'delta_curl', 'mean |r|', 'mean err deg/s']))
    print('best per kind (highest mean |r|):', best_A12)
else:
    print('Live: the 10 ms runs just made (the 20 ms half and the search need the report runs: Redraw)')
    display(pd.DataFrame([[os.path.basename(d), *track_stats(d)] for d in LIVE_RUNS['A12'].values()],
                         columns=['run', 'mean |r|', 'mean err deg/s']))

In [ ]:
%%section A12
# step 3 -- table_image, unrolled (Redraw)
from make_tables import tabular
table_A12 = ''
if p['mode'] == 'Redraw':
    body_A12 = []
    for model, title in (('thesis', r'none (\texttt{thesis})'), ('thesis_cmax', r'\texttt{CMax-anchor}')):
        if body_A12:
            body_A12.append(r'\midrule')
        for i, cfg in enumerate(c for c in IMG_SEARCH if c[0] == model):
            cells = []
            for kind in ('real', 'synthetic'):
                if (kind, cfg) not in score_A12:
                    cells.append('--- & ---')
                    continue
                r, err = score_A12[kind, cfg]
                rr = rf'\textbf{{{r:.2f}}}' if cfg == best_A12[kind] else f'{r:.2f}'
                cells.append(f'{rr} & {err:.1f}')
            body_A12.append(f'{title if i == 0 else "":<22} & {cfg[1] * 1000:.0f} & {cfg[2]:g} & '
                        + ' & '.join(cells) + r' \\')
    head_A12 = [r' & & & \multicolumn{2}{c}{real} & \multicolumn{2}{c}{synthetic} \\',
            r'\cmidrule(lr){4-5}\cmidrule(lr){6-7}',
            r'Anchor & $\Delta t$ & $\delta_{\text{curl}}$ & $|r|$ & err & $|r|$ & err \\', r'\midrule']
    table_A12 = tabular('@{}lccrrrr@{}', head_A12, body_A12)
    print(table_A12)
else:
    print('Live: tab:image needs the report runs; switch to Redraw.')

### fig:image_settings -- `make_figures.py::fig_image_settings`

Every sequence: the APS frame, the read-out at the ω configuration (the Table
III CMax-anchor run, A3) and at the image configuration of its kind (step 2).

In [ ]:
%%section A12
# step 4 -- fig_image_settings, unrolled (Redraw; it needs every sequence's data and runs)
from interacting_maps.network_dissertation import solve_poisson_exact
from interacting_maps.viz import draw_map
from report_runs import aps_frame
from make_figures import grid_size, PAPER_WIDTH, IMAGE_DURATION
img_A12 = None
if p['mode'] == 'Live':
    print('Live: fig:image_settings needs the report runs; switch to Redraw.')
elif need([os.path.join('data', s[0]) for s in SEGMENTS], ['(raw data: download_gdrive.sh)']) \
        and None not in best_A12.values():
    omega_cfg_A12 = ('thesis_cmax', 0.02, 0.0)            # tab:params
    with use_style('paper'):
        fig_A12, axes = plt.subplots(len(SEGMENTS), 3, figsize=grid_size(
            len(SEGMENTS), 3, width=PAPER_WIDTH, below=2, title_lines=2))
        for row, (ds, sid, t0, lab) in enumerate(SEGMENTS):
            kind = 'synthetic' if lab in RR_SYN else 'real'
            panels = [('APS', aps_frame(ds, t0 + IMAGE_DURATION), None, 'aps')]
            for ttl, cfg in (('configured\n' r'for $\omega$', omega_cfg_A12),
                             ('configured\nfor the image', best_A12[kind])):
                d = rdir_A12(ds, sid, t0, *cfg)
                img = solve_poisson_exact(np.load(os.path.join(d, 'maps_final.npz'))['G'], 'fft')
                panels.append((ttl, img, track_stats(d) + cfg, 'I'))
            for c, (ttl, img, st, kind_) in enumerate(panels):
                a = axes[row, c]
                if img is None:
                    a.set_axis_off()
                    continue
                draw_map(a, kind_, img, title=ttl if row == 0 else '')
                if c == 0:
                    a.set_ylabel(f'{lab}\n({kind})')
                if st is not None:
                    r, err = st[:2]
                    a.set_xlabel(f'$|r|={r:.2f}$\n$\\omega$ err ${err:.1f}$ °/s')
        img_A12 = fig_images([fig_A12])
        style_save(fig_A12, os.path.join(walk_dir('A12'), 'fig_image_settings.png'), show=True)

In [ ]:
%%section A12
# step 5 -- text_numbers.image_window, unrolled (Redraw): |r| at 20 -> 10 ms per sequence
def text_image_window_A12():
    L = []
    try:                                    # text_numbers.main reports a failed section so
        for ds, sid, t0, lab in SEGMENTS:
            parts = []
            for model in ('thesis', 'thesis_cmax'):
                for curl in (0.0, 0.6):
                    r20 = track_stats(rdir_A12(ds, sid, t0, model, 0.02, curl))[0]
                    r10 = track_stats(rdir_A12(ds, sid, t0, model, 0.01, curl))[0]
                    parts.append(f'{model} curl {curl:g}: {r20:.2f} -> {r10:.2f}')
            L.append(f'  {lab:8s} ' + ' | '.join(parts))
    except Exception as e:
        L.append(f'   (image_window failed: {type(e).__name__}: {e})')
    return L

lines_A12 = []
if p['mode'] == 'Redraw':
    lines_A12 = text_image_window_A12()
    print('== image_window'); print('\n'.join(lines_A12))
else:
    print('Live: the text numbers read the report runs; switch to Redraw to see them.')

In [ ]:
%%section A12
# same-code check -- cmd_sweep (Live); image_search, table_image, fig_image_settings, text (Redraw)
import report_runs
def report_call_A12(fn, *a):
    """A report function; a failure is returned as text_numbers.main would print it."""
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            return fn(*a), None
    except Exception as e:
        return None, f'{type(e).__name__}: {e}'

if p['mode'] == 'Live':
    check_live_sweep(p, 'A12', LIVE_JOB['A12'], rows_A12)
else:
    got, err = report_call_A12(report_runs.image_search, 1.5)
    if err:
        print(f'[DIFF] image_search: the report function failed: {err}')
    else:
        flat = lambda sc, b: {**{f'{k}|{c}|{j}': x for (k, c), v in sc.items() for j, x in enumerate(v)},
                              **{f'best {k}': IMG_SEARCH.index(c) for k, c in b.items() if c}}
        check('image_search', flat(score_A12, best_A12), flat(got[0], got[1]))
    got, err = report_call_A12(make_tables.table_image)
    if err:
        print(f'[DIFF] table_image: the report function failed: {err}')
    else:
        check('table_image', nums(table_A12.splitlines()), nums(got.splitlines()))
    if img_A12 is not None:
        check('fig_image_settings images', img_A12,
              fig_images(report_figs(make_figures.fig_image_settings, 'A12')))
    else:
        need([os.path.join('data', s[0]) for s in SEGMENTS], ['(fig_image_settings: raw data)'])
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        try:
            text_numbers.image_window()
            theirs = list(text_numbers.OUT)
        except Exception as e:
            theirs = text_numbers.OUT + [f'   (image_window failed: {type(e).__name__}: {e})']
    theirs = [l for l in theirs if l and not l.startswith(('==', '   (missing'))]
    check('text_numbers.image_window', nums(lines_A12), nums(theirs))

<a id="A13"></a>

## A13 · Maps on poster (every model stopped at frame 44) · [↑ overview](#overview)

**Question.** What do the maps (V, in-loop I, read-out I, G, F) look like at the
frame each sequence is shown at? poster's last Table III frame falls in the
anchored excursion, so it is shown at frame 43 (counted from 0), the frame of
highest |r|, from runs stopped there: 44 frames = 0.88 s, every model.

**Made by** `report/run_experiments.py::cmd_sweep`
(`--what sweep --name maps_poster --mode coord --duration 0.88 --sequences poster
--vary n_iters=75`), `run_all.sh` jobs `maps_poster_<model>` → runs (with
`maps_final.npz`) and `sweep_<model>.csv` in `experiments/maps_poster/`. The
other four rows of the figures are the last frames of the Table III runs (A3).

**Read by** `report/make_figures.py::compute_maps` → `::fig_maps` (fig:maps,
CMax-anchor) and `::fig_appendix` (fig:appendix_cook, _thesis, _imu,
_cmax_v2), and `report/text_numbers.py::polarity`. `compute_maps` runs no
network: it reads each run's saved `maps_final.npz` and bins the raw events of
the shown frame (V), so it is unrolled and also called here; the binning is
done once per row and shared by the models (the same call, same arguments).

In [ ]:
# step 1 -- the runs: the saved maps_poster rows and run folders (Redraw), or the panel's model (Live)
p = params()
rows_A13 = run_sweep(p, 'A13', [f'maps_poster_{m}' for m in MODELS])
POSTER_A13 = next(s for s in SEGMENTS if s[3] == 'poster')
if p['mode'] == 'Redraw':
    runs_A13 = {m: run_dir(POSTER_A13, m, 44, root=os.path.join('experiments', 'maps_poster'))
                for m in MODELS}
    for m, d in runs_A13.items():
        print(f'{m:16s} {d}')
    if READY['A13'] and runs_A13['thesis_cmax'] is None:
        READY['A13'] = need([os.path.join('experiments', 'maps_poster', 'poster_rotation', 'thesis_cmax')],
                            ['maps_poster_thesis_cmax'])
else:
    runs_A13 = {LIVE_JOB['A13'].split('maps_poster_')[1]: d for d in LIVE_RUNS.get('A13', {}).values()}

In [ ]:
%%section A13
# step 2 -- make_figures.py::compute_maps, unrolled: (V, I, read-out, G, F) per MAP_ROWS row
from interacting_maps.data_loader import EventFrameSequence
from interacting_maps.network_dissertation import solve_poisson_exact
from make_figures import MAP_ROWS, APPENDIX_MODELS, TUNED, _MAP_COLUMNS, grid_size
V_A13 = {}                                       # (events, t0, n) -> last binned frame

def last_V_A13(ds, t0, n):
    """compute_maps' V: the last of n event frames from t0 (cached: same for every model)."""
    ev = E.get_dataset_paths(ds)['events']
    if (ev, t0, n) not in V_A13:
        pp = E.get_dataset_paths(ds)
        sq = EventFrameSequence(pp['events'], pp['calib'], frame_duration=0.02, t_start=t0,
                                n_frames=n, clip_value=10.0, undistort=False, sensor_size=(180, 240))
        V_A13[ev, t0, n] = list(sq)[-1]
    return V_A13[ev, t0, n][0]

def compute_maps_A13(model, rows):
    d_anchor = TUNED['delta_IMU'] if model in ('thesis_imu', 'thesis_cmax') else None
    maps = []
    for i, _lab, n, root in rows:
        ds, sid, t0, label = SEGMENTS[i]
        d = make_figures.find_run(ds, sid, t0, model, TUNED['delta_FR'], d_anchor, n_frames=n, root=root)
        if d is None or not os.path.exists(os.path.join(d, 'maps_final.npz')):
            print(f'    (missing {ds}/{model})')
            maps.append(None)
            continue
        if not need([os.path.join('data', ds)]):  # the events of V (the report would stop here)
            maps.append(None)
            continue
        m = np.load(os.path.join(d, 'maps_final.npz'))
        maps.append((last_V_A13(ds, t0, n), m['I'], solve_poisson_exact(m['G'], 'fft'), m['G'], m['F']))
        print(f'    {label}/{model} ({os.path.basename(d)})')
    return maps

if p['mode'] == 'Redraw':
    rows_maps_A13 = MAP_ROWS
    models_A13 = ['thesis_cmax'] + APPENDIX_MODELS
else:                                            # Live: the poster row of the run just made
    nf = int(round(sweep_duration(p, SWEEP_JOBS[LIVE_JOB['A13']]) / 0.02))
    rows_maps_A13 = [(1, f'poster\n(Live, frame {nf - 1})', nf, walk_dir('A13'))]
    models_A13 = list(runs_A13)
maps_A13 = {m: compute_maps_A13(m, rows_maps_A13) for m in models_A13}
for m, mm in maps_A13.items():
    print(f'{m:16s} ' + ', '.join(f'{lab.splitlines()[0]} {"(none)" if x is None else x[0].shape}'
                                  for (_, lab, _, _), x in zip(rows_maps_A13, mm)))

### fig:maps, fig:appendix -- `make_figures.py::fig_maps`, `::fig_appendix` (`_draw_maps`)

One row per sequence, columns V, I in-loop, I read-out, G, F. fig_maps is
CMax-anchor; the appendix repeats the frames for the other four models.

In [ ]:
%%section A13
# step 3 -- _draw_maps, unrolled, for fig_maps (CMax-anchor) and the four appendix figures
from interacting_maps.viz import draw_map
def draw_maps_A13(maps, labels, name):
    labels = [lab for m, lab in zip(maps, labels) if m is not None]
    maps = [m for m in maps if m is not None]
    if not maps:
        print(f'  {name}: no runs found, skipped')
        return None
    with use_style('paper'):
        fig, axes = plt.subplots(len(maps), len(_MAP_COLUMNS),
                                 figsize=grid_size(len(maps), len(_MAP_COLUMNS)))
        axes = np.atleast_2d(axes)
        for r, (arrs, lab) in enumerate(zip(maps, labels)):
            for c, ((kind, ttl), arr) in enumerate(zip(_MAP_COLUMNS, arrs)):
                draw_map(axes[r, c], kind, arr, title=ttl if r == 0 else '')
            axes[r, 0].set_ylabel(lab)
        imgs = fig_images([fig])
        print(name)
        style_save(fig, os.path.join(walk_dir('A13'), f'{name}.png'), show=True)
    return imgs

imgs_A13 = {}
for m in maps_A13:
    name = 'fig_maps' if m == 'thesis_cmax' else f'fig_appendix_maps_{m}'
    imgs_A13[name] = draw_maps_A13(maps_A13[m], [r[1] for r in rows_maps_A13], name)

In [ ]:
%%section A13
# step 4 -- text_numbers.polarity, unrolled (Redraw): sign of the read-out, unanchored models
def text_polarity_A13():
    from report_runs import aps_frame, SYNTHETIC
    from interacting_maps.metrics import crop_interior
    L = []
    for s in [s for s in SEGMENTS if s[3] not in SYNTHETIC]:      # text_numbers.REAL
        ds, sid, t0, lab = s
        root, nf = ((os.path.join('experiments', 'maps_poster'), 44) if lab == 'poster'
                    else ('results', 75))
        for m in ('cook', 'thesis'):
            d = run_dir(s, m, nf, root=root)
            if d is None or not os.path.exists(os.path.join(d, 'maps_final.npz')):
                continue                         # text_numbers says '(missing maps ...)'
            try:
                aps = np.log(aps_frame(ds, t0 + (nf - 0.5) * 0.02).astype(np.float64) + 1.0)
            except Exception as e:               # no APS images in this checkout
                L.append(f'   (APS frame not available: {e})')
                return L
            img = solve_poisson_exact(np.load(os.path.join(d, 'maps_final.npz'))['G'], 'fft')
            a, b = crop_interior(img).ravel(), crop_interior(aps).ravel()
            a, b = a - a.mean(), b - b.mean()
            r = float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))
            with open(os.path.join(d, 'tracking.csv')) as f:
                last = list(csv.DictReader(f))[-1]
            e = np.array([float(last[f'est_w{k}']) for k in 'xyz'])
            g = np.array([float(last[f'gt_w{k}']) for k in 'xyz'])
            cos = float(e @ g / (np.linalg.norm(e) * np.linalg.norm(g)))
            L.append(f'{lab:8s} {m:7s} frame {int(last["frame"])}: signed r of the read-out {r:+.2f}, '
                     f'cosine between estimated and true omega {cos:+.2f}')
    return L

lines_A13 = []
if p['mode'] == 'Redraw':
    lines_A13 = text_polarity_A13()
    print('== polarity'); print('\n'.join(lines_A13))
else:
    print('Live: the text numbers read the report runs; switch to Redraw to see them.')

In [ ]:
%%section A13
# same-code check -- cmd_sweep (Live); stop at 44, compute_maps, fig_maps, fig_appendix, text (Redraw)
import interacting_maps.data_loader as data_loader_A13
def cached_frames_A13(events, calib, frame_duration=None, t_start=None, n_frames=None, **kw):
    """The binning step 2 already did, for compute_maps' identical call; any other call is real."""
    same = (frame_duration == 0.02 and kw == dict(clip_value=10.0, undistort=False, sensor_size=(180, 240)))
    if same and (events, t_start, n_frames) in V_A13:
        return [V_A13[events, t_start, n_frames]]
    return EventFrameSequence(events, calib, frame_duration=frame_duration, t_start=t_start,
                              n_frames=n_frames, **kw)

if p['mode'] == 'Live':
    check_live_sweep(p, 'A13', LIVE_JOB['A13'], rows_A13)
else:
    for m, d in runs_A13.items():       # deterministic: frame 43 is the Table III run's own frame 43
        d75 = run_dir(POSTER_A13, m, 75)
        if d and d75:
            check(f'maps_poster {m} = first 44 frames of its Table III run',
                  tracking(d)['err'], tracking(d75)['err'][:44])
    have = [r for r in MAP_ROWS if os.path.exists(os.path.join('data', SEGMENTS[r[0]][0]))]
    if len(have) < len(MAP_ROWS):       # the report functions run on the rows whose events are here
        need([os.path.join('data', SEGMENTS[r[0]][0]) for r in MAP_ROWS], ['(raw data: download_gdrive.sh)'])
    keep_rows = [k for k, r in enumerate(MAP_ROWS) if r in have]
    with redirect(data_loader_A13, EventFrameSequence=cached_frames_A13), \
         redirect(make_figures, MAP_ROWS=have):
        for m in maps_A13:
            with contextlib.redirect_stdout(io.StringIO()):
                theirs = make_figures.compute_maps(E, m)
            mine = [maps_A13[m][k] for k in keep_rows]
            flat = lambda ms: (np.concatenate([np.asarray(a, float).ravel() for x in ms if x is not None
                                               for a in x]) if any(x is not None for x in ms)
                               else np.array([]))
            check(f'compute_maps {m}', flat(mine), flat(theirs))
        for name, fn in (('fig_maps', make_figures.fig_maps), ('fig_appendix', make_figures.fig_appendix)):
            figs = report_figs(fn, 'A13')
            mine = [imgs_A13.get(n) for n in imgs_A13 if (n == 'fig_maps') == (name == 'fig_maps')]
            mine = np.concatenate([x for x in mine if x is not None]) if any(
                x is not None for x in mine) else np.array([])
            check(f'{name} images', mine, fig_images(figs))
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        text_numbers.polarity()
        theirs = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   (missing'))]
    check('text_numbers.polarity', nums(lines_A13), nums(theirs))

<a id="A14"></a>

## A14 · Convergence within a frame · [↑ overview](#overview)

**Question.** How does the intensity map I sharpen over the message-passing
iterations of one frame, model by model?

**Made by** `report/run_experiments.py::cmd_converge` (`--what converge`),
`run_all.sh` jobs `converge_dynamic` (`--sequences dynamic --frame 45`) and
`converge_bicycle` (`--sequences bicycle --frame 40`) → one npz per sequence at
`report_runs.converge_path(label, frame)` = `experiments/converge/<label>_frame<frame>.npz`,
holding I of every model at the checkpoints `1,3,10,25,50,75`.

**Read by** `report/make_figures.py::fig_converge` → `fig_converge_dynamic`
(fig:converge_dynamic) and `fig_converge_bicycle` (appendix).

**Steps** (those of `cmd_converge`, then `fig_converge`): the frame's events →
a network per model, warm-started like a run → `step(V, n_iters=1)` repeated,
I kept at every checkpoint → the grid of images → same-code check.

The form picks the cases (sequence, frame), dt and the iteration count. Redraw
reads the saved npz of each case (the report's need the synthetic `ecrot_bicycle`
and `dynamic` runs of the cluster); Live iterates now on the cases without one, into
`experiments/walkthrough/converge*/` (converge_path's file names).

In [ ]:
# image-section definitions (A14-A18): capture a report figure instead of saving
# it, and read what it shows -- the figure-data checks compare these
import io


def fig_pixels(fig):
    """Every image drawn on a figure, in axes order, as one flat array."""
    arrs = [np.asarray(im.get_array(), float).ravel() for ax in fig.axes for im in ax.get_images()]
    return np.concatenate(arrs) if arrs else np.array([])


def fig_text(fig):
    """Every title and axis label of a figure, in axes order."""
    return [s for ax in fig.axes for s in (ax.get_title(), ax.get_xlabel(), ax.get_ylabel()) if s]


@contextlib.contextmanager
def caught_figures(module, **attrs):
    """Run a report drawing function with its save() replaced by a collector and
    its output globals redirected (**attrs). Yields {'pixels': {file: array},
    'text': {file: [labels]}, 'out': its stdout}; the figures are closed."""
    got = {'pixels': {}, 'text': {}, 'out': io.StringIO()}
    def keep(fig, path, show=False):
        name = os.path.basename(path)
        got['pixels'][name], got['text'][name] = fig_pixels(fig), fig_text(fig)
        plt.close(fig)
    with redirect(module, save=keep, **attrs), contextlib.redirect_stdout(got['out']):
        yield got

In [ ]:
# the A14 form: which (sequence, frame) cases, at which dt and iteration count
from report_runs import converge_path
CASES_A14 = list(make_figures.CONVERGE) + [(s[3], 74) for s in SEGMENTS
                                           if (s[3], 74) not in make_figures.CONVERGE]
form('A14', [Knob('cases', list(make_figures.CONVERGE), 'many', choices=CASES_A14),
             Knob('dt', 0.02, choices=[0.02, 0.01]), Knob('n_iters', 75, choices=[75, 150])])

In [ ]:
# step 1 -- the snapshots per case: a saved npz (shown, not recomputed) or made now (Live)
p = params('A14')
rep_A14 = is_report('A14')                      # the report-only steps run only then
ckpts_A14 = sorted({c for c in (1, 3, 10, 25, 50, 75) if c <= p['n_iters']} | {p['n_iters']})
same_as_report = p['dt'] == 0.02 and p['n_iters'] == 75
# Live npz: converge_path's file name, in a folder per (dt, n_iters)
NAME_A14 = 'walkthrough/converge' + ('' if same_as_report else f"_dt{p['dt']*1000:g}ms_i{p['n_iters']}")
cases_A14, paths_A14, make_A14 = list(p['cases']), {}, []
for lab, fr in cases_A14:
    saved = ([converge_path(lab, fr)] if same_as_report else []) + [converge_path(lab, fr, NAME_A14)]
    hit = next((x for x in saved if os.path.exists(x)), None)
    if hit:
        paths_A14[lab, fr] = hit
        if p['mode'] == 'Live':
            print(f'>>> SAVED RUN (not rerun): {hit}')
    elif p['mode'] == 'Live':
        seg = next(s for s in SEGMENTS if s[3] == lab)
        if need([os.path.join('data', seg[0])]):
            paths_A14[lab, fr] = converge_path(lab, fr, NAME_A14)
            make_A14.append((lab, fr))
    else:
        print(f'missing: {saved[0]} -- no saved run for {lab} frame {fr}'
              + (f' (run_all.sh job converge_{lab})' if rep_A14 else ''))
READY['A14'] = bool(paths_A14)
for (lab, fr), path in paths_A14.items():
    print(f'{lab:8s} frame {fr}: {path}' + ('  (made now)' if (lab, fr) in make_A14 else ''))

In [ ]:
%%section A14
# step 2 -- run_experiments.py::cmd_converge, unrolled (Live), or its npz read (Redraw)
from interacting_maps.data_loader import (EventFrameSequence, CameraCalibration,
                                          load_events_fast, undistort_events)
from interacting_maps.angular_velocity import CMaxAngularVelocity
from interacting_maps.viz import net_maps


def converge_one_A14(seg, model, dt, frame, checkpoints):
    """cmd_converge for one (segment, model): {checkpoint: I}."""
    ds, sid, t0, lab = seg
    pth = E.get_dataset_paths(ds)
    calib = CameraCalibration(pth['calib'])
    nf = frame + 1
    rc = run_cfg(ds, sid, t0, model, dt, nf)
    seq = EventFrameSequence(pth['events'], pth['calib'], frame_duration=dt, t_start=t0,
                             n_frames=nf, clip_value=10.0,
                             undistort=rc.undistort_at_event_level, sensor_size=(180, 240))
    H, W = seq.H, seq.W
    V, _tm = list(seq)[frame]                   # the event frame iterated on
    net = E.make_network(rc, H, W, calib.fx, calib.fy, calib.cx, calib.cy)
    if hasattr(net, 'initialize_from_rotation'):  # warm start from the gyro, as a run
        net.initialize_from_rotation(rc.initial_R)
    else:
        net.R = rc.initial_R.copy()
    lo, hi = t0 + frame * dt, t0 + (frame + 1) * dt
    kw = {}                                     # the anchor target of this frame
    if getattr(rc, 'use_cmax', False) or getattr(rc, 'use_imu', False):
        cev = undistort_events(load_events_fast(pth['events'], t_start=lo, duration=dt + 0.05), calib)
        win = cev[(cev[:, 0] >= lo) & (cev[:, 0] < hi)]
        kw['events'] = win
        if getattr(rc, 'use_cmax', False) and len(win) > 10:
            est = CMaxAngularVelocity(H, W, calib.fx, calib.fy, calib.cx, calib.cy, use_polarity=True)
            kw['omega_imu'] = est.estimate(win, t_ref=0.5*(lo+hi), omega_init=np.zeros(3))
        elif getattr(rc, 'use_imu', False):
            kw['omega_imu'] = E.get_gyro_for_frame(E.load_imu(pth['imu']), lo, hi)
    snaps = {}
    for it in range(1, max(checkpoints) + 1):   # the network's own step(), one iteration
        try:
            net.step(V, n_iters=1, **kw)
        except TypeError:                       # cook's step takes no anchor
            net.step(V, n_iters=1)
        if it in checkpoints:
            snaps[it] = net_maps(net, H, W)['I'].copy()
    return snaps


snaps_A14 = {}                                  # {(label, frame): {model: {checkpoint: I}}}
for (lab, fr), path in paths_A14.items():
    if (lab, fr) in make_A14:                   # cmd_converge, unrolled: made now
        seg = next(s for s in SEGMENTS if s[3] == lab)
        snaps_A14[lab, fr] = {m: converge_one_A14(seg, m, p['dt'], fr, ckpts_A14) for m in MODELS}
        os.makedirs(os.path.dirname(path), exist_ok=True)
        np.savez_compressed(path, checkpoints=ckpts_A14, segment=seg[1],
                            **{f'{m}_{it}': snaps_A14[lab, fr][m][it] for m in MODELS for it in ckpts_A14})
        print(f'-> {path}')
    else:                                       # its npz read
        z = np.load(path)
        ck = [int(c) for c in z['checkpoints']]
        snaps_A14[lab, fr] = {m: {it: z[f'{m}_{it}'] for it in ck} for m in MODELS}
# how I sharpens: its spatial std at every checkpoint, per model
for (lab, fr), by_model in snaps_A14.items():
    print(f'== {lab} frame {fr}: std of I at each checkpoint (rows: models, columns: iterations)')
    display(pd.DataFrame({m: {it: float(np.std(I)) for it, I in s.items()}
                          for m, s in by_model.items()}).T.round(4))

In [ ]:
%%section A14
# step 3 -- make_figures.py::fig_converge, unrolled: rows models, columns checkpoints
from interacting_maps.style import use_style, save
from interacting_maps.viz import draw_map
pix_A14 = {}
for (lab, fr), by_model in snaps_A14.items():
    one_per_lab = len({l for l, _ in snaps_A14}) == len(snaps_A14)
    name = f'fig_converge_{lab}' + ('' if one_per_lab else f'_frame{fr}')   # fig_converge's names
    ck = sorted(next(iter(by_model.values())))
    with use_style('paper'):
        fig_A14, axes_A14 = plt.subplots(len(MODELS), len(ck),
                                 figsize=make_figures.grid_size(len(MODELS), len(ck), label_lines=1))
        axes_A14 = np.atleast_2d(axes_A14)
        for r, model in enumerate(MODELS):
            for c, it in enumerate(ck):
                draw_map(axes_A14[r, c], 'I', by_model[model][it], title=f'iter {it}' if r == 0 else '')
            axes_A14[r, 0].set_ylabel(make_figures.MODELS[model][0])
        pix_A14[f'{name}.png'] = fig_pixels(fig_A14)
        save(fig_A14, os.path.join(walk_dir('A14'), f'{name}.png'), show=True)

In [ ]:
%%section A14
# same-code check -- cmd_converge on the cases made now; fig_converge itself on
# the report's npz (report configuration only)
import argparse
for lab, frame in make_A14:
    chk_A14 = os.path.join(walk_dir('A14'), 'check', f'{lab}_frame{frame}.npz')
    os.makedirs(os.path.dirname(chk_A14), exist_ok=True)
    args_A14 = argparse.Namespace(checkpoints=','.join(map(str, ckpts_A14)), sequences=lab,
                                  frame=frame, dt=p['dt'])
    with redirect(run_experiments, converge_path=lambda l, f, name='converge': chk_A14), \
         contextlib.redirect_stdout(io.StringIO()):
        run_experiments.cmd_converge(E, args_A14)
    z = np.load(chk_A14)
    check(f'cmd_converge snapshots {lab} frame {frame}',
          np.concatenate([z[f'{m}_{it}'].ravel() for m in MODELS for it in ckpts_A14]),
          np.concatenate([snaps_A14[lab, frame][m][it].ravel() for m in MODELS for it in ckpts_A14]))
# fig_converge on the form's own npz: its CONVERGE and converge_path pointed at them
# (the report's own when the form is the report's)
if len({l for l, _ in paths_A14}) < len(paths_A14):
    print('[skip] fig_converge: one figure per sequence, and two cases share one')
else:
    report_npz = rep_A14 and all(paths_A14[c] == converge_path(*c) for c in paths_A14)
    attrs_A14 = {} if report_npz else dict(
        CONVERGE=list(paths_A14), converge_path=lambda l, f, name='converge': paths_A14[l, f])
    with caught_figures(make_figures, FIG=os.path.join(walk_dir('A14'), 'check'), **attrs_A14) as got_A14:
        make_figures.fig_converge(E)
    for name, mine in pix_A14.items():
        if name in got_A14['pixels']:
            check(f'fig_converge {name}', mine, got_A14['pixels'][name])
        else:
            print(f'[DIFF] fig_converge {name}: the report function drew no such figure')

<a id="A15"></a>

## A15 · Reconstruction: 20 against 10 ms, iterative against exact · [↑ overview](#overview)

**Question.** What does the intensity map depend on? Halving the accumulation
window cleans up G; solving the Poisson equation exactly recovers the low
frequencies the iterative update leaves out.

**Made by** `report/run_experiments.py::cmd_recon` (`--what recon`), `run_all.sh`
job `recon` (`--sequences boxes,poster --duration 0.8`) → for each sequence two
CMax-anchor runs, `report_runs.recon_cfg` at dt = 20 ms (40 frames) and 10 ms
(80 frames), iterative update, adopted step sizes, in `experiments/recon_figure/`.

**Read by** `report/make_figures.py::fig_recon` → fig:recon (APS, iterative at 20
and 10 ms, the exact fft read-out at 10 ms; |r| under each panel).

**Steps**: the two runs per sequence (found or made) → their final maps →
the |r| of each panel against log(1+APS) → the figure → same-code check.

**Flag.** The local `recon_figure` runs predate the cluster's csvs (stage-1
finding): their numbers may differ from what the cluster's runs give now.
Step 2 prints each run's commit and date. The same-code check compares the
figure against `fig_recon` drawn from the *same* runs, so it is unaffected.

In [ ]:
# the A15 form: which sequences, which model, how long
from report_runs import recon_cfg
RECON_A15 = make_figures.RECON                  # the report's sequences, duration, model, root
form('A15', [Knob('segments', list(RECON_A15['sequences']), 'many', choices=[s[3] for s in SEGMENTS]),
             Knob('model', RECON_A15['model'], choices=MODELS[1:]),
             Knob('duration', RECON_A15['duration'], choices=[0.06, 0.2, 0.8, 1.5])])

In [ ]:
# step 1 -- the runs, recon_cfg at 20 and 10 ms: found (Redraw) or made now (Live;
# a saved run is shown, not rerun). Live: choose a short duration here (0.06 s).
p = params('A15')
rep_A15 = is_report('A15')                      # the report-only steps run only then
seqs_A15 = [s for s in SEGMENTS if s[3] in p['segments']]
model_A15, dur_A15 = p['model'], p['duration']
runs_A15, cfgs_A15 = {}, {}                     # {label: {dt: run folder}}, {label: {dt: RunConfig}}
for ds, sid, t0, lab in seqs_A15:
    if not need([os.path.join('data', ds)]):    # recon_cfg reads the dataset's gyro
        continue
    cfgs_A15[lab] = {dt: recon_cfg(ds, sid, t0, model_A15, dt, dur_A15, 'experiments/walkthrough')
                     for dt in (0.02, 0.01)}
    got = {}
    for dt, rc in cfgs_A15[lab].items():
        if p['mode'] == 'Live':
            got[dt] = saved_or_run(rc)
        else:                                   # the report's root first
            got[dt] = report_or_saved(rc, RECON_A15['root'])
            if got[dt] is None:
                print(f'no saved run for {lab} {dt*1000:.0f} ms' + (' (run_all.sh job recon)' if rep_A15 else ''))
    if all(got.values()):
        runs_A15[lab] = got
READY['A15'] = bool(runs_A15)
for lab, d in runs_A15.items():
    for dt, f in d.items():
        print(f'{lab:8s} {dt*1000:.0f} ms  {f}')

In [ ]:
%%section A15
# step 2 -- what each run is, and which code made it (the local runs may be stale)
info_A15 = []
for lab, d in runs_A15.items():
    for dt, f in d.items():
        r = experiment_tracking.read_run(f)
        c, s, pv = r['config'], r['summary'] or {}, r.get('provenance', {})
        info_A15.append([lab, f'{dt*1000:.0f} ms', c['n_frames'], c['poisson'],
                     round(s.get('mean_err_deg_s', np.nan), 2), round(s.get('recon_r_mean', np.nan), 3),
                     round(s.get('recon_r_mean_inloop', np.nan), 3),
                     str(pv.get('git_commit'))[:7], str(pv.get('timestamp'))[:10]])
display(pd.DataFrame(info_A15, columns=['sequence', 'dt', 'n_frames', 'poisson', 'err deg/s',
                                    '|r| read-out', '|r| in-loop', 'commit', 'date']))

In [ ]:
%%section A15
# step 3 -- the four panels of each row, and |r| = abs_corr on crop_interior against log(1+APS)
from report_runs import aps_frame
from interacting_maps.metrics import abs_corr, crop_interior
from interacting_maps.network_dissertation import solve_poisson_exact
panels_A15, r_A15 = {}, []
for ds, sid, t0, lab in seqs_A15:
    if lab not in runs_A15:
        continue
    got_A15 = {dt: np.load(os.path.join(f, 'maps_final.npz')) for dt, f in runs_A15[lab].items()}
    aps = aps_frame(ds, t0 + dur_A15)           # the APS frame at the end of the track
    ref = np.log(aps.astype(np.float64) + 1.0) if aps is not None else None
    panels_A15[lab] = [('APS', aps, 'aps'),
                       (r'iterative, $\Delta t = 20$ ms', got_A15[0.02]['I'], 'I'),
                       (r'iterative, $\Delta t = 10$ ms', got_A15[0.01]['I'], 'I'),
                       (r'exact read-out, $\Delta t = 10$ ms', solve_poisson_exact(got_A15[0.01]['G'], 'fft'), 'I')]
    r_A15.append([lab] + [abs_corr(crop_interior(img), crop_interior(ref)) if ref is not None else np.nan
                          for _t, img, _k in panels_A15[lab][1:]])
    if aps is None:
        print(f'{lab}: no APS frame in data/{ds} -- no APS panel and no |r|')
display(pd.DataFrame(r_A15, columns=['sequence', 'iterative 20 ms', 'iterative 10 ms',
                                     'exact fft 10 ms']).round(3))

In [ ]:
%%section A15
# step 4 -- make_figures.py::fig_recon, unrolled
from interacting_maps.style import use_style, save
from interacting_maps.viz import draw_map
with use_style('paper'):
    fig_A15, axes_A15 = plt.subplots(len(panels_A15), 4,
                                 figsize=make_figures.grid_size(len(panels_A15), 4, below=1, label_lines=1))
    axes_A15 = np.atleast_2d(axes_A15)
    for row, (lab, panels) in enumerate(panels_A15.items()):
        aps = panels[0][1]
        ref = np.log(aps.astype(np.float64) + 1.0) if aps is not None else None
        for col, (ttl, img, kind) in enumerate(panels):
            a = axes_A15[row, col]
            if img is None:
                a.set_axis_off()
                continue
            draw_map(a, kind, img, title=ttl if row == 0 else '')
            if col == 0:
                a.set_ylabel(lab)
            if col > 0 and ref is not None:
                a.set_xlabel(f'$|r|={abs_corr(crop_interior(img), crop_interior(ref)):.2f}$')
    pix_A15, text_A15 = fig_pixels(fig_A15), fig_text(fig_A15)
    save(fig_A15, os.path.join(walk_dir('A15'), 'fig_recon.png'), show=True)

In [ ]:
%%section A15
# same-code check -- in Live, cmd_recon's RunConfigs; fig_recon itself on the
# report's runs (report configuration only: its RECON is the report's)
import argparse
if p['mode'] == 'Live':
    for lab in runs_A15:
        seen_A15 = []
        args_A15 = argparse.Namespace(name='walkthrough/A15', sequences=lab, model=model_A15,
                                      duration=dur_A15, no_frames=True, frame_stride=1)
        with redirect(run_experiments, _run=lambda E_, rc, sf, st: seen_A15.append(rc)), \
             contextlib.redirect_stdout(io.StringIO()):
            run_experiments.cmd_recon(E, args_A15)
        strip = lambda rc: {k: v for k, v in rc.to_dict().items() if k != 'out_root'}
        mine_A15 = [strip(rc) for rc in cfgs_A15[lab].values()]
        theirs_A15 = [strip(rc) for rc in seen_A15]
        print(f'[ok] config {lab}' if mine_A15 == theirs_A15 else f'[DIFF] config {lab}')
# fig_recon on the form's own runs: its RECON pointed at them (sequences, duration,
# model, and the root they lie in -- recon_cfg names the folders the same way)
roots_A15 = {os.path.dirname(os.path.dirname(os.path.dirname(os.path.normpath(f))))
             for d in runs_A15.values() for f in d.values()}
if len(roots_A15) != 1:
    print('[skip] fig_recon: the runs lie in several roots' if roots_A15 else '[skip] fig_recon: no runs')
else:
    labs_A15 = tuple(runs_A15)
    if len(labs_A15) < len(seqs_A15):
        print(f'fig_recon is checked on {labs_A15} only (the other rows are missing above)')
    attrs_A15 = dict(RECON=dict(RECON_A15, sequences=labs_A15, duration=dur_A15, model=model_A15,
                                root=roots_A15.pop()))
    with caught_figures(make_figures, FIG=os.path.join(walk_dir('A15'), 'check'), **attrs_A15) as got_A15:
        make_figures.fig_recon(E)
    check('fig_recon pixels', pix_A15, got_A15['pixels'].get('fig_recon.png', []))
    check('fig_recon |r|', nums(text_A15), nums(got_A15['text'].get('fig_recon.png', [])))

<a id="A16"></a>

## A16 · Read-out: fft against dct · [↑ overview](#overview)

**Question.** Which exact Poisson solver gives the better image when applied
once, as a read-out, to the G a run produced: fft (periodic border) or dct
(mirrored border)? And does halving the window to 10 ms help every read-out?

**Made by** `report/run_experiments.py::cmd_readout` (`--what readout`),
`run_all.sh` jobs `readout_20ms` (`--sequences boxes,poster,bicycle,street`) and
`readout_10ms` (same, `--dt 0.01 --name readout_dt10ms`) → CMax-anchor runs
(iterative update, adopted step sizes) at track lengths 0.4, 0.8, 1.2 s, and one
csv each: `experiments/readout_fft_vs_dct/readout.csv`,
`experiments/readout_dt10ms/readout.csv`.

**Scoring** (paired, same run, same G): |Pearson r| (`metrics.abs_corr`) of
each read-out against log(1+APS), on the full frame and on `crop_interior`
(10 % border cut). Correlation is blind to the scale and offset the
reconstruction is free in.

**Read by** `report/make_tables.py::table_recon` (tab:recon) and the second
heading of `report/text_numbers.py::intensity`, *"Intensity: 10 against 20 ms
on the same read-outs"* -- that is where the read-out and 10-vs-20 ms lines are
printed (`intensity_update` prints the in-loop poisson job, A17).

In [ ]:
# the A16 form: segment, model, window and the track lengths read out
from report_runs import ADOPTED
form('A16', [Knob('segment', 'poster', choices=[s[3] for s in SEGMENTS]),
             Knob('model', 'thesis_cmax', choices=MODELS[1:]),
             Knob('dt', 0.02, choices=[0.02, 0.01]),
             Knob('durations', [0.4, 0.8, 1.2], 'many', choices=[0.06, 0.4, 0.8, 1.2])])

In [ ]:
# step 1 -- the readout runs (cmd_readout's run_cfg) per track length: found (Redraw)
# or made now (Live; a saved run is shown, not rerun). Live: pick 0.06 s here.
p = params('A16')
rep_A16 = is_report('A16')                      # the report-only steps run only then
CSV_A16 = {20: os.path.join('experiments', 'readout_fft_vs_dct', 'readout.csv'),
           10: os.path.join('experiments', 'readout_dt10ms', 'readout.csv')}
ROOT_A16 = os.path.dirname(CSV_A16[20 if p['dt'] == 0.02 else 10])   # the report job's runs
ds_A16, sid_A16, t0_A16, lab_A16 = p['seg']
# the report-only steps read both csvs of the report jobs
csv_A16 = rep_A16 and need(list(CSV_A16.values()), ['readout_20ms', 'readout_10ms'])
model_A16, dt_A16 = p['model'], p['dt']
runs_A16, cfgs_A16 = {}, {}                     # {duration: run folder}, {duration: RunConfig}
if need([os.path.join('data', ds_A16)]):        # run_cfg reads the dataset's gyro
    for dur in p['durations']:
        nf = int(round(dur / dt_A16))
        rc = run_cfg(ds_A16, sid_A16, t0_A16, model_A16, dt_A16, nf, out_root='experiments/walkthrough',
                     poisson='iterative', deltas=dict(ADOPTED))
        cfgs_A16[dur] = rc
        if p['mode'] == 'Live':
            runs_A16[dur] = saved_or_run(rc)
            continue
        d = report_or_saved(rc, ROOT_A16)
        if d:
            runs_A16[dur] = d
        else:
            print(f'no saved run for {lab_A16} {dur} s at {dt_A16*1000:.0f} ms'
                  + (' (run_all.sh jobs readout_20ms, readout_10ms)' if rep_A16 else ''))
READY['A16'] = bool(runs_A16)
for dur, f in runs_A16.items():
    print(f'{lab_A16} {dur} s: {f}')

In [ ]:
%%section A16
# step 2 -- run_experiments.py::cmd_readout's scoring, unrolled on the panel's segment
from report_runs import aps_frame
from interacting_maps.metrics import abs_corr, crop_interior
from interacting_maps.network_dissertation import solve_poisson_exact
READOUT_COLS_A16 = ['sequence', 'duration_s', 'n_frames', 'variant', 'r_full', 'r_crop',
                    'low_freq', 'contrast']


def score_A16(ds, t0, dur, nf, dt, folder):
    """cmd_readout's rows for one run: the three read-outs of its final G."""
    m = np.load(os.path.join(folder, 'maps_final.npz'))
    aps = aps_frame(ds, t0 + nf * dt)
    if aps is None:
        print(f'  {ds}: no APS, skipped')
        return []
    ref = np.log(aps.astype(np.float64) + 1.0)
    variants = {'iterative': m['I'],
                'fft': solve_poisson_exact(m['G'], 'fft'),
                'dct': solve_poisson_exact(m['G'], 'dct')}
    rows = []
    for name, I in variants.items():
        st = E._intensity_stats(I)
        rows.append([ds, dur, nf, name, round(abs_corr(I, ref), 4),
                     round(abs_corr(crop_interior(I), crop_interior(ref)), 4),
                     round(st['low_freq_share'], 4), round(st['contrast'], 4)])
    return rows


rows_A16 = []
for dur, f in runs_A16.items():                 # the form's runs, one block of rows each
    rows_A16 += score_A16(ds_A16, t0_A16, dur, int(round(dur / dt_A16)), dt_A16, f)
display(pd.DataFrame(rows_A16, columns=READOUT_COLS_A16))
if rows_A16:
    wins_A16 = {}
    for r in rows_A16:
        wins_A16.setdefault(r[1], {})[r[3]] = r[5]
    print(f"paired wins on r_crop: fft {sum(w['fft'] > w['dct'] for w in wins_A16.values())}, "
          f"dct {sum(w['fft'] <= w['dct'] for w in wins_A16.values())}")

In [ ]:
%%section A16
# step 3 -- make_tables.py::table_recon, unrolled: mean |r_crop| per read-out, fft-over-dct wins
from make_tables import tabular
if csv_A16:
    body_A16, means_A16 = [], {}
    for dt, path in CSV_A16.items():
        r, runs = {}, {}
        for row in csv.DictReader(open(path)):
            v = abs(float(row['r_crop']))
            r.setdefault(row['variant'], []).append(v)
            runs.setdefault((row['sequence'], row['duration_s']), {})[row['variant']] = v
        mean = {k: float(np.mean(v)) for k, v in r.items()}
        best = max(mean.values())
        cell = lambda k: (rf'$\mathbf{{{mean[k]:.2f}}}$' if dt == 10 and mean[k] == best
                          else f'${mean[k]:.2f}$')
        wins_A16 = sum(v['fft'] > v['dct'] for v in runs.values())
        body_A16.append(f'${dt}$\\,ms & {cell("iterative")} & {cell("fft")} & {cell("dct")} & '
                    f'${wins_A16}/{len(runs)}$ \\\\')
        means_A16[f'{dt} ms'] = {**{k: round(v, 3) for k, v in mean.items()},
                                 'fft > dct': f'{wins_A16}/{len(runs)}'}
    head__A16 = [r'$\Delta t$ & iterative~\eqref{eq:cook_poisson}',
             r'           & exact, FFT & exact, DCT & FFT $>$ DCT \\', r'\midrule']
    table_A16 = tabular('@{}lcccc@{}', head__A16, body_A16)
    display(pd.DataFrame(means_A16).T)
    print(table_A16)
else:
    print('[skip] report-only step (table_recon reads both readout csvs of the report); '
          'the mean |r_crop| of the form runs:')
    print({v: np.mean([r[5] for r in rows_A16 if r[3] == v]).round(3)
           for v in ('iterative', 'fft', 'dct')} if rows_A16 else '(no rows)')

In [ ]:
%%section A16
# step 4 -- text_numbers.py::intensity, its heading "10 against 20 ms on the same read-outs"
lines_A16 = []
if csv_A16:
    rd = text_numbers.rows_of
    r20, r10 = rd(CSV_A16[20]), rd(CSV_A16[10])
    key = lambda r: (r['sequence'], r['duration_s'], r['variant'])
    a = {key(r): abs(float(r['r_crop'])) for r in r20}
    b = {key(r): abs(float(r['r_crop'])) for r in r10}
    for v in ('iterative', 'fft', 'dct'):
        ks = [k for k in a if k[2] == v and k in b]
        if ks:
            lines_A16.append(f'{v:9s}: 10 ms better on {sum(b[k] > a[k] for k in ks)}/{len(ks)} runs; '
                             f'mean {np.mean([a[k] for k in ks]):.2f} -> {np.mean([b[k] for k in ks]):.2f}')
    for name, tab in (('20 ms', a), ('10 ms', b)):
        pairs = {(k[0], k[1]) for k in tab}
        for x, y in (('fft', 'iterative'), ('dct', 'iterative'), ('fft', 'dct')):
            n = [q for q in pairs if (q + (x,)) in tab and (q + (y,)) in tab]
            if n:
                lines_A16.append(f'{name}: {x} beats {y} on '
                                 f'{sum(tab[q + (x,)] > tab[q + (y,)] for q in n)}/{len(n)}')
    print('\n'.join(lines_A16) or '(no rows)')
else:
    print('[skip] report-only step')

In [ ]:
%%section A16
# same-code check -- cmd_readout's RunConfigs (Live); rows vs the csv, table vs
# table_recon, text vs text_numbers (report configuration only)
import io, argparse
if p['mode'] == 'Live':
    seen_A16 = []
    args_A16 = argparse.Namespace(name='walkthrough', sequences=lab_A16,   # the Live runs' root
                                  durations=','.join(str(d) for d in p['durations']),
                                  dt=dt_A16, model=model_A16, no_frames=True, frame_stride=1)
    with redirect(run_experiments, _run=lambda E_, rc_, sf, st: seen_A16.append(rc_)), \
         contextlib.redirect_stdout(io.StringIO()):
        try:
            run_experiments.cmd_readout(E, args_A16)
        except Exception as e:                  # it scores runs the recorder did not make
            print(f'   (cmd_readout stopped after its configs: {type(e).__name__})')
    strip = lambda rc: {k: v for k, v in rc.to_dict().items() if k != 'out_root'}
    same_A16 = [strip(r) for r in seen_A16] == [strip(cfgs_A16[d]) for d in p['durations']]
    print('[ok] config' if same_A16 else '[DIFF] config')
    out_csv_A16 = os.path.join('experiments', 'walkthrough', 'readout.csv')   # cmd_readout's rows
    if same_A16 and os.path.exists(out_csv_A16):
        theirs_A16 = list(csv.DictReader(open(out_csv_A16)))
        check('cmd_readout rows', [x for r in rows_A16 for x in r[4:]],
              [float(r[c]) for r in theirs_A16 for c in READOUT_COLS_A16[4:]])
    else:
        print('[skip] cmd_readout rows: it scores only runs under experiments/walkthrough')
from_report = all(os.path.normpath(f).startswith(os.path.normpath(ROOT_A16)) for f in runs_A16.values())
if not rep_A16 or not from_report:
    print('[skip] not the report configuration')
elif not csv_A16:
    print('[skip] the report csvs are missing (above)')
else:
    want_A16 = {(r['sequence'], float(r['duration_s']), r['variant']): r
                for r in csv.DictReader(open(CSV_A16[20]))}
    mine_A16 = {(r[0], float(r[1]), r[3]): r for r in rows_A16}
    if mine_A16:
        check('readout.csv rows', [x for k in sorted(mine_A16) for x in mine_A16[k][4:]],
              [float(want_A16[k][c]) if k in want_A16 else np.nan
               for k in sorted(mine_A16) for c in READOUT_COLS_A16[4:]])
    check('table_recon', nums(table_A16.splitlines()), nums(make_tables.table_recon().splitlines()))
    try:
        with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
            text_numbers.intensity()
            out_A16 = list(text_numbers.OUT)
        i = next(k for k, l in enumerate(out_A16) if l.startswith('== Intensity: 10 against 20 ms'))
        theirs_A16 = [l for l in out_A16[i + 1:] if l and not l.startswith('   (missing')]
        check('text_numbers.intensity 10 vs 20 ms', nums(lines_A16), nums(theirs_A16))
    except Exception as e:
        print(f'[DIFF] text_numbers.intensity 10 vs 20 ms: {type(e).__name__}: {e}')

<a id="A17"></a>

## A17 · In-loop Poisson · [↑ overview](#overview)

**Question.** Where should the exact Poisson solve be applied: inside the loop,
on every iteration (`poisson=fft/dct`), or once at the end as a read-out of the
G the iterative network produced? In the loop, the exact solve's 1/|k|² gain
integrates every low-frequency error of G into blobs that feed back into G.

**Made by** `report/run_experiments.py::cmd_poisson` (`--what poisson`),
`run_all.sh` job `poisson` (its defaults: `--sequences poster,bicycle`,
3 s of 20 ms, CMax-anchor, adopted step sizes) → one run per in-loop mode
(iterative, fft, dct) in `experiments/poisson_solvers/`, and a five-panel
figure per sequence `experiments/poisson_solvers/poisson_<label>.png` (APS, the
three in-loop I, the fft and dct read-outs of the iterative run's G).

**Read by** `report/text_numbers.py::intensity_update` (text only, no figure or
table in the report).

**Steps**: the three runs (found or made) → their summaries → the five panels
and their low-frequency share and contrast (`_intensity_stats`) → the text
numbers → same-code check.

In [ ]:
# the A17 form: which sequences, which in-loop modes, which model, how long
MODES_A17 = ('iterative', 'fft', 'dct')
form('A17', [Knob('segments', ['poster', 'bicycle'], 'many', choices=[s[3] for s in SEGMENTS]),
             Knob('modes', list(MODES_A17), 'many', choices=list(MODES_A17)),
             Knob('model', 'thesis_cmax', choices=MODELS[1:]), Knob('n_frames', 150)])

In [ ]:
# step 1 -- the runs: found (Redraw, as intensity_update finds them: by stored config,
# the poisson job's folder first) or made now (Live; a saved run is shown, not rerun)
from report_runs import ADOPTED
p = params('A17')
rep_A17 = is_report('A17')                      # the report-only steps run only then
ROOT_A17 = os.path.join('experiments', 'poisson_solvers')
model_A17, nf_A17 = p['model'], p['n_frames']
runs_A17 = {}                                   # {segment: {mode: run folder}}
for s in [s for s in SEGMENTS if s[3] in p['segments']]:
    ds, sid, t0, lab = s
    for mode in [m for m in MODES_A17 if m in p['modes']]:
        d = None
        if p['mode'] == 'Redraw':
            # the poisson job's folder as intensity_update reads it (the last match) ...
            for f in sorted(glob.glob(os.path.join(ROOT_A17, ds, model_A17, f'{sid}_*'))):
                try:
                    c = experiment_tracking.read_config(f)
                except (OSError, ValueError):
                    continue
                if c.get('n_frames') == nf_A17 and c.get('poisson', 'iterative') == mode:
                    d = f
            if d is None:                       # ... else cmd_poisson's config in any saved root
                need([os.path.join(ROOT_A17, ds, model_A17)], ['poisson'] if rep_A17 else [])
                prm = {**make_tables.expected_params(model_A17), **ADOPTED}   # every step size
                with contextlib.redirect_stdout(io.StringIO()):
                    d = next((h for h in (experiment_tracking.find_run(
                             r, ds, model_A17, sid, nf_A17, prm, t_start=t0, poisson=mode)
                             for r in roots()) if h), None)
        elif need([os.path.join('data', ds)]):  # cmd_poisson's run_cfg: adopted step sizes
            d = saved_or_run(run_cfg(ds, sid, t0, model_A17, 0.02, nf_A17, out_root='experiments/walkthrough',
                                     poisson=mode, deltas=dict(ADOPTED)))
        if d is None:
            print(f'no saved run for {lab} {mode}' + (' (run_all.sh job poisson)' if rep_A17 else ''))
        else:
            runs_A17.setdefault(s, {})[mode] = d
READY['A17'] = bool(runs_A17)
for s, by in runs_A17.items():
    for mode, d in by.items():
        print(f'{s[3]:8s} {mode:9s} {d}')

In [ ]:
%%section A17
# step 2 -- the summaries of the runs, the fields intensity_update quotes
info_A17 = []
for s, by in runs_A17.items():
    for mode in MODES_A17:
        if mode in by:
            c, sm = experiment_tracking.read_config(by[mode]), experiment_tracking.read_summary(by[mode])
            info_A17.append([s[3], mode, round(c['n_frames'] * c['frame_duration'], 2),
                         sm['mean_err_deg_s'], sm['low_freq_share'], sm['recon_r_mean_inloop'],
                         sm['recon_r_mean'], sm['I_std_median'], sm['blowup_frac']])
display(pd.DataFrame(info_A17, columns=['sequence', 'in-loop', 'secs', 'err deg/s', 'low-freq share',
                                    '|r| in-loop', '|r| read-out', 'median I std', 'blow-up']).round(3))

In [ ]:
%%section A17
# step 3 -- cmd_poisson's five panels, unrolled: APS, the in-loop I of each mode,
# and the exact fft / dct read-outs of the iterative run's G (solve_poisson_exact)
from report_runs import aps_frame
from interacting_maps.network_dissertation import solve_poisson_exact
from interacting_maps.style import use_style, save, PAPER_WIDE
from interacting_maps.viz import draw_map
full_A17 = [s for s, by in runs_A17.items() if 'iterative' in by]
seg_A17 = full_A17[0] if full_A17 else None       # the first chosen sequence with an iterative run
stats_A17, pix_A17 = [], None
if seg_A17 is None:
    print('no iterative run: cmd_poisson draws nothing')
else:
    ds, sid, t0, lab = seg_A17
    c = experiment_tracking.read_config(runs_A17[seg_A17]['iterative'])
    maps_A17 = {m: np.load(os.path.join(runs_A17[seg_A17][m], 'maps_final.npz'))
            for m in MODES_A17 if m in runs_A17[seg_A17]}
    panels_A17 = [('APS reference', aps_frame(ds, t0 + c['n_frames'] * c['frame_duration']), 'aps')]
    for m in maps_A17:
        panels_A17.append((f'{m} in-loop' if m != 'iterative' else 'iterative (Eq. 6.61)', maps_A17[m]['I'], 'I'))
    for m in ('fft', 'dct'):
        if 'iterative' not in maps_A17:
            break
        panels_A17.append((f'{m} read-out', solve_poisson_exact(maps_A17['iterative']['G'], m), 'I'))
    print(f"{lab}: {'low_freq':>10} {'contrast':>10}")
    with use_style('paper'):
        fig_A17, ax_A17 = plt.subplots(1, len(panels_A17), figsize=(PAPER_WIDE, 1.9))
        for a, (ttl, img, kind) in zip(ax_A17, panels_A17):
            if img is None:
                a.set_axis_off()
                continue
            st = E._intensity_stats(img)
            stats_A17.append(f"  {ttl:<22} {st['low_freq_share']:10.3f} {st['contrast']:10.3f}")
            print(stats_A17[-1])
            draw_map(a, kind, img, title=ttl)
        fig_A17.suptitle(f'{lab}: the same G, five ways of turning it into I')
        pix_A17 = fig_pixels(fig_A17)
        save(fig_A17, os.path.join(walk_dir('A17'), f'poisson_{lab}.png'), show=True)

In [ ]:
%%section A17
# step 4 -- text_numbers.py::intensity_update, unrolled (report configuration only:
# it reads the poisson job)
lines_A17 = []
if rep_A17:
    for s in SEGMENTS:
        ds, sid, t0, lab = s
        by = runs_A17.get(s, {})
        for mode in MODES_A17:
            if mode not in by:
                continue
            c, sm = experiment_tracking.read_config(by[mode]), experiment_tracking.read_summary(by[mode])
            lines_A17.append(f'{lab:8s} {mode:9s} ({c["n_frames"] * c["frame_duration"]:.1f} s): '
                             f'err {sm["mean_err_deg_s"]:.2f}, low-freq share of the final I '
                             f'{sm["low_freq_share"]:.3f}, track-mean |r| in-loop {sm["recon_r_mean_inloop"]:.2f} '
                             f'read-out {sm["recon_r_mean"]:.2f}, median I std {sm["I_std_median"]:.2f}, '
                             f'blow-up {sm["blowup_frac"]:.2f}')
        if 'iterative' in by and lab == 'poster':   # the APS's own low-frequency share
            c = experiment_tracking.read_config(by['iterative'])
            aps = aps_frame(ds, t0 + c['n_frames'] * c['frame_duration'])
            if aps is not None:
                lines_A17.append(f'{lab:8s} APS frame at the end of the track: low-freq share '
                                 f'{E._intensity_stats(np.log(aps + 1.0))["low_freq_share"]:.3f}')
    print('\n'.join(lines_A17) or '(no runs)')
else:
    print('[skip] report-only step')

In [ ]:
%%section A17
# same-code check -- intensity_update itself (Redraw); cmd_poisson with its run step
# replaced by a no-op on the same runs: same stats, same figure, same RunConfigs
import argparse
from_report = all(os.path.normpath(d).startswith(os.path.normpath(ROOT_A17))
                  for by in runs_A17.values() for d in by.values())
under_A17 = lambda root: all(os.path.normpath(d).startswith(os.path.normpath(root))
                             for by in runs_A17.values() for d in by.values())
if not rep_A17:
    print('[skip] not the report configuration')
elif not from_report:
    need([ROOT_A17], ['poisson'])
    print("[skip] intensity_update: these runs are not the poisson job's")
else:
    with redirect(text_numbers, OUT=[]), contextlib.redirect_stdout(io.StringIO()):
        text_numbers.intensity_update()
        theirs_A17 = [l for l in text_numbers.OUT if l and not l.startswith(('==', '   ('))]
    check('text_numbers.intensity_update', nums(lines_A17), nums(theirs_A17))
# cmd_poisson names its runs under experiments/<name>: the report job's, or the walkthrough's
name_A17 = '' if from_report else ('walkthrough' if under_A17(os.path.join('experiments', 'walkthrough'))
                                   else None)
if name_A17 is None:
    print('[skip] cmd_poisson: the runs come from several roots')
elif seg_A17 is not None and need([os.path.join('data', seg_A17[0])]):
    c = experiment_tracking.read_config(runs_A17[seg_A17]['iterative'])
    base_A17 = {k: v for k, v in c['params'].items() if k in ('delta_FR', 'delta_IMU')
            or v != THESIS_PARAMS.get(k)}
    args_A17 = argparse.Namespace(name=name_A17,
                              duration=c['n_frames'] * c['frame_duration'], dt=c['frame_duration'],
                              sequences=seg_A17[3], model=model_A17,
                              modes=','.join(m for m in MODES_A17 if m in runs_A17[seg_A17]),
                              base=','.join(f'{k}={v}' for k, v in base_A17.items()),
                              tag='', no_frames=True, frame_stride=1)
    seen_A17 = []
    try:
        with caught_figures(run_experiments, _run=lambda E_, rc, sf, st: seen_A17.append(rc)) as got_A17:
            run_experiments.cmd_poisson(E, args_A17)
        KEYS_A17 = ('dataset', 'model', 'segment_id', 'n_frames', 'frame_duration', 'poisson', 'params')
        strip = lambda c_: {k: c_.get(k) for k in KEYS_A17}
        mine_A17 = [strip(experiment_tracking.read_config(runs_A17[seg_A17][m]))
                    for m in MODES_A17 if m in runs_A17[seg_A17]]
        same_A17 = [strip(rc.to_dict()) for rc in seen_A17] == mine_A17
        print('[ok] config' if same_A17 else f'[DIFF] config: {[rc.output_dir for rc in seen_A17]}')
        out_A17 = [l for l in got_A17['out'].getvalue().splitlines()
               if l.startswith('  ') and not l.startswith(('  running', '  ->'))]
        check('cmd_poisson stats', nums(stats_A17), nums(out_A17))
        check('cmd_poisson pixels', pix_A17, next(iter(got_A17['pixels'].values()), []))
    except Exception as e:
        print(f'[DIFF] cmd_poisson stats: {type(e).__name__}: {e}')
    finally:
        get_ipython().run_line_magic('matplotlib', 'inline')   # cmd_poisson selected Agg

<a id="A18"></a>

## A18 · Lens distortion · [↑ overview](#overview)

**Question.** How far does the lens move the events, and does it matter how
the distortion is handled? The report carries it inside the kinematic matrix C
(`DISTORTION_MODE='C_full'`) instead of undistorting the events before binning;
undistortion would leave pixels of the frame empty and spread neighbouring
events apart.

**Made by** `report/make_figures.py::fig_distortion` (`python report/make_figures.py`;
needs only the raw events of `poster_rotation`) → `report/figures/fig_distortion.png/.pdf`;
it calls `make_figures.py::distortion_numbers` → `report/experiments/distortion_numbers.csv`.
Also `run_all.sh` job `distortion_test` = `python tests/test_distortion_cmax.py` (stdout).

**Read by** fig:distortion and the numbers of the Lens Distortion paragraph.

**Steps**: where undistortion sends the sensor pixels (`distortion_numbers`) →
the events of 0.2 s, binned raw and undistorted → the figure → same-code check.

In [ ]:
# step 1 -- the raw events and the calibration: fig_distortion's fixed window (Redraw),
# or the panel's segment for n_frames * dt (Live). No run is made.
from interacting_maps.data_loader import (CameraCalibration, load_events_fast,
                                          undistort_events, events_to_vframe)
p = params()
if p['mode'] == 'Redraw':
    note_redraw(p, 10)                          # 0.2 s = 10 frames of 20 ms
    ds_A18, t0_A18, dur_A18 = 'poster_rotation', 8.816, 0.20   # fixed in fig_distortion
else:
    ds_A18, _sid, t0_A18, _lab = p['seg']
    dur_A18 = p['n_frames'] * p['dt']
H_A18, W_A18 = 180, 240
paths_A18 = E.get_dataset_paths(ds_A18)
READY['A18'] = need([paths_A18['events'], paths_A18['calib']])
if READY['A18']:
    calib_A18 = CameraCalibration(paths_A18['calib'])
    print(f'{ds_A18}: t {t0_A18} s + {dur_A18:.2f} s; fx {calib_A18.fx:.1f} fy {calib_A18.fy:.1f} '
          f'cx {calib_A18.cx:.1f} cy {calib_A18.cy:.1f}; dist {np.round(calib_A18.dist, 4)}')

In [ ]:
%%section A18
# step 2 -- make_figures.py::distortion_numbers, unrolled: undistort every sensor pixel
import cv2
c_A18 = calib_A18
K_A18 = np.array([[c_A18.fx, 0, c_A18.cx], [0, c_A18.fy, c_A18.cy], [0, 0, 1]])
ys_A18, xs_A18 = np.mgrid[0:H_A18, 0:W_A18]
pts_A18 = np.stack([xs_A18.ravel(), ys_A18.ravel()], -1).astype(np.float64).reshape(-1, 1, 2)
u_A18 = cv2.undistortPoints(pts_A18, K_A18, c_A18.dist.astype(np.float64), P=K_A18)[:, 0, :].reshape(H_A18, W_A18, 2)
cells_A18 = u_A18.astype(np.int32)                  # binned as events_to_vframe bins
hit_A18 = ((cells_A18[..., 0] >= 0) & (cells_A18[..., 0] < W_A18) & (cells_A18[..., 1] >= 0) & (cells_A18[..., 1] < H_A18))
inside_A18 = ((u_A18[..., 0] >= 0) & (u_A18[..., 0] < W_A18) & (u_A18[..., 1] >= 0) & (u_A18[..., 1] < H_A18))
dx_A18 = np.linalg.norm(u_A18[:, 1:] - u_A18[:, :-1], axis=-1)[inside_A18[:, 1:] & inside_A18[:, :-1]]
dy_A18 = np.linalg.norm(u_A18[1:] - u_A18[:-1], axis=-1)[inside_A18[1:] & inside_A18[:-1]]
num_A18 = {'sensor_pixels': H_A18 * W_A18,
           'occupied_undistorted': int(np.unique(cells_A18[hit_A18][:, 1] * W_A18 + cells_A18[hit_A18][:, 0]).size),
           'max_spacing_px': round(float(max(dx_A18.max(), dy_A18.max())), 3)}
shift_A18 = np.linalg.norm(u_A18 - np.stack([xs_A18, ys_A18], -1), axis=-1)
print(', '.join(f'{k} {v}' for k, v in num_A18.items()))
print(f'{num_A18["sensor_pixels"] - num_A18["occupied_undistorted"]} frame pixels get no sensor pixel; '
      f'shift of a pixel by undistortion: median {np.median(shift_A18):.2f} px, max {shift_A18.max():.2f} px (corners)')

In [ ]:
%%section A18
# step 3 -- fig_distortion's inputs: the events binned raw and after undistort_events
ev_A18 = load_events_fast(paths_A18['events'], t_start=t0_A18, duration=dur_A18 + 0.02)
ev_A18 = ev_A18[ev_A18[:, 0] < t0_A18 + dur_A18]
V_raw_A18 = events_to_vframe(ev_A18, H_A18, W_A18, clip_value=40.0, normalise=True)
V_und_A18 = events_to_vframe(undistort_events(ev_A18, calib_A18), H_A18, W_A18, clip_value=40.0, normalise=True)
print(f'{len(ev_A18)} events; pixels with events: raw {(V_raw_A18 != 0).sum()}, '
      f'undistorted {(V_und_A18 != 0).sum()} of {H_A18 * W_A18}')

In [ ]:
%%section A18
# step 4 -- make_figures.py::fig_distortion, unrolled
from interacting_maps.style import use_style, save, PAPER_WIDTH
from interacting_maps.viz import draw_map, map_scale
panels_A18 = [(V_raw_A18, 'raw events\n' r'(lens in $\mathcal{C}$)'),
              (V_und_A18, 'undistorted\nevents'),
              (V_und_A18[10:80, 140:230], 'detail\n(undistorted)')]
scale_A18 = map_scale('V', V_und_A18)
with use_style('paper'):
    fig_A18, ax_A18 = plt.subplots(1, 3, figsize=(PAPER_WIDTH, 1.25))
    for a, (img, ttl) in zip(ax_A18, panels_A18):
        draw_map(a, 'V', img, title=ttl, scale=scale_A18)
    pix_A18 = fig_pixels(fig_A18)
    save(fig_A18, os.path.join(walk_dir('A18'), 'fig_distortion.png'), show=True)

### What `tests/test_distortion_cmax.py` asks (not unrolled)

The `distortion_test` job checks, on poster seg_C without the network, that the
distortion-aware warp CMax uses -- the same lens geometry as the
distortion-aware C matrix -- agrees with the trusted undistort-first path:

- **A, forward model.** Is the forward Brown–Conrady projection the same as
  `cv2.projectPoints`? (max error < 1e-6 px, pass/fail)
- **B, geometry.** For a fixed rotation (the gyro's), does warping the raw
  events the distortion-aware way and then undistorting land where the plain
  pinhole warp of the undistorted events lands? (max < 0.5 px, pass/fail: the
  real confirmation of the C-matrix re-distortion)
- **C, estimator (context).** Do the two CMax estimates of ω agree, and how far
  is each from the gyro? They need not match exactly: the CMax objective is
  computed on different pixel grids. Reported in deg/s, not a gate.

It prints `RESULT: PASS` when A and B pass. Run it with
`PYTHONPATH=. python tests/test_distortion_cmax.py`.

In [ ]:
%%section A18
# same-code check -- distortion_numbers and fig_distortion themselves (their ROOT and
# FIG redirected), and the csv the report job wrote
chk = os.path.join(walk_dir('A18'), 'check')
csv_rel = os.path.join('report', 'experiments', 'distortion_numbers.csv')
read_num = lambda path: {k: float(v) for k, v in next(csv.DictReader(open(path))).items()}
if p['mode'] == 'Redraw' and need([csv_rel], ['make_figures (fig_distortion)']):
    check('distortion_numbers vs csv', num_A18, read_num(csv_rel))
with redirect(make_figures, ROOT=chk), contextlib.redirect_stdout(io.StringIO()):
    make_figures.distortion_numbers(calib_A18, H_A18, W_A18)
check('distortion_numbers', num_A18, read_num(os.path.join(chk, csv_rel)))
if p['mode'] == 'Redraw':
    with caught_figures(make_figures, FIG=chk, ROOT=chk) as got:
        make_figures.fig_distortion(E)
    check('fig_distortion pixels', pix_A18, got['pixels'].get('fig_distortion.png', []))
else:
    print("Live: fig_distortion's window is fixed (poster, 0.2 s); its pixels are checked in Redraw")

<a id="A19"></a>

## A19 · Adopted parameters (tab:params) · [↑ overview](#overview)

**Question.** Which step sizes does the report use, per relation, for Cook's
network and for the thesis network?

**Made by** nothing run: the values are `config.py`'s `COOK_PARAMS`,
`THESIS_PARAMS` and `ITERS_PER_FRAME` (and the 20 ms window).

**Read by** `report/make_tables.py::table_params` → `report/experiments/table_params.tex`
→ tab:params. The table is written from the code, so it cannot disagree with it.

**Steps**: the six relaxation rates and the loop settings → the LaTeX table →
the panel's defaults against the table → same-code check. Redraw and Live are
the same here.

In [ ]:
# step 1 -- the values: config.py, nothing to find or run
from config import COOK_PARAMS
p = params()
READY['A19'] = True
print(f"config.py: ITERS_PER_FRAME {ITERS_PER_FRAME}; window 20 ms ({p['mode']} reads the same values)")
display(pd.DataFrame({'cook': COOK_PARAMS, 'thesis': THESIS_PARAMS}))

In [ ]:
%%section A19
# step 2 -- make_tables.py::table_params, unrolled: one row per relation
from make_tables import tabular
ROWS_A19 = [(r'$\delta_{\Vmap\!\Fmap\!\Gmap}$', r'flow constraint~\eqref{eq:ofce}', 'delta_VFG'),
            (r'$\delta_{I\!G}$', r'$\Gmap$ from $\Imap$', 'delta_IG'),
            (r'$\delta_{G\!I}$', r'$\Imap$ from $\Gmap$', 'delta_GI'),
            (r'$\delta_{R\!F}$', r'$\Fmap$ from $\wvec$', 'delta_RF'),
            (r'$\delta_{F\!R}$', r'$\wvec$ from $\Fmap$', 'delta_FR'),
            (r'$\delta_{\text{anchor}}$', r'external $\wvec$ anchor~\eqref{eq:anchorcost}', 'delta_IMU')]
cell_A19 = lambda d, k: f'{d[k]:.2f}' if k in d else '---'
body_A19 = [f'{s} & {rel} & {cell_A19(COOK_PARAMS, k)} & {cell_A19(THESIS_PARAMS, k)} \\\\' for s, rel, k in ROWS_A19]
body_A19 += [r'\midrule',
         rf'\multicolumn{{2}}{{@{{}}l}}{{iterations per frame}} & \multicolumn{{2}}{{c}}{{{ITERS_PER_FRAME}}} \\',
         r'\multicolumn{2}{@{}l}{window $\Delta t$} & \multicolumn{2}{c}{20\,ms} \\']
table_A19 = tabular('@{}llcc@{}', [r'Symbol & Relation & Cook & Martel \\', r'\midrule'], body_A19)
display(pd.DataFrame([[k, cell_A19(COOK_PARAMS, k), cell_A19(THESIS_PARAMS, k)] for _s, _r, k in ROWS_A19],
                     columns=['parameter', 'Cook', 'thesis (Martel)']))
print(table_A19)

In [ ]:
%%section A19
# step 3 -- the panel against the table: its defaults are the table's thesis column
want_A19 = {**{k: float(cell_A19(THESIS_PARAMS, k)) for _s, _r, k in ROWS_A19},
            'n_iters': ITERS_PER_FRAME, 'dt': 0.02}
display(pd.DataFrame({'table': want_A19,
                      'panel default': {k: DEFAULTS[k] for k in want_A19},
                      'panel now': {k: p[k] for k in want_A19}}))
changed_A19 = [k for k in want_A19 if p[k] != DEFAULTS[k]]
if changed_A19:
    print(f'the panel differs from the report in {changed_A19}: reset it to get the report back')

In [ ]:
%%section A19
# same-code check -- table_params itself, the tex the report job wrote, and the panel defaults
theirs_A19 = make_tables.table_params()
check('table_params tex', nums(table_A19.splitlines()), nums(theirs_A19.splitlines()))
if table_A19 != theirs_A19:
    print('   the numbers agree but the LaTeX text differs')
tex_A19 = os.path.join('report', 'experiments', 'table_params.tex')
if need([tex_A19], ['make_tables (python report/make_tables.py)']):
    with open(tex_A19, encoding='utf-8') as f:
        rep_A19 = [l for l in f.read().splitlines() if not l.startswith('%')]
    check('table_params vs report tex', nums(table_A19.splitlines()), nums(rep_A19))
check('panel defaults = table_params', {k: round(float(DEFAULTS[k]), 2) for k in want_A19}, want_A19)